**Install and Import CTGAN**

In [25]:
# ============================================================
# CELL 1 — IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.linear_model import (
    LinearRegression,
    LogisticRegression
)

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error,
    accuracy_score,
    classification_report,
    confusion_matrix
)

from ctgan import CTGAN

# Reproducibility
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)

print("Libraries imported successfully.")

Libraries imported successfully.


In [26]:
# ============================================================
# CELL 2 — LOAD FINAL_DS
# ============================================================

FINAL_DS_FILE = "Final_DS.csv"

macro_df = pd.read_csv(
    FINAL_DS_FILE
)

print("Final_DS loaded successfully.")

Final_DS loaded successfully.


In [27]:
# ============================================================
# CELL 3 — INSPECT FINAL_DS COLUMNS
# ============================================================

print("Final_DS columns:")
print("=" * 60)

for i, column in enumerate(macro_df.columns):
    print(f"{i}: {column}")

Final_DS columns:
0: Date
1: Year
2: Month
3: Bank_Rate
4: CPI
5: Unemployment_Rate
6: House_Price_Index
7: GDP_Growth
8: Current_Accounts
9: Savings_Accounts
10: Mortgage_Approvals
11: Net_Consumer_Credit
12: Credit_Card_Lending
13: Consumer_Credit


In [28]:
# ============================================================
# CELL 4 — INSPECT TIME COLUMNS
# ============================================================

print("Date column:")
print(macro_df["Date"].head(10))

print("\nYear column:")
print(macro_df["Year"].head(10))

print("\nMonth column:")
print(macro_df["Month"].head(10))

Date column:
0    2008-01
1    2008-02
2    2008-03
3    2008-04
4    2008-05
5    2008-06
6    2008-07
7    2008-08
8    2008-09
9    2008-10
Name: Date, dtype: object

Year column:
0    2008
1    2008
2    2008
3    2008
4    2008
5    2008
6    2008
7    2008
8    2008
9    2008
Name: Year, dtype: int64

Month column:
0      January
1     February
2        March
3        April
4          May
5         June
6         July
7       August
8    September
9      October
Name: Month, dtype: object


In [29]:
# ============================================================
# CELL 5 — CONVERT DATE TO DATETIME
# ============================================================

macro_df["Date"] = pd.to_datetime(
    macro_df["Date"],
    format="%Y-%m"
)

print("Date converted to datetime successfully.")

Date converted to datetime successfully.


In [30]:
# ============================================================
# CELL 6 — VERIFY DATE RANGE
# ============================================================

print(
    f"Start date: {macro_df['Date'].min():%Y-%m}"
)

print(
    f"End date: {macro_df['Date'].max():%Y-%m}"
)

Start date: 2008-01
End date: 2025-12


In [31]:
# ============================================================
# CELL 7 — CHECK DUPLICATE MONTHS
# ============================================================

duplicate_dates = macro_df["Date"].duplicated().sum()

print(
    f"Duplicate monthly dates: {duplicate_dates}"
)

Duplicate monthly dates: 0


In [32]:
# ============================================================
# CELL 8 — SELECT PROJECT VARIABLES
# ============================================================

project_columns = [
    "Date",
    "Year",
    "Month",
    "Bank_Rate",
    "CPI",
    "Unemployment_Rate",
    "House_Price_Index",
    "GDP_Growth",
    "Mortgage_Approvals",
    "Net_Consumer_Credit",
    "Credit_Card_Lending"
]

macro_df = macro_df[
    project_columns
].copy()

print("Project variables selected successfully.")

Project variables selected successfully.


In [34]:
# ============================================================
# CELL 10 — CHECK DATA TYPES
# ============================================================

print("Data types of selected variables:")
print("=" * 60)

display(
    macro_df.dtypes.to_frame(
        name="Data_Type"
    )
)

Data types of selected variables:


,Data_Type
Date,datetime64[ns]
Year,int64
Month,object
Bank_Rate,float64
CPI,float64
Unemployment_Rate,float64
House_Price_Index,float64
GDP_Growth,float64
Mortgage_Approvals,int64
Net_Consumer_Credit,int64


In [35]:
# ============================================================
# CELL 11 — LOAD ONS AWE DATASET
# ============================================================

ONS_FILE = "emp.csv"

awe_df = pd.read_csv(
    ONS_FILE
)

print("ONS AWE dataset loaded successfully.")

ONS AWE dataset loaded successfully.


In [36]:
# ============================================================
# CELL 12 — INSPECT ONS AWE COLUMNS
# ============================================================

print("ONS AWE columns:")
print("=" * 60)

for i, column in enumerate(awe_df.columns):
    print(f"{i}: {column}")

ONS AWE columns:
0: Title
1: AWE: Whole Economy Real Terms Index: Seasonally Adjusted Regular Pay
2: AWE: Whole Economy Real Terms Year on Year Single Month Growth (%): Seasonally Adjusted Regular Pay
3: AWE: Whole Economy Real Terms Year on Year three Month Growth (%): Seasonally Adjusted Regular Pay
4: AWE: Whole Economy Real Terms Level (£): Seasonally Adjusted Regular Pay
5: AWE: Whole Economy Real Terms Index: Seasonally Adjusted Total Pay
6: AWE: Whole Economy Real Terms Year on Year Single Month Growth (%): Seasonally Adjusted Total Pay
7: AWE: Whole Economy Real Terms Year on Year Three Month Growth (%): Seasonally Adjusted Total Pay
8: AWE: Whole Economy Real Terms Level (£): Seasonally Adjusted Total Pay
9: AWE: Whole Economy Year on Year Single Month Growth (%): Non Seasonally Adjusted Total Pay Excluding Arrears
10: AWE: Manufacturing Year on Year Single Month Growth (%): Non Seasonally Adjusted Total Pay Excluding Arrears
11: AWE: Construction Year on Year Single Month Gro

In [37]:
# ============================================================
# CELL 13 — IDENTIFY ONS SALARY SERIES
# ============================================================

ONS_SALARY_COLUMN = (
    "AWE: Whole Economy Level (£): "
    "Seasonally Adjusted Regular Pay Excluding Arrears"
)

print("ONS salary series:")

print(ONS_SALARY_COLUMN)

ONS salary series:
AWE: Whole Economy Level (£): Seasonally Adjusted Regular Pay Excluding Arrears


In [38]:
# ============================================================
# CELL 14 — INSPECT ONS SALARY DATA
# ============================================================

print("ONS Title and salary values:")
print("=" * 70)

display(
    awe_df[
        [
            "Title",
            ONS_SALARY_COLUMN
        ]
    ].head(20)
)

ONS Title and salary values:


,Title,AWE: Whole Economy Level (£): Seasonally Adjusted Regular Pay Excluding Arrears
0,2008 JAN,398
1,2008 FEB,400
2,2008 MAR,402
3,2008 APR,404
4,2008 MAY,403
5,2008 JUN,404
6,2008 JUL,405
7,2008 AUG,406
8,2008 SEP,407
9,2008 OCT,409


In [39]:
# ============================================================
# CELL 15 — EXTRACT MONTHLY ONS AWE
# ============================================================

salary_history = awe_df[
    [
        "Title",
        ONS_SALARY_COLUMN
    ]
].copy()


# ------------------------------------------------------------
# EXTRACT YEAR AND MONTH
# ------------------------------------------------------------

salary_history["Year"] = (
    salary_history["Title"]
    .str.extract(r"^(\d{4})", expand=False)
    .astype(float)
)


salary_history["Month_Name"] = (
    salary_history["Title"]
    .str.extract(
        r"^\d{4}\s+([A-Za-z]{3})",
        expand=False
    )
)


# ------------------------------------------------------------
# CONVERT SALARY TO NUMERIC
# ------------------------------------------------------------

salary_history["ONS_AWE"] = pd.to_numeric(
    salary_history[ONS_SALARY_COLUMN],
    errors="coerce"
)


# ------------------------------------------------------------
# KEEP 2008–2025
# ------------------------------------------------------------

salary_history = salary_history[
    salary_history["Year"].between(
        START_YEAR,
        END_YEAR
    )
].copy()


# ------------------------------------------------------------
# CREATE MONTH NUMBER
# ------------------------------------------------------------

month_map = {
    "JAN": 1,
    "FEB": 2,
    "MAR": 3,
    "APR": 4,
    "MAY": 5,
    "JUN": 6,
    "JUL": 7,
    "AUG": 8,
    "SEP": 9,
    "OCT": 10,
    "NOV": 11,
    "DEC": 12
}

salary_history["Month"] = (
    salary_history["Month_Name"]
    .str.upper()
    .map(month_map)
)


# ------------------------------------------------------------
# CREATE DATE
# ------------------------------------------------------------

salary_history["Date"] = pd.to_datetime(
    dict(
        year=salary_history["Year"].astype(int),
        month=salary_history["Month"],
        day=1
    )
)


# ------------------------------------------------------------
# KEEP ONLY REQUIRED COLUMNS
# ------------------------------------------------------------

salary_history = salary_history[
    [
        "Date",
        "Year",
        "Month",
        "ONS_AWE"
    ]
].sort_values(
    "Date"
).reset_index(drop=True)


print("Monthly ONS AWE extracted successfully.")

Monthly ONS AWE extracted successfully.


In [40]:
# ============================================================
# CELL 16 — VALIDATE ONS AWE PERIOD
# ============================================================

print(
    f"ONS AWE start date: {salary_history['Date'].min():%Y-%m}"
)

print(
    f"ONS AWE end date: {salary_history['Date'].max():%Y-%m}"
)

print(
    f"ONS AWE observations: {len(salary_history)}"
)

ONS AWE start date: 2008-01
ONS AWE end date: 2025-12
ONS AWE observations: 216


In [41]:
# ============================================================
# CELL 17 — CHECK MISSING ONS AWE VALUES
# ============================================================

missing_awe = salary_history["ONS_AWE"].isna().sum()

print(
    f"Missing ONS AWE values: {missing_awe}"
)

Missing ONS AWE values: 0


In [42]:
# ============================================================
# CELL 18 — CHECK DUPLICATE ONS MONTHLY DATES
# ============================================================

duplicate_awe_dates = (
    salary_history["Date"]
    .duplicated()
    .sum()
)

print(
    f"Duplicate ONS monthly dates: {duplicate_awe_dates}"
)

Duplicate ONS monthly dates: 0


In [46]:
# ============================================================
# CELL 19 — LOAD BANK OF ENGLAND DATA
# ============================================================

BOE_FILE = "Bank_of_England_volumes.csv"

boe_df = pd.read_csv(
    BOE_FILE
)

print("Bank of England dataset loaded successfully.")

Bank of England dataset loaded successfully.


In [47]:
# ============================================================
# CELL 20 — INSPECT BANK OF ENGLAND COLUMNS
# ============================================================

print("Bank of England columns:")
print("=" * 60)

for i, column in enumerate(boe_df.columns):
    print(f"{i}: {column}")

Bank of England columns:
0: Date
1: Month_year
2: YEAR
3: Monthly amounts outstanding of monetary financial institutions' sterling net lending  to household sector (in sterling millions) seasonally adjusted  						[a] [b] [c] [d] [e] [f] [g] [h] [i] [j] [k] [l] 						LPMBC44
4: Monthly amounts outstanding of monetary financial institutions' sterling consumer credit lending excluding securitisations to individuals (in sterling millions) seasonally adjusted  						[a] [m] [n] [o] [p] [q] [r] [s] [d] [e] [f] [g] [j] [t] [l] 						LPMBC46
5: Monthly amounts outstanding of monetary financial institutions' sterling net credit card lending to individuals (in sterling millions) seasonally adjusted  						[a] [n] [u] [o] [v] [s] [e] [w] [g] 						LPMBC53
6: Monthly amounts outstanding of monetary financial institutions' sterling consumer credit (excluding credit card) excluding securitisations to individuals (in sterling millions) seasonally adjusted  						[x] [a] [m] [n] [y] [q] [z] [r] [r] [

In [48]:
# ============================================================
# CELL 21 — INSPECT BOE DATE VALUES
# ============================================================

print("BoE Date values:")
print("=" * 60)

display(
    boe_df[
        [
            "Date",
            "Month_year",
            "YEAR"
        ]
    ].head(10)
)

BoE Date values:


,Date,Month_year,YEAR
0,31-Dec-08,Dec-08,8
1,30-Nov-08,Nov-08,8
2,31-Oct-08,Oct-08,8
3,30-Sep-08,Sep-08,8
4,31-Aug-08,Aug-08,8
5,31-Jul-08,Jul-08,8
6,30-Jun-08,Jun-08,8
7,31-May-08,May-08,8
8,30-Apr-08,Apr-08,8
9,31-Mar-08,Mar-08,8


In [49]:
# ============================================================
# CELL 22 — CONVERT BOE DATE
# ============================================================

boe_df["Date"] = pd.to_datetime(
    boe_df["Date"],
    format="%d-%b-%y"
)

print("BoE Date converted to datetime successfully.")

BoE Date converted to datetime successfully.


In [50]:
# ============================================================
# CELL 23 — VERIFY BOE DATE RANGE
# ============================================================

print(
    f"BoE start date: {boe_df['Date'].min():%Y-%m}"
)

print(
    f"BoE end date: {boe_df['Date'].max():%Y-%m}"
)

BoE start date: 2008-01
BoE end date: 2025-12


In [51]:
# ============================================================
# CELL 24 — CHECK DUPLICATE BOE MONTHLY DATES
# ============================================================

duplicate_boe_dates = (
    boe_df["Date"]
    .duplicated()
    .sum()
)

print(
    f"Duplicate BoE monthly dates: {duplicate_boe_dates}"
)

Duplicate BoE monthly dates: 0


In [53]:
# ============================================================
# CELL 25 — IDENTIFY BOE SERIES COLUMNS
# ============================================================

boe_codes = [
    "LPMBC44",
    "LPMBC46",
    "LPMBC53",
    "LPMBC54",
    "LPMBC55",
    "LPMZ5G7"
]


for code in boe_codes:

    matching_columns = [
        column
        for column in boe_df.columns
        if code in str(column)
    ]

    print(f"\n{code}:")

    for column in matching_columns:
        print(column)


LPMBC44:
Monthly amounts outstanding of monetary financial institutions' sterling net lending  to household sector (in sterling millions) seasonally adjusted  						[a] [b] [c] [d] [e] [f] [g] [h] [i] [j] [k] [l] 						LPMBC44

LPMBC46:
Monthly amounts outstanding of monetary financial institutions' sterling consumer credit lending excluding securitisations to individuals (in sterling millions) seasonally adjusted  						[a] [m] [n] [o] [p] [q] [r] [s] [d] [e] [f] [g] [j] [t] [l] 						LPMBC46

LPMBC53:
Monthly amounts outstanding of monetary financial institutions' sterling net credit card lending to individuals (in sterling millions) seasonally adjusted  						[a] [n] [u] [o] [v] [s] [e] [w] [g] 						LPMBC53

LPMBC54:
Monthly amounts outstanding of monetary financial institutions' sterling consumer credit (excluding credit card) excluding securitisations to individuals (in sterling millions) seasonally adjusted  						[x] [a] [m] [n] [y] [q] [z] [r] [r] [d] [1] [t] [l] 						LPMBC5

In [54]:
# ============================================================
# CELL 26 — CREATE CLEAN BOE VARIABLE NAMES
# ============================================================

boe_retail_df = boe_df.rename(
    columns={
        next(
            col for col in boe_df.columns
            if "LPMBC44" in str(col)
        ): "Household_Net_Lending",

        next(
            col for col in boe_df.columns
            if "LPMBC46" in str(col)
        ): "Consumer_Credit",

        next(
            col for col in boe_df.columns
            if "LPMBC53" in str(col)
        ): "Credit_Card_Lending",

        next(
            col for col in boe_df.columns
            if "LPMBC54" in str(col)
        ): "Consumer_Credit_Excl_Cards",

        next(
            col for col in boe_df.columns
            if "LPMBC55" in str(col)
        ): "Secured_Lending",

        next(
            col for col in boe_df.columns
            if "LPMZ5G7" in str(col)
        ): "Consumer_Loans_Excl_Cards_Overdrafts"
    }
)

print("BoE variables renamed for analysis.")

BoE variables renamed for analysis.


In [55]:
# ============================================================
# CELL 27 — SELECT CLEAN BOE VARIABLES
# ============================================================

boe_retail_df = boe_retail_df[
    [
        "Date",
        "Household_Net_Lending",
        "Consumer_Credit",
        "Credit_Card_Lending",
        "Consumer_Credit_Excl_Cards",
        "Secured_Lending",
        "Consumer_Loans_Excl_Cards_Overdrafts"
    ]
].copy()

print("Required BoE variables selected successfully.")

Required BoE variables selected successfully.


In [56]:
# ============================================================
# CELL 29 — CHECK BOE DATA TYPES
# ============================================================

print("BoE variable data types:")
print("=" * 60)

display(
    boe_retail_df.dtypes.to_frame(
        name="Data_Type"
    )
)

BoE variable data types:


,Data_Type
Date,datetime64[ns]
Household_Net_Lending,int64
Consumer_Credit,int64
Credit_Card_Lending,int64
Consumer_Credit_Excl_Cards,int64
Secured_Lending,int64
Consumer_Loans_Excl_Cards_Overdrafts,int64


In [57]:
# ============================================================
# CELL 30 — VERIFY BOE DATE RANGE
# ============================================================

print(
    f"BoE start date: {boe_retail_df['Date'].min():%Y-%m}"
)

print(
    f"BoE end date: {boe_retail_df['Date'].max():%Y-%m}"
)

print(
    f"BoE observations: {len(boe_retail_df)}"
)

BoE start date: 2008-01
BoE end date: 2025-12
BoE observations: 216


In [58]:
# ============================================================
# CELL 31 — COMPARE FINAL_DS AND BOE PERIODS
# ============================================================

print("Final_DS period:")
print(
    f"{macro_df['Date'].min():%Y-%m} to "
    f"{macro_df['Date'].max():%Y-%m}"
)

print("\nBoE period:")
print(
    f"{boe_retail_df['Date'].min():%Y-%m} to "
    f"{boe_retail_df['Date'].max():%Y-%m}"
)

Final_DS period:
2008-01 to 2025-12

BoE period:
2008-01 to 2025-12


In [79]:
# ============================================================
# CELL 32 — LOAD FCA SOURCE WORKBOOK
# ============================================================

FCA_FILE = "FCA_DATA.xlsx"

fca_excel = pd.ExcelFile(
    FCA_FILE
)

print("FCA source workbook loaded successfully.")

FCA source workbook loaded successfully.


In [80]:
# ============================================================
# CELL 33 — IDENTIFY FCA WORKBOOK SHEETS
# ============================================================

print("FCA workbook sheets:")
print("=" * 60)

for sheet in fca_excel.sheet_names:
    print("-", sheet)

FCA workbook sheets:
- Intro
- How to read the tables
- Table of contents
- Demographics


In [81]:
# ============================================================
# CELL 34 — LOAD FCA DEMOGRAPHICS
# ============================================================

fca_demographics = pd.read_excel(
    FCA_FILE,
    sheet_name="Demographics",
    header=None
)

print("FCA Demographics sheet loaded successfully.")

FCA Demographics sheet loaded successfully.


In [82]:
# ============================================================
# CELL 35 — INSPECT FCA DEMOGRAPHICS STRUCTURE
# ============================================================

display(
    fca_demographics.iloc[:30, :12]
)

,0,1,2,3,4,5,6,7,8,9,10,11
0,Table 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,D1. What is your sex? A question about gender ...,NaN,Total,Sex,NaN,Age,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,Male,Female,18-24,25-34,35-44,45-54,55-64,65-74,75+
3,"A rebased version of this table follows, rebas...",NaN,NaN,A1,B1,C1,D1,E1,F1,G1,H1,I1
4,Total,NaN,17950,8828,8826,1173,3173,3038,2842,3046,2912,1766
5,NaN,NaN,17950,8527,9104,1867,3031,3010,2853,2923,2538,1728
6,NaN,NaN,1,1,1,1,1,1,1,1,1,1
7,NaN,NaN,1,0.48,0.52,0.1,0.17,0.17,0.16,0.16,0.14,0.1
8,Male,Unweighted,8828,8828,-,470,1282,1328,1347,1578,1672,1151
9,NaN,Weighted,8527,8527,-,902,1449,1432,1378,1417,1117,833


In [90]:
# ============================================================
# CELL 36 — EXTRACT FCA AGE DISTRIBUTION
# ============================================================

# ------------------------------------------------------------
# AGE GROUPS
# ------------------------------------------------------------

age_categories = (
    fca_demographics.iloc[2, 6:12]
    .tolist()
)


# ------------------------------------------------------------
# FCA COLUMN PERCENTAGES
# ------------------------------------------------------------

age_percentages = (
    fca_demographics.iloc[7, 6:12]
    .astype(float)
    .tolist()
)


# ------------------------------------------------------------
# CREATE AGE DISTRIBUTION
# ------------------------------------------------------------

fca_age_distribution = pd.DataFrame({
    "Age_Group": age_categories,
    "FCA_Proportion": age_percentages
})


print("FCA age distribution extracted successfully.")

FCA age distribution extracted successfully.


In [91]:
# ============================================================
# CELL 37 — VERIFY FCA AGE DISTRIBUTION
# ============================================================

print(
    fca_age_distribution
)

  Age_Group  FCA_Proportion
0     25-34            0.17
1     35-44            0.17
2     45-54            0.16
3     55-64            0.16
4     65-74            0.14
5       75+            0.10


In [92]:
# ============================================================
# CELL 39 — DEFINE FCA AGE BASELINE
# ============================================================

fca_age_baseline = fca_age_distribution.copy()

fca_age_baseline["Source_Year"] = 2024

fca_age_baseline["Source"] = (
    "FCA Financial Lives Survey 2024"
)

print("FCA 2024 age baseline created.")

FCA 2024 age baseline created.


In [95]:
# ============================================================
# CELL 40 — INSPECT FCA PERSONAL INCOME TABLE
# ============================================================

# Table 127 = Total annual personal income
table_127_row = fca_demographics[
    fca_demographics.iloc[:, 0]
    .astype(str)
    .str.strip()
    .eq("Table 127")
].index[0]

print("FCA Table 127 located at row:", table_127_row)

display(
    fca_demographics.iloc[
        table_127_row : table_127_row + 20,
        :15
    ]
)

FCA Table 127 located at row: 6347


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
6347,Table 127,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6348,D39_1. What is your total annual personal inco...,NaN,Total,Sex,NaN,Age,NaN,NaN,NaN,NaN,NaN,NaN,Ethnicity,NaN,NaN
6349,NaN,NaN,NaN,Male,Female,18-24,25-34,35-44,45-54,55-64,65-74,75+,White,Black & Black British,Asian & Asian British
6350,"A rebased version of this table follows, rebas...",NaN,NaN,A1,B1,C1,D1,E1,F1,G1,H1,I1,J1,K1,L1
6351,Total,NaN,17950,8828,8826,1173,3173,3038,2842,3046,2912,1766,15644,316,957
6352,NaN,NaN,17950,8527,9104,1867,3031,3010,2853,2923,2538,1728,14721,647,1342
6353,NaN,NaN,1,1,1,1,1,1,1,1,1,1,1,1,1
6354,NaN,NaN,1,0.48,0.52,0.1,0.17,0.17,0.16,0.16,0.14,0.1,0.85,0.04,0.08
6355,"Less than £5,000",Unweighted,790,307,461,234,143,113,102,144,34,20,638,21,66
6356,NaN,Weighted,897,381,490,303,155,114,107,147,44,26,673,43,85


In [96]:
# ============================================================
# CELL 41 — INSPECT FCA PERSONAL INCOME TABLE
# ============================================================

display(
    fca_demographics.iloc[
        table_127_row : table_127_row + 20,
        :15
    ]
)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
6347,Table 127,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6348,D39_1. What is your total annual personal inco...,NaN,Total,Sex,NaN,Age,NaN,NaN,NaN,NaN,NaN,NaN,Ethnicity,NaN,NaN
6349,NaN,NaN,NaN,Male,Female,18-24,25-34,35-44,45-54,55-64,65-74,75+,White,Black & Black British,Asian & Asian British
6350,"A rebased version of this table follows, rebas...",NaN,NaN,A1,B1,C1,D1,E1,F1,G1,H1,I1,J1,K1,L1
6351,Total,NaN,17950,8828,8826,1173,3173,3038,2842,3046,2912,1766,15644,316,957
6352,NaN,NaN,17950,8527,9104,1867,3031,3010,2853,2923,2538,1728,14721,647,1342
6353,NaN,NaN,1,1,1,1,1,1,1,1,1,1,1,1,1
6354,NaN,NaN,1,0.48,0.52,0.1,0.17,0.17,0.16,0.16,0.14,0.1,0.85,0.04,0.08
6355,"Less than £5,000",Unweighted,790,307,461,234,143,113,102,144,34,20,638,21,66
6356,NaN,Weighted,897,381,490,303,155,114,107,147,44,26,673,43,85


In [97]:
# ============================================================
# CELL 42 — INSPECT FCA INCOME TABLE
# ============================================================

income_section = fca_demographics.iloc[
    table_127_row : table_127_row + 80,
    0:4
].copy()

display(
    income_section
)


,0,1,2,3
6347,Table 127,NaN,NaN,NaN
6348,D39_1. What is your total annual personal inco...,NaN,Total,Sex
6349,NaN,NaN,NaN,Male
6350,"A rebased version of this table follows, rebas...",NaN,NaN,A1
6351,Total,NaN,17950,8828
...,...,...,...,...
6422,NaN,Col %,0,0
6423,NaN,Row %,1,0.8
6424,NaN,Sigtest,B1 X1 D2 G2 H2 M2 N2,B1
6425,Don't know,Unweighted,516,220


In [98]:
# ============================================================
# CELL 43 — EXTRACT FCA INCOME BANDS
# ============================================================

income_rows = income_section[
    income_section.iloc[:, 1]
    .notna()
].copy()

print("FCA income categories identified:")

display(
    income_rows.iloc[:, :4]
)

FCA income categories identified:


,0,1,2,3
6355,"Less than £5,000",Unweighted,790,307
6356,NaN,Weighted,897,381
6357,NaN,Col %,0.05,0.04
6358,NaN,Row %,1,0.44
6359,NaN,Sigtest,C1 E1 F1 H1 I1 M1 N1 O1 P1 R1 S1 T1 U1 W1 X1 Y...,B1
...,...,...,...,...
6422,NaN,Col %,0,0
6423,NaN,Row %,1,0.8
6424,NaN,Sigtest,B1 X1 D2 G2 H2 M2 N2,B1
6425,Don't know,Unweighted,516,220


In [99]:
# ============================================================
# CELL 44 — EXTRACT FCA INCOME DISTRIBUTION
# ============================================================

# Identify rows containing an income category.
# The following row is "Unweighted", followed by "Weighted"
# and then "Col %".

income_distribution = []

for i in range(len(income_section) - 3):

    category = income_section.iloc[i, 1]
    row_type = income_section.iloc[i, 2]

    if (
        pd.notna(category)
        and row_type == "Unweighted"
    ):

        col_percent_row = income_section.iloc[
            i + 2
        ]

        income_distribution.append({
            "Income_Band": str(category).strip(),
            "FCA_Proportion": pd.to_numeric(
                col_percent_row.iloc[3],
                errors="coerce"
            )
        })


fca_income_distribution = pd.DataFrame(
    income_distribution
)

print("FCA income distribution extracted.")

display(
    fca_income_distribution
)

FCA income distribution extracted.


""


In [102]:
# ============================================================
# CELL 44A — LOCATE FCA INCOME TABLE DIRECTLY
# ============================================================

income_table_matches = fca_demographics[
    fca_demographics.apply(
        lambda row: row.astype(str)
        .str.contains(
            "Less than £5,000",
            case=False,
            na=False
        )
        .any(),
        axis=1
    )
]

print("Rows containing 'Less than £5,000':")
print("=" * 70)

display(
    income_table_matches.iloc[:, :6]
)

Rows containing 'Less than £5,000':


,0,1,2,3,4,5
6355,"Less than £5,000",Unweighted,790,307,461,234
6471,"Less than £5,000",Unweighted,790,307,461,234
6577,"Less than £5,000",Unweighted,377,172,193,93
6693,"Less than £5,000",Unweighted,377,172,193,93


In [103]:
# ============================================================
# CELL 44B — LOCATE TABLE 127 EXACTLY
# ============================================================

table_127_matches = fca_demographics[
    fca_demographics.iloc[:, 0]
    .astype(str)
    .str.strip()
    .eq("Table 127")
]

print("Table 127 row:")
print("=" * 60)

display(
    table_127_matches
)

Table 127 row:


,0,1,2,3,4,5,6,7,8,9,...,98,99,100,101,102,103,104,105,106,107
6347,Table 127,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [104]:
# ============================================================
# CELL 45 — INSPECT FCA TABLE 127
# ============================================================

table_127_row = 6347

display(
    fca_demographics.iloc[
        table_127_row : table_127_row + 80,
        :15
    ]
)

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14
6347,Table 127,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6348,D39_1. What is your total annual personal inco...,NaN,Total,Sex,NaN,Age,NaN,NaN,NaN,NaN,NaN,NaN,Ethnicity,NaN,NaN
6349,NaN,NaN,NaN,Male,Female,18-24,25-34,35-44,45-54,55-64,65-74,75+,White,Black & Black British,Asian & Asian British
6350,"A rebased version of this table follows, rebas...",NaN,NaN,A1,B1,C1,D1,E1,F1,G1,H1,I1,J1,K1,L1
6351,Total,NaN,17950,8828,8826,1173,3173,3038,2842,3046,2912,1766,15644,316,957
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6422,NaN,Col %,0,0,0,-,0,0,0,0,0,-,0,-,0
6423,NaN,Row %,1,0.8,0.2,-,0.23,0.13,0.28,0.16,0.19,-,0.96,-,0.04
6424,NaN,Sigtest,B1 X1 D2 G2 H2 M2 N2,B1,A1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6425,Don't know,Unweighted,516,220,282,136,120,64,67,58,25,46,382,23,52


In [105]:
# ============================================================
# CELL 46 — IDENTIFY FCA TABLE 127 INCOME BANDS
# ============================================================

# First Table 127 income block
income_block = fca_demographics.iloc[
    6351:6427,
    0:4
].copy()

income_band_rows = income_block[
    (
        income_block.iloc[:, 1]
        .astype(str)
        .str.strip()
        .eq("Unweighted")
    )
]

print("FCA Table 127 income-band rows:")
print("=" * 60)

display(
    income_band_rows.iloc[:, :4]
)

FCA Table 127 income-band rows:


,0,1,2,3
6355,"Less than £5,000",Unweighted,790,307
6360,"£5,000 to £9,999",Unweighted,926,318
6365,"£10,000 to £14,999",Unweighted,1427,497
6370,"£15,000 to £19,999",Unweighted,1593,640
6375,"£20,000 to £29,999",Unweighted,3008,1406
6380,"£30,000 to £39,999",Unweighted,2098,1132
6385,"£40,000 to £49,999",Unweighted,1395,836
6390,"£50,000 to £59,999",Unweighted,836,523
6395,"£60,000 to £69,999",Unweighted,512,319
6400,"£70,000 to £99,999",Unweighted,752,537


In [106]:
# ============================================================
# CELL 47 — EXTRACT FCA WEIGHTED INCOME DISTRIBUTION
# ============================================================

income_rows = []

for row in income_band_rows.index:

    income_band = fca_demographics.loc[row, 0]

    # Weighted row is 1 row below Unweighted
    weighted_value = pd.to_numeric(
        fca_demographics.loc[row + 1, 3],
        errors="coerce"
    )

    # Col % row is 2 rows below Unweighted
    col_percentage = pd.to_numeric(
        fca_demographics.loc[row + 2, 3],
        errors="coerce"
    )

    if income_band != "Don't know":

        income_rows.append({
            "Income_Band": income_band,
            "FCA_Weighted": weighted_value,
            "FCA_Proportion": col_percentage
        })


fca_income_distribution = pd.DataFrame(
    income_rows
)

print("FCA income distribution extracted successfully.")

display(
    fca_income_distribution
)

FCA income distribution extracted successfully.


,Income_Band,FCA_Weighted,FCA_Proportion
0,"Less than £5,000",381,0.04
1,"£5,000 to £9,999",384,0.05
2,"£10,000 to £14,999",485,0.06
3,"£15,000 to £19,999",684,0.08
4,"£20,000 to £29,999",1421,0.17
5,"£30,000 to £39,999",1040,0.12
6,"£40,000 to £49,999",719,0.08
7,"£50,000 to £59,999",445,0.05
8,"£60,000 to £69,999",263,0.03
9,"£70,000 to £99,999",447,0.05


In [107]:
# ============================================================
# CELL 48 — NORMALIZE FCA INCOME DISTRIBUTION
# ============================================================

income_total = (
    fca_income_distribution["FCA_Proportion"].sum()
)

fca_income_distribution["Sampling_Probability"] = (
    fca_income_distribution["FCA_Proportion"]
    / income_total
)

print(
    "Original proportion total:",
    round(income_total, 4)
)

print(
    "Normalized probability total:",
    round(
        fca_income_distribution[
            "Sampling_Probability"
        ].sum(),
        4
    )
)

display(
    fca_income_distribution
)

Original proportion total: 0.8
Normalized probability total: 1.0


,Income_Band,FCA_Weighted,FCA_Proportion,Sampling_Probability
0,"Less than £5,000",381,0.04,0.0500
1,"£5,000 to £9,999",384,0.05,0.0625
2,"£10,000 to £14,999",485,0.06,0.0750
3,"£15,000 to £19,999",684,0.08,0.1000
4,"£20,000 to £29,999",1421,0.17,0.2125
5,"£30,000 to £39,999",1040,0.12,0.1500
6,"£40,000 to £49,999",719,0.08,0.1000
7,"£50,000 to £59,999",445,0.05,0.0625
8,"£60,000 to £69,999",263,0.03,0.0375
9,"£70,000 to £99,999",447,0.05,0.0625


In [108]:
# ============================================================
# CELL 49 — CHECK FCA INCOME DISTRIBUTION
# ============================================================

print("Number of income bands:", len(fca_income_distribution))

print(
    "Duplicate income bands:",
    fca_income_distribution["Income_Band"].duplicated().sum()
)

print(
    "Sampling probability total:",
    round(
        fca_income_distribution["Sampling_Probability"].sum(),
        4
    )
)

Number of income bands: 14
Duplicate income bands: 0
Sampling probability total: 1.0


In [109]:
# ============================================================
# CELL 50 — CREATE FCA INCOME BASELINE
# ============================================================

fca_income_baseline = fca_income_distribution[
    [
        "Income_Band",
        "FCA_Weighted",
        "FCA_Proportion",
        "Sampling_Probability"
    ]
].copy()

fca_income_baseline["Source_Year"] = 2024

fca_income_baseline["Source"] = (
    "FCA Financial Lives Survey 2024"
)

print("FCA income baseline created successfully.")

display(
    fca_income_baseline
)

FCA income baseline created successfully.


,Income_Band,FCA_Weighted,FCA_Proportion,Sampling_Probability,Source_Year,Source
0,"Less than £5,000",381,0.04,0.0500,2024,FCA Financial Lives Survey 2024
1,"£5,000 to £9,999",384,0.05,0.0625,2024,FCA Financial Lives Survey 2024
2,"£10,000 to £14,999",485,0.06,0.0750,2024,FCA Financial Lives Survey 2024
3,"£15,000 to £19,999",684,0.08,0.1000,2024,FCA Financial Lives Survey 2024
4,"£20,000 to £29,999",1421,0.17,0.2125,2024,FCA Financial Lives Survey 2024
5,"£30,000 to £39,999",1040,0.12,0.1500,2024,FCA Financial Lives Survey 2024
6,"£40,000 to £49,999",719,0.08,0.1000,2024,FCA Financial Lives Survey 2024
7,"£50,000 to £59,999",445,0.05,0.0625,2024,FCA Financial Lives Survey 2024
8,"£60,000 to £69,999",263,0.03,0.0375,2024,FCA Financial Lives Survey 2024
9,"£70,000 to £99,999",447,0.05,0.0625,2024,FCA Financial Lives Survey 2024


In [113]:
# ============================================================
# CELL 52 — CALCULATE ONS AWE ADJUSTMENT FACTOR
# ============================================================

awe_df["AWE_Adjustment_Factor"] = (
    awe_df[ONS_SALARY_COLUMN]
    / awe_df[ONS_SALARY_COLUMN].iloc[-1]
)

print("AWE adjustment factor calculated.")

display(
    awe_df[
        [
            "Title",
            ONS_SALARY_COLUMN,
            "AWE_Adjustment_Factor"
        ]
    ].head(12)
)

AWE adjustment factor calculated.


,Title,AWE: Whole Economy Level (£): Seasonally Adjusted Regular Pay Excluding Arrears,AWE_Adjustment_Factor
0,2008 JAN,398,0.576812
1,2008 FEB,400,0.579710
2,2008 MAR,402,0.582609
3,2008 APR,404,0.585507
4,2008 MAY,403,0.584058
5,2008 JUN,404,0.585507
6,2008 JUL,405,0.586957
7,2008 AUG,406,0.588406
8,2008 SEP,407,0.589855
9,2008 OCT,409,0.592754


In [114]:
# ============================================================
# CELL 53 — PREPARE HISTORICAL AWE ADJUSTMENT
# ============================================================

awe_income_adjustment = awe_df[
    [
        "Title",
        "AWE_Adjustment_Factor"
    ]
].copy()

print("Historical AWE adjustment prepared.")

display(
    awe_income_adjustment.head(12)
)

Historical AWE adjustment prepared.


,Title,AWE_Adjustment_Factor
0,2008 JAN,0.576812
1,2008 FEB,0.579710
2,2008 MAR,0.582609
3,2008 APR,0.585507
4,2008 MAY,0.584058
5,2008 JUN,0.585507
6,2008 JUL,0.586957
7,2008 AUG,0.588406
8,2008 SEP,0.589855
9,2008 OCT,0.592754


In [118]:
# ============================================================
# CELL 54 — CREATE REPRESENTATIVE FCA INCOME
# ============================================================

def income_band_to_value(band):

    band = str(band).strip()

    if band == "Less than £5,000":
        return 2500

    elif band == "£1,000,000 or more":
        return 1000000

    else:
        # Remove £ and commas
        values = (
            band.replace("£", "")
                .replace(",", "")
        )

        # Split the range
        lower, upper = values.split(" to ")

        lower = float(lower)
        upper = float(upper)

        return (lower + upper) / 2


fca_income_baseline["Representative_Income"] = (
    fca_income_baseline["Income_Band"]
    .apply(income_band_to_value)
)



In [117]:
# ============================================================
# CELL 55 — PREPARE FCA INCOME FOR SYNTHETIC GENERATION
# ============================================================

synthetic_income_distribution = fca_income_baseline[
    [
        "Income_Band",
        "Representative_Income",
        "Sampling_Probability"
    ]
].copy()

print("Synthetic income distribution prepared.")

display(
    synthetic_income_distribution
)

Synthetic income distribution prepared.


,Income_Band,Representative_Income,Sampling_Probability
0,"Less than £5,000",2500.0,0.0500
1,"£5,000 to £9,999",7499.5,0.0625
2,"£10,000 to £14,999",12499.5,0.0750
3,"£15,000 to £19,999",17499.5,0.1000
4,"£20,000 to £29,999",24999.5,0.2125
5,"£30,000 to £39,999",34999.5,0.1500
6,"£40,000 to £49,999",44999.5,0.1000
7,"£50,000 to £59,999",54999.5,0.0625
8,"£60,000 to £69,999",64999.5,0.0375
9,"£70,000 to £99,999",84999.5,0.0625


In [121]:
# ============================================================
# CELL 56 — GENERATE SYNTHETIC CUSTOMER INCOME
# ============================================================

# Number of synthetic customers
n = 10000

# Random seed for reproducibility
np.random.seed(42)

# Sample income bands using FCA probabilities
synthetic_income_bands = np.random.choice(
    synthetic_income_distribution["Income_Band"],
    size=n,
    p=synthetic_income_distribution["Sampling_Probability"]
)

# Map each selected income band to its representative income
income_value_map = dict(
    zip(
        synthetic_income_distribution["Income_Band"],
        synthetic_income_distribution["Representative_Income"]
    )
)

synthetic_income = pd.DataFrame({
    "Customer_ID": range(1, n + 1),
    "Income_Band": synthetic_income_bands
})

synthetic_income["Annual_Income_2024"] = (
    synthetic_income["Income_Band"]
    .map(income_value_map)
)

print("Synthetic customer income generated.")

display(
    synthetic_income.head(10)
)

Synthetic customer income generated.


,Customer_ID,Income_Band,Annual_Income_2024
0,1,"£20,000 to £29,999",24999.5
1,2,"£150,000 to £249,999",199999.5
2,3,"£40,000 to £49,999",44999.5
3,4,"£30,000 to £39,999",34999.5
4,5,"£10,000 to £14,999",12499.5
5,6,"£10,000 to £14,999",12499.5
6,7,"£5,000 to £9,999",7499.5
7,8,"£70,000 to £99,999",84999.5
8,9,"£30,000 to £39,999",34999.5
9,10,"£40,000 to £49,999",44999.5


In [122]:
# ============================================================
# CELL 57 — VALIDATE SYNTHETIC INCOME DISTRIBUTION
# ============================================================

synthetic_income_check = (
    synthetic_income["Income_Band"]
    .value_counts(normalize=True)
    .rename("Synthetic_Proportion")
    .reset_index()
)

synthetic_income_check.columns = [
    "Income_Band",
    "Synthetic_Proportion"
]

# Add FCA target probabilities
synthetic_income_check = synthetic_income_check.merge(
    synthetic_income_distribution[
        [
            "Income_Band",
            "Sampling_Probability"
        ]
    ],
    on="Income_Band",
    how="right"
)

synthetic_income_check = synthetic_income_check.fillna(
    {"Synthetic_Proportion": 0}
)

display(
    synthetic_income_check.sort_values(
        "Income_Band"
    )
)

,Income_Band,Synthetic_Proportion,Sampling_Probability
0,"Less than £5,000",0.0493,0.0500
13,"£1,000,000 or more",0.0000,0.0000
2,"£10,000 to £14,999",0.0765,0.0750
10,"£100,000 to £149,999",0.0371,0.0375
3,"£15,000 to £19,999",0.1022,0.1000
11,"£150,000 to £249,999",0.0249,0.0250
4,"£20,000 to £29,999",0.2137,0.2125
12,"£250,000 to £999,999",0.0225,0.0250
5,"£30,000 to £39,999",0.1527,0.1500
6,"£40,000 to £49,999",0.0984,0.1000


In [123]:
# ============================================================
# CELL 58 — SYNTHETIC INCOME SUMMARY
# ============================================================

income_summary = synthetic_income[
    "Annual_Income_2024"
].describe()

print("Synthetic customer income summary:")
print("=" * 60)

display(income_summary)

Synthetic customer income summary:


count     10000.000000
mean      52951.774650
std       94339.088368
min        2500.000000
25%       17499.500000
50%       24999.500000
75%       44999.500000
max      624999.500000
Name: Annual_Income_2024, dtype: float64

In [127]:
# ============================================================
# CELL 60 — GENERATE ACTUAL AGE FROM FCA 2024 AGE DISTRIBUTION
# ============================================================

np.random.seed(42)

# FCA 2024 age bands and proportions
age_bands = [
    "18-24",
    "25-34",
    "35-44",
    "45-54",
    "55-64",
    "65-74",
    "75+"
]

age_probabilities = [
    0.10,
    0.17,
    0.17,
    0.16,
    0.16,
    0.14,
    0.10
]

# Select an age band for each customer
selected_age_band = np.random.choice(
    age_bands,
    size=len(synthetic_income),
    p=age_probabilities
)

# Generate an actual age within the selected FCA band
def generate_age(band):

    if band == "18-24":
        return np.random.randint(18, 25)

    elif band == "25-34":
        return np.random.randint(25, 35)

    elif band == "35-44":
        return np.random.randint(35, 45)

    elif band == "45-54":
        return np.random.randint(45, 55)

    elif band == "55-64":
        return np.random.randint(55, 65)

    elif band == "65-74":
        return np.random.randint(65, 75)

    elif band == "75+":
        return np.random.randint(75, 91)


synthetic_income["Age_2024"] = [
    generate_age(band)
    for band in selected_age_band
]

# Calculate approximate birth year
synthetic_income["Birth_Year"] = (
    2024 - synthetic_income["Age_2024"]
)

display(
    synthetic_income[
        [
            "Customer_ID",
            "Annual_Income_2024",
            "Age_2024",
            "Birth_Year"
        ]
    ].head(10)
)

,Customer_ID,Annual_Income_2024,Age_2024,Birth_Year
0,1,24999.5,41,1983
1,2,199999.5,90,1934
2,3,44999.5,58,1966
3,4,34999.5,46,1978
4,5,12499.5,29,1995
5,6,12499.5,27,1997
6,7,7499.5,24,2000
7,8,84999.5,73,1951
8,9,34999.5,60,1964
9,10,44999.5,55,1969


In [128]:
# ============================================================
# CELL 61 — HISTORICAL AGE CALCULATION
# ============================================================

def calculate_age(birth_year, year):
    return year - birth_year


# Example: check age progression for first 5 customers
age_check = synthetic_income[
    [
        "Customer_ID",
        "Birth_Year"
    ]
].head(5).copy()

for year in [2008, 2015, 2020, 2024, 2025]:
    age_check[f"Age_{year}"] = (
        year - age_check["Birth_Year"]
    )

display(age_check)

,Customer_ID,Birth_Year,Age_2008,Age_2015,Age_2020,Age_2024,Age_2025
0,1,1983,25,32,37,41,42
1,2,1934,74,81,86,90,91
2,3,1966,42,49,54,58,59
3,4,1978,30,37,42,46,47
4,5,1995,13,20,25,29,30


In [137]:
# ============================================================
# CELL 61 — GENERATE PRODUCT PARTICIPATION
# ============================================================

np.random.seed(42)

# ------------------------------------------------------------
# 1. CURRENT ACCOUNT
# FCA 2024: 97% of adults
# ------------------------------------------------------------

CURRENT_ACCOUNT_PROBABILITY = 0.97

synthetic_income["Has_Current_Account"] = (
    np.random.random(len(synthetic_income))
    < CURRENT_ACCOUNT_PROBABILITY
).astype(int)

In [138]:
# ------------------------------------------------------------
# 2. SAVINGS
# FCA 2024 age-specific cash savings ownership
# ------------------------------------------------------------

def get_savings_probability(age):

    if 18 <= age <= 24:
        return 0.82
    elif 25 <= age <= 34:
        return 0.86
    elif 35 <= age <= 44:
        return 0.87
    elif 45 <= age <= 54:
        return 0.89
    elif 55 <= age <= 64:
        return 0.93
    elif 65 <= age <= 74:
        return 0.95
    elif age >= 75:
        return 0.97
    else:
        return 0.0


synthetic_income["Savings_Probability"] = (
    synthetic_income["Age_2024"]
    .apply(get_savings_probability)
)

synthetic_income["Has_Savings"] = (
    np.random.random(len(synthetic_income))
    < synthetic_income["Savings_Probability"]
).astype(int)

In [140]:
# ------------------------------------------------------------
# 3. CREDIT CARD
# FCA 2024: 65%
# ------------------------------------------------------------

CREDIT_CARD_PROBABILITY = 0.65

synthetic_income["Has_Credit_Card"] = (
    np.random.random(len(synthetic_income))
    < CREDIT_CARD_PROBABILITY
).astype(int)

In [141]:
# ------------------------------------------------------------
# 4. PERSONAL LOAN
# FCA 2024: 14%
# ------------------------------------------------------------

PERSONAL_LOAN_PROBABILITY = 0.14

synthetic_income["Has_Personal_Loan"] = (
    np.random.random(len(synthetic_income))
    < PERSONAL_LOAN_PROBABILITY
).astype(int)

In [142]:
# ------------------------------------------------------------
# 5. MORTGAGE
# FCA 2024 age-specific residential mortgage ownership
# ------------------------------------------------------------

def get_mortgage_probability(age):

    if 18 <= age <= 24:
        return 0.11
    elif 25 <= age <= 34:
        return 0.19
    elif 35 <= age <= 44:
        return 0.26
    elif 45 <= age <= 54:
        return 0.31
    elif 55 <= age <= 64:
        return 0.34
    elif 65 <= age <= 74:
        return 0.36
    elif age >= 75:
        return 0.25
    else:
        return 0.0


synthetic_income["Mortgage_Probability"] = (
    synthetic_income["Age_2024"]
    .apply(get_mortgage_probability)
)

synthetic_income["Has_Mortgage"] = (
    np.random.random(len(synthetic_income))
    < synthetic_income["Mortgage_Probability"]
).astype(int)

In [143]:
# ============================================================
# SUMMARY
# ============================================================

product_summary = pd.DataFrame({
    "Product": [
        "Current Account",
        "Savings",
        "Credit Card",
        "Personal Loan",
        "Mortgage"
    ],
    "Customers": [
        synthetic_income["Has_Current_Account"].sum(),
        synthetic_income["Has_Savings"].sum(),
        synthetic_income["Has_Credit_Card"].sum(),
        synthetic_income["Has_Personal_Loan"].sum(),
        synthetic_income["Has_Mortgage"].sum()
    ]
})

product_summary["Proportion"] = (
    product_summary["Customers"] / len(synthetic_income)
)

display(product_summary)

,Product,Customers,Proportion
0,Current Account,9736,0.9736
1,Savings,8919,0.8919
2,Credit Card,6506,0.6506
3,Personal Loan,1413,0.1413
4,Mortgage,2621,0.2621


In [145]:
# ============================================================
# CELL 62 — FINALISE 5 PRODUCT OWNERSHIP VARIABLES
# ============================================================

product_columns = [
    "Has_Current_Account",
    "Has_Savings",
    "Has_Credit_Card",
    "Has_Personal_Loan",
    "Has_Mortgage"
]

# Check that all five already exist
missing_products = [
    col for col in product_columns
    if col not in synthetic_income.columns
]

if missing_products:
    print("Missing product columns:", missing_products)
else:
    print("All 5 product ownership columns are available.")

# Summary
product_summary = pd.DataFrame({
    "Product": [
        "Current Account",
        "Savings",
        "Credit Card",
        "Personal Loan",
        "Mortgage"
    ],
    "Customers": [
        synthetic_income["Has_Current_Account"].sum(),
        synthetic_income["Has_Savings"].sum(),
        synthetic_income["Has_Credit_Card"].sum(),
        synthetic_income["Has_Personal_Loan"].sum(),
        synthetic_income["Has_Mortgage"].sum()
    ]
})

product_summary["Proportion"] = (
    product_summary["Customers"]
    / len(synthetic_income)
)

display(product_summary)

All 5 product ownership columns are available.


,Product,Customers,Proportion
0,Current Account,9736,0.9736
1,Savings,8919,0.8919
2,Credit Card,6506,0.6506
3,Personal Loan,1413,0.1413
4,Mortgage,2621,0.2621


In [146]:
# ============================================================
# CELL 63 — GENERATE 2024 SAVINGS BALANCE
# ONLY FOR CUSTOMERS WHO HAVE SAVINGS
# ============================================================

np.random.seed(42)

# FCA 2024 median savings bands by age
# We use the midpoint of each reported band
def get_savings_midpoint(age):

    if 18 <= age <= 24:
        return 1500
    elif 25 <= age <= 34:
        return 2500
    elif 35 <= age <= 44:
        return 3500
    elif 45 <= age <= 54:
        return 4500
    elif 55 <= age <= 64:
        return 7500
    elif 65 <= age <= 74:
        return 25000
    elif age >= 75:
        return 40000
    else:
        return 0


# Base amount from FCA age-specific median band
synthetic_income["Savings_Base_2024"] = (
    synthetic_income["Age_2024"]
    .apply(get_savings_midpoint)
)


# Add controlled variation so customers in the
# same age group do not all have exactly the same balance.
variation = np.random.uniform(
    0.75,
    1.25,
    size=len(synthetic_income)
)


synthetic_income["Savings_2024"] = np.where(
    synthetic_income["Has_Savings"] == 1,
    synthetic_income["Savings_Base_2024"] * variation,
    0
)


# Round to nearest £1
synthetic_income["Savings_2024"] = (
    synthetic_income["Savings_2024"]
    .round(0)
)


# ------------------------------------------------------------
# CHECK
# ------------------------------------------------------------

print(
    "Customers with savings:",
    synthetic_income["Has_Savings"].sum()
)

print(
    "Customers without savings:",
    (synthetic_income["Has_Savings"] == 0).sum()
)

print(
    "Savings balance for non-holders:",
    synthetic_income.loc[
        synthetic_income["Has_Savings"] == 0,
        "Savings_2024"
    ].unique()
)

display(
    synthetic_income[
        [
            "Customer_ID",
            "Age_2024",
            "Has_Savings",
            "Savings_2024"
        ]
    ].head(20)
)

Customers with savings: 8919
Customers without savings: 1081
Savings balance for non-holders: [0.]


,Customer_ID,Age_2024,Has_Savings,Savings_2024
0,1,41,1,3280.0
1,2,90,1,49014.0
2,3,58,1,8370.0
3,4,46,1,4722.0
4,5,29,1,2070.0
5,6,27,0,0.0
6,7,24,1,1169.0
7,8,73,1,29577.0
8,9,60,1,7879.0
9,10,55,1,8280.0


In [147]:
# ============================================================
# CELL 63 — INCOME-ADJUSTED FCA SAVINGS BALANCE
# ============================================================

np.random.seed(42)

# ------------------------------------------------------------
# 1. FCA age-based representative savings amounts
# ------------------------------------------------------------

def get_fca_savings_amount(age):

    if 18 <= age <= 24:
        return 1500
    elif 25 <= age <= 34:
        return 2500
    elif 35 <= age <= 44:
        return 3500
    elif 45 <= age <= 54:
        return 4500
    elif 55 <= age <= 64:
        return 7500
    elif 65 <= age <= 74:
        return 25000
    elif age >= 75:
        return 40000
    else:
        return 0


synthetic_income["FCA_Savings_Base"] = (
    synthetic_income["Age_2024"]
    .apply(get_fca_savings_amount)
)


# ------------------------------------------------------------
# 2. Income benchmark
# Use median income among customers who have savings
# ------------------------------------------------------------

savings_holders = synthetic_income[
    synthetic_income["Has_Savings"] == 1
].copy()

income_median = savings_holders[
    "Annual_Income_2024"
].median()

print(
    "Median income of savings holders:",
    round(income_median, 2)
)


# ------------------------------------------------------------
# 3. Income adjustment
#
# sqrt() prevents income from dominating the FCA
# age-based savings distribution.
# ------------------------------------------------------------

income_ratio = (
    synthetic_income["Annual_Income_2024"]
    / income_median
)

income_adjustment = np.sqrt(
    income_ratio.clip(lower=0.25, upper=4.0)
)


# ------------------------------------------------------------
# 4. Add controlled variation
# ------------------------------------------------------------

random_variation = np.random.uniform(
    0.75,
    1.25,
    size=len(synthetic_income)
)


# ------------------------------------------------------------
# 5. Generate Savings_2024
# ------------------------------------------------------------

synthetic_income["Savings_2024"] = np.where(
    synthetic_income["Has_Savings"] == 1,

    synthetic_income["FCA_Savings_Base"]
    * income_adjustment
    * random_variation,

    0
)


# Round to nearest £1
synthetic_income["Savings_2024"] = (
    synthetic_income["Savings_2024"]
    .round(0)
)


# ------------------------------------------------------------
# 6. Remove temporary columns if not needed
# ------------------------------------------------------------

synthetic_income.drop(
    columns=[
        "FCA_Savings_Base"
    ],
    inplace=True
)


# ------------------------------------------------------------
# 7. Check
# ------------------------------------------------------------

display(
    synthetic_income[
        [
            "Customer_ID",
            "Annual_Income_2024",
            "Age_2024",
            "Has_Savings",
            "Savings_2024"
        ]
    ].head(20)
)

Median income of savings holders: 34999.5


,Customer_ID,Annual_Income_2024,Age_2024,Has_Savings,Savings_2024
0,1,24999.5,41,1,2772.0
1,2,199999.5,90,1,98029.0
2,3,44999.5,58,1,9491.0
3,4,34999.5,46,1,4722.0
4,5,12499.5,29,1,1237.0
5,6,12499.5,27,0,0.0
6,7,7499.5,24,1,584.0
7,8,84999.5,73,1,46093.0
8,9,34999.5,60,1,7879.0
9,10,44999.5,55,1,9389.0


In [148]:
# ============================================================
# CELL 64 — GENERATE HISTORICAL SAVINGS 2008–2025
# ============================================================

# Make sure BoE savings data is ordered
boe_savings_growth = boe_savings[
    [
        "Date",
        "Savings_Accounts",
        "Savings_Monthly_Growth"
    ]
].copy()

boe_savings_growth["Date"] = pd.to_datetime(
    boe_savings_growth["Date"]
)

boe_savings_growth = (
    boe_savings_growth
    .sort_values("Date")
    .reset_index(drop=True)
)

# Keep 2008–2025
boe_savings_growth = boe_savings_growth[
    (boe_savings_growth["Date"] >= "2008-01-01") &
    (boe_savings_growth["Date"] <= "2025-12-01")
].copy()

print(
    "BoE savings period:",
    boe_savings_growth["Date"].min(),
    "to",
    boe_savings_growth["Date"].max()
)

print(
    "Number of months:",
    len(boe_savings_growth)
)

BoE savings period: 2008-01-01 00:00:00 to 2025-12-01 00:00:00
Number of months: 216


In [149]:
# ============================================================
# CELL 64B — CUSTOMER SAVINGS HISTORY
# ============================================================

# Start from the existing 2024 customer dataset
savings_customers = synthetic_income[
    [
        "Customer_ID",
        "Annual_Income_2024",
        "Age_2024",
        "Has_Savings",
        "Savings_2024"
    ]
].copy()

# Only customers who actually have savings
savings_customers = savings_customers[
    savings_customers["Has_Savings"] == 1
].copy()

print(
    "Savings customers:",
    len(savings_customers)
)

Savings customers: 8919


In [153]:
# ============================================================
# CELL 64C — CORRECT SAVINGS HISTORY
# ============================================================

historical_rows = []

# We use 2024 as the customer baseline
# because Savings_2024 was generated for each customer.

boe_savings_growth = boe_savings_growth.copy()

# Split historical period and future period
# Baseline = December 2024
baseline_date = pd.Timestamp("2024-12-01")

pre_2024 = boe_savings_growth[
    boe_savings_growth["Date"] < baseline_date
].copy()

post_2024 = boe_savings_growth[
    boe_savings_growth["Date"] > baseline_date
].copy()

baseline_row = boe_savings_growth[
    boe_savings_growth["Date"] == baseline_date
]

print("Pre-2024 months:", len(pre_2024))
print("Post-2024 months:", len(post_2024))

Pre-2024 months: 203
Post-2024 months: 12


In [154]:
# ============================================================
# CELL 64D — BUILD CORRECT CUSTOMER SAVINGS TRAJECTORY
# ============================================================

for _, customer in savings_customers.iterrows():

    customer_id = customer["Customer_ID"]
    savings_2024 = customer["Savings_2024"]

    customer_history = {}

    # --------------------------------------------------------
    # 1. Set December 2024 baseline
    # --------------------------------------------------------

    customer_history[baseline_date] = savings_2024

    # --------------------------------------------------------
    # 2. BACKWARD: 2024 → 2008
    # --------------------------------------------------------

    current_balance = savings_2024

    dates_backward = (
        pre_2024["Date"]
        .sort_values(ascending=False)
        .tolist()
    )

    for date in dates_backward:

        # Growth belonging to the NEXT month
        next_date = date + pd.DateOffset(months=1)

        growth_row = boe_savings_growth[
            boe_savings_growth["Date"] == next_date
        ]

        if len(growth_row) > 0:

            growth = growth_row[
                "Savings_Monthly_Growth"
            ].iloc[0]

            if pd.notna(growth):

                current_balance = (
                    current_balance / (1 + growth)
                )

        customer_history[date] = current_balance

    # --------------------------------------------------------
    # 3. FORWARD: 2024 → 2025
    # --------------------------------------------------------

    current_balance = savings_2024

    for _, row in post_2024.sort_values("Date").iterrows():

        growth = row["Savings_Monthly_Growth"]

        if pd.notna(growth):

            current_balance = (
                current_balance * (1 + growth)
            )

        customer_history[row["Date"]] = current_balance

    # --------------------------------------------------------
    # 4. Store customer history
    # --------------------------------------------------------

    for date, balance in sorted(customer_history.items()):

        historical_rows.append({
            "Customer_ID": customer_id,
            "Date": date,
            "Savings_Balance": balance
        })


savings_history = pd.DataFrame(
    historical_rows
)

print(
    "Savings history rows:",
    len(savings_history)
)

print(
    "Expected rows:",
    8919 * 216
)

display(
    savings_history.head(20)
)

Savings history rows: 1926504
Expected rows: 1926504


,Customer_ID,Date,Savings_Balance
0,1.0,2008-01-01,2303.583999
1,1.0,2008-02-01,2330.436658
2,1.0,2008-03-01,2340.682390
3,1.0,2008-04-01,2402.310936
4,1.0,2008-05-01,2443.972121
5,1.0,2008-06-01,2478.542478
6,1.0,2008-07-01,2524.314288
7,1.0,2008-08-01,2561.505168
8,1.0,2008-09-01,2582.325484
9,1.0,2008-10-01,2661.362527


In [155]:
# ============================================================
# CELL 66 — CREATE 2024 CURRENT ACCOUNT BALANCE
# ONLY FOR CURRENT-ACCOUNT HOLDERS
# ============================================================

np.random.seed(42)

current_account_customers = synthetic_income[
    synthetic_income["Has_Current_Account"] == 1
].copy()

print(
    "Current account customers:",
    len(current_account_customers)
)

# ------------------------------------------------------------
# Income-based starting balance
# ------------------------------------------------------------

CURRENT_ACCOUNT_RATE = 0.05

random_variation = np.random.uniform(
    0.75,
    1.25,
    size=len(current_account_customers)
)

current_account_customers["Current_Account_2024"] = (
    current_account_customers["Annual_Income_2024"]
    * CURRENT_ACCOUNT_RATE
    * random_variation
).round(0)

# ------------------------------------------------------------
# Add £0 for customers without current accounts
# ------------------------------------------------------------

synthetic_income["Current_Account_2024"] = 0.0

synthetic_income.loc[
    synthetic_income["Has_Current_Account"] == 1,
    "Current_Account_2024"
] = current_account_customers[
    "Current_Account_2024"
].values

# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

display(
    synthetic_income[
        [
            "Customer_ID",
            "Annual_Income_2024",
            "Has_Current_Account",
            "Current_Account_2024"
        ]
    ].head(20)
)

Current account customers: 9736


,Customer_ID,Annual_Income_2024,Has_Current_Account,Current_Account_2024
0,1,24999.5,1,1172.0
1,2,199999.5,1,12254.0
2,3,44999.5,1,2511.0
3,4,34999.5,1,1836.0
4,5,12499.5,1,517.0
5,6,12499.5,1,517.0
6,7,7499.5,1,292.0
7,8,84999.5,1,5028.0
8,9,34999.5,1,1838.0
9,10,44999.5,1,2484.0


In [156]:
# ============================================================
# CELL 67 — PREPARE BOE CURRENT ACCOUNT GROWTH
# ============================================================

boe_current_growth = boe_final[
    [
        "Date",
        "Current_Accounts"
    ]
].copy()

boe_current_growth["Date"] = pd.to_datetime(
    boe_current_growth["Date"]
)

boe_current_growth = (
    boe_current_growth
    .sort_values("Date")
    .reset_index(drop=True)
)

# Keep required period
boe_current_growth = boe_current_growth[
    (boe_current_growth["Date"] >= "2008-01-01") &
    (boe_current_growth["Date"] <= "2025-12-01")
].copy()

# Calculate monthly growth
boe_current_growth["Current_Account_Monthly_Growth"] = (
    boe_current_growth["Current_Accounts"].pct_change()
)

print(
    "BoE current account period:",
    boe_current_growth["Date"].min(),
    "to",
    boe_current_growth["Date"].max()
)

print(
    "Number of months:",
    len(boe_current_growth)
)

display(
    boe_current_growth.head(12)
)

BoE current account period: 2008-01-01 00:00:00 to 2025-12-01 00:00:00
Number of months: 216


,Date,Current_Accounts,Current_Account_Monthly_Growth
0,2008-01-01,33417,NaN
1,2008-02-01,31492,-0.057605
2,2008-03-01,32823,0.042265
3,2008-04-01,31895,-0.028273
4,2008-05-01,32210,0.009876
5,2008-06-01,33348,0.035331
6,2008-07-01,46071,0.381522
7,2008-08-01,44578,-0.032407
8,2008-09-01,48091,0.078806
9,2008-10-01,44654,-0.071469


In [157]:
# ============================================================
# CELL 68 — GENERATE CURRENT ACCOUNT HISTORY 2008–2025
# ============================================================

current_account_customers = synthetic_income[
    synthetic_income["Has_Current_Account"] == 1
][
    [
        "Customer_ID",
        "Annual_Income_2024",
        "Age_2024",
        "Has_Current_Account",
        "Current_Account_2024"
    ]
].copy()

print(
    "Current account customers:",
    len(current_account_customers)
)

# 2024 baseline
baseline_date = pd.Timestamp("2024-12-01")

# Store historical rows
current_account_rows = []

# Dates before baseline
pre_2024 = boe_current_growth[
    boe_current_growth["Date"] < baseline_date
].copy()

# Dates after baseline
post_2024 = boe_current_growth[
    boe_current_growth["Date"] > baseline_date
].copy()


for _, customer in current_account_customers.iterrows():

    customer_id = customer["Customer_ID"]
    balance_2024 = customer["Current_Account_2024"]

    customer_history = {}

    # --------------------------------------------------------
    # December 2024 baseline
    # --------------------------------------------------------

    customer_history[baseline_date] = balance_2024

    # --------------------------------------------------------
    # BACKWARD: December 2024 → January 2008
    # --------------------------------------------------------

    current_balance = balance_2024

    dates_backward = (
        pre_2024["Date"]
        .sort_values(ascending=False)
        .tolist()
    )

    for date in dates_backward:

        next_date = date + pd.DateOffset(months=1)

        growth_row = boe_current_growth[
            boe_current_growth["Date"] == next_date
        ]

        if len(growth_row) > 0:

            growth = growth_row[
                "Current_Account_Monthly_Growth"
            ].iloc[0]

            if pd.notna(growth):

                current_balance = (
                    current_balance / (1 + growth)
                )

        customer_history[date] = current_balance

    # --------------------------------------------------------
    # FORWARD: December 2024 → December 2025
    # --------------------------------------------------------

    current_balance = balance_2024

    for _, row in post_2024.sort_values("Date").iterrows():

        growth = row[
            "Current_Account_Monthly_Growth"
        ]

        if pd.notna(growth):

            current_balance = (
                current_balance * (1 + growth)
            )

        customer_history[row["Date"]] = current_balance

    # --------------------------------------------------------
    # Store
    # --------------------------------------------------------

    for date, balance in sorted(customer_history.items()):

        current_account_rows.append({
            "Customer_ID": customer_id,
            "Date": date,
            "Current_Account_Balance": balance
        })


current_account_history = pd.DataFrame(
    current_account_rows
)

print(
    "Current account history rows:",
    len(current_account_history)
)

print(
    "Expected rows:",
    9736 * 216
)

print(
    "Unique customers:",
    current_account_history["Customer_ID"].nunique()
)

print(
    "Unique months:",
    current_account_history["Date"].nunique()
)

display(
    current_account_history.head(20)
)

Current account customers: 9736
Current account history rows: 2102976
Expected rows: 2102976
Unique customers: 9736
Unique months: 216


,Customer_ID,Date,Current_Account_Balance
0,1.0,2008-01-01,129.955616
1,1.0,2008-02-01,122.469469
2,1.0,2008-03-01,127.645605
3,1.0,2008-04-01,124.036699
4,1.0,2008-05-01,125.261705
5,1.0,2008-06-01,129.687281
6,1.0,2008-07-01,179.165849
7,1.0,2008-08-01,173.359711
8,1.0,2008-09-01,187.021442
9,1.0,2008-10-01,173.655268


In [165]:
# ============================================================
# CELL 70 — CREDIT CARD FLAGS
# USE EXISTING synthetic_income DATAFRAME
# ============================================================

# Check current columns
print("Current columns:")
print(synthetic_income.columns.tolist())


# ------------------------------------------------------------
# Check that the existing credit-card ownership column exists
# ------------------------------------------------------------

if "Has_Credit_Card" not in synthetic_income.columns:

    raise KeyError(
        "Has_Credit_Card is not in synthetic_income. "
        "Send me synthetic_income.columns.tolist()"
    )


# ------------------------------------------------------------
# Credit-card holders already created:
# 6,506 out of 10,000
# ------------------------------------------------------------

credit_card_holders = synthetic_income[
    synthetic_income["Has_Credit_Card"] == 1
].copy()

print(
    "Credit card holders:",
    len(credit_card_holders)
)


# ------------------------------------------------------------
# FCA evidence:
# 35.3m credit-card holders
# 10.1m revolvers
#
# Revolver proportion among card holders
# = 10.1 / 35.3
# ------------------------------------------------------------

REVOLVER_PROPORTION = 10.1 / 35.3

print(
    "FCA revolver proportion:",
    round(REVOLVER_PROPORTION, 4)
)


# ------------------------------------------------------------
# Create revolver flag ONLY for credit-card holders
# ------------------------------------------------------------

np.random.seed(42)

synthetic_income["Credit_Card_Revolver"] = 0

n_revolvers = round(
    len(credit_card_holders) * REVOLVER_PROPORTION
)

revolver_ids = np.random.choice(
    credit_card_holders.index,
    size=n_revolvers,
    replace=False
)

synthetic_income.loc[
    revolver_ids,
    "Credit_Card_Revolver"
] = 1


# ------------------------------------------------------------
# CHECK
# ------------------------------------------------------------

print("=" * 60)
print("CREDIT CARD SUMMARY")
print("=" * 60)

print(
    "Total customers:",
    len(synthetic_income)
)

print(
    "Credit card holders:",
    synthetic_income["Has_Credit_Card"].sum()
)

print(
    "Credit card revolvers:",
    synthetic_income["Credit_Card_Revolver"].sum()
)

print(
    "Non-revolvers:",
    (
        (synthetic_income["Has_Credit_Card"] == 1) &
        (synthetic_income["Credit_Card_Revolver"] == 0)
    ).sum()
)

print(
    "Revolver proportion among card holders:",
    round(
        synthetic_income.loc[
            synthetic_income["Has_Credit_Card"] == 1,
            "Credit_Card_Revolver"
        ].mean(),
        4
    )
)


display(
    synthetic_income[
        [
            "Customer_ID",
            "Annual_Income_2024",
            "Age_2024",
            "Has_Credit_Card",
            "Credit_Card_Revolver"
        ]
    ].head(20)
)

Current columns:
['Customer_ID', 'Income_Band', 'Annual_Income_2024', 'Age_Group', 'Age_2024', 'Birth_Year', 'Savings_2024', 'Has_Current_Account', 'Savings_Probability', 'Has_Savings', 'Has_Credit_Card', 'Has_Personal_Loan', 'Mortgage_Probability', 'Has_Mortgage', 'Current_Account_2024', 'Savings_Base_2024']
Credit card holders: 6506
FCA revolver proportion: 0.2861
CREDIT CARD SUMMARY
Total customers: 10000
Credit card holders: 6506
Credit card revolvers: 1861
Non-revolvers: 4645
Revolver proportion among card holders: 0.286


,Customer_ID,Annual_Income_2024,Age_2024,Has_Credit_Card,Credit_Card_Revolver
0,1,24999.5,41,1,0
1,2,199999.5,90,1,0
2,3,44999.5,58,0,0
3,4,34999.5,46,1,0
4,5,12499.5,29,1,0
5,6,12499.5,27,0,0
6,7,7499.5,24,0,0
7,8,84999.5,73,1,0
8,9,34999.5,60,0,0
9,10,44999.5,55,1,0


In [166]:
# ============================================================
# CELL 71 — INSPECT BOE CREDIT CARD LENDING
# ============================================================

credit_card_boe = boe_final[
    [
        "Date",
        "Credit_Card_Lending"
    ]
].copy()

credit_card_boe["Date"] = pd.to_datetime(
    credit_card_boe["Date"]
)

credit_card_boe = (
    credit_card_boe
    .sort_values("Date")
    .reset_index(drop=True)
)

credit_card_boe = credit_card_boe[
    (credit_card_boe["Date"] >= "2008-01-01") &
    (credit_card_boe["Date"] <= "2025-12-01")
].copy()

print(
    "Credit-card BoE period:",
    credit_card_boe["Date"].min(),
    "to",
    credit_card_boe["Date"].max()
)

print(
    "Number of observations:",
    len(credit_card_boe)
)

print("\n2024 observations:")

display(
    credit_card_boe[
        credit_card_boe["Date"].dt.year == 2024
    ]
)

Credit-card BoE period: 2008-01-01 00:00:00 to 2025-12-01 00:00:00
Number of observations: 216

2024 observations:


,Date,Credit_Card_Lending
192,2024-01-01,619
193,2024-02-01,461
194,2024-03-01,718
195,2024-04-01,59
196,2024-05-01,721
197,2024-06-01,411
198,2024-07-01,483
199,2024-08-01,481
200,2024-09-01,429
201,2024-10-01,562


In [167]:
# ============================================================
# CELL 71 — PREPARE BOE CREDIT CARD OUTSTANDING
# ============================================================

# Find the column containing LPMBC53
credit_card_column = [
    col for col in boe_df.columns
    if "LPMBC53" in col
][0]

print("Credit Card column found:")
print(credit_card_column)


# ------------------------------------------------------------
# Create clean dataframe
# ------------------------------------------------------------

credit_card_boe = boe_df[
    [
        "Date",
        credit_card_column
    ]
].copy()


# ------------------------------------------------------------
# Convert data types
# ------------------------------------------------------------

credit_card_boe["Date"] = pd.to_datetime(
    credit_card_boe["Date"],
    errors="coerce"
)

credit_card_boe["Credit_Card_Outstanding"] = pd.to_numeric(
    credit_card_boe[credit_card_column],
    errors="coerce"
)


# ------------------------------------------------------------
# Keep required columns
# ------------------------------------------------------------

credit_card_boe = credit_card_boe[
    [
        "Date",
        "Credit_Card_Outstanding"
    ]
].copy()


# ------------------------------------------------------------
# Sort
# ------------------------------------------------------------

credit_card_boe = (
    credit_card_boe
    .sort_values("Date")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Keep 2008–2025
# ------------------------------------------------------------

credit_card_boe = credit_card_boe[
    (credit_card_boe["Date"] >= "2008-01-01") &
    (credit_card_boe["Date"] <= "2025-12-01")
].copy()


# ------------------------------------------------------------
# Calculate monthly growth
# ------------------------------------------------------------

credit_card_boe["Credit_Card_Monthly_Growth"] = (
    credit_card_boe["Credit_Card_Outstanding"]
    .pct_change()
)


# ------------------------------------------------------------
# CHECK
# ------------------------------------------------------------

print(
    "Credit card period:",
    credit_card_boe["Date"].min().strftime("%Y-%m"),
    "to",
    credit_card_boe["Date"].max().strftime("%Y-%m")
)

print(
    "Credit card observations:",
    len(credit_card_boe)
)

print(
    "Missing outstanding values:",
    credit_card_boe["Credit_Card_Outstanding"].isna().sum()
)

display(
    credit_card_boe.head(12)
)

Credit Card column found:
Monthly amounts outstanding of monetary financial institutions' sterling net credit card lending to individuals (in sterling millions) seasonally adjusted  						[a] [n] [u] [o] [v] [s] [e] [w] [g] 						LPMBC53
Credit card period: 2008-01 to 2025-11
Credit card observations: 215
Missing outstanding values: 0


,Date,Credit_Card_Outstanding,Credit_Card_Monthly_Growth
0,2008-01-31,54232,NaN
1,2008-02-29,54375,0.002637
2,2008-03-31,54471,0.001766
3,2008-04-30,54404,-0.001230
4,2008-05-31,55211,0.014833
5,2008-06-30,54955,-0.004637
6,2008-07-31,54977,0.000400
7,2008-08-31,55432,0.008276
8,2008-09-30,56534,0.019880
9,2008-10-31,53894,-0.046698


In [168]:
# ============================================================
# CELL 72 — GET 2024 BOE CREDIT CARD OUTSTANDING
# ============================================================

credit_card_2024_boe = credit_card_boe[
    credit_card_boe["Date"] == "2024-12-31"
].copy()

display(credit_card_2024_boe)

boe_2024_credit_card_millions = (
    credit_card_2024_boe[
        "Credit_Card_Outstanding"
    ].iloc[0]
)

print(
    "BoE December 2024 credit-card outstanding (£m):",
    boe_2024_credit_card_millions
)

print(
    "BoE December 2024 credit-card outstanding (£):",
    boe_2024_credit_card_millions * 1_000_000
)

,Date,Credit_Card_Outstanding,Credit_Card_Monthly_Growth
203,2024-12-31,52675,0.00692


BoE December 2024 credit-card outstanding (£m): 52675
BoE December 2024 credit-card outstanding (£): 52675000000


In [169]:
# ============================================================
# CELL 73 — INITIAL 2024 CREDIT-CARD BALANCES
# ============================================================

np.random.seed(42)

# Only customers who:
# 1. Have a credit card
# 2. Revolve a balance

credit_card_revolvers = synthetic_income[
    synthetic_income["Credit_Card_Revolver"] == 1
].copy()

print(
    "Credit-card revolvers:",
    len(credit_card_revolvers)
)

# ------------------------------------------------------------
# Income-based initial balance
# ------------------------------------------------------------

income = credit_card_revolvers["Annual_Income_2024"]

# Higher income -> potentially higher credit-card balance
income_factor = np.sqrt(
    income / income.median()
)

# Customer-level variation
random_factor = np.random.lognormal(
    mean=0,
    sigma=0.45,
    size=len(credit_card_revolvers)
)

# Initial balance
credit_card_revolvers["Credit_Card_2024_Raw"] = (
    income_factor
    * random_factor
    * income.median()
    * 0.10
)

# Ensure positive
credit_card_revolvers["Credit_Card_2024_Raw"] = (
    credit_card_revolvers["Credit_Card_2024_Raw"]
    .clip(lower=100)
)

display(
    credit_card_revolvers[
        [
            "Customer_ID",
            "Annual_Income_2024",
            "Credit_Card_Revolver",
            "Credit_Card_2024_Raw"
        ]
    ].head(20)
)

Credit-card revolvers: 1861


,Customer_ID,Annual_Income_2024,Credit_Card_Revolver,Credit_Card_2024_Raw
16,17,24999.5,1,3126.118379
22,23,24999.5,1,2349.145855
26,27,17499.5,1,2799.354391
28,29,34999.5,1,5870.108647
30,31,34999.5,1,2662.167628
33,34,124999.5,1,5031.086966
39,40,24999.5,1,5088.161153
45,46,44999.5,1,4737.521633
46,47,24999.5,1,2023.861374
48,49,34999.5,1,3775.986703


In [170]:
# ============================================================
# CELL 73B — CALIBRATE CREDIT CARD BALANCES TO BOE
# ============================================================

boe_target = (
    credit_card_2024_boe[
        "Credit_Card_Outstanding"
    ].iloc[0]
    * 1_000_000
)

synthetic_raw_total = (
    credit_card_revolvers[
        "Credit_Card_2024_Raw"
    ].sum()
)

calibration_factor = (
    boe_target / synthetic_raw_total
)

print(
    "BoE target (£):",
    round(boe_target, 2)
)

print(
    "Raw synthetic total (£):",
    round(synthetic_raw_total, 2)
)

print(
    "Calibration factor:",
    round(calibration_factor, 6)
)

credit_card_revolvers["Credit_Card_2024"] = (
    credit_card_revolvers["Credit_Card_2024_Raw"]
    * calibration_factor
).round(2)

print(
    "Calibrated synthetic total (£):",
    round(
        credit_card_revolvers["Credit_Card_2024"].sum(),
        2
    )
)

display(
    credit_card_revolvers[
        [
            "Customer_ID",
            "Annual_Income_2024",
            "Credit_Card_2024_Raw",
            "Credit_Card_2024"
        ]
    ].head(20)
)

BoE target (£): 52675000000
Raw synthetic total (£): 6544318.04
Calibration factor: 8048.967013
Calibrated synthetic total (£): 52675000000.07


,Customer_ID,Annual_Income_2024,Credit_Card_2024_Raw,Credit_Card_2024
16,17,24999.5,3126.118379,25162023.71
22,23,24999.5,2349.145855,18908197.50
26,27,17499.5,2799.354391,22531911.15
28,29,34999.5,5870.108647,47248310.87
30,31,34999.5,2662.167628,21427699.42
33,34,124999.5,5031.086966,40495053.03
39,40,24999.5,5088.161153,40954441.28
45,46,44999.5,4737.521633,38132155.35
46,47,24999.5,2023.861374,16289993.44
48,49,34999.5,3775.986703,30392792.41


In [171]:
# ============================================================
# CELL 73B — CORRECT CREDIT CARD CALIBRATION
# ============================================================

# ------------------------------------------------------------
# 1. BoE December 2024 outstanding credit-card lending
# ------------------------------------------------------------

boe_target_uk = (
    credit_card_2024_boe[
        "Credit_Card_Outstanding"
    ].iloc[0]
    * 1_000_000
)

print(
    "BoE UK credit-card outstanding (£):",
    round(boe_target_uk, 2)
)


# ------------------------------------------------------------
# 2. FCA 2024 UK revolver population
# ------------------------------------------------------------

FCA_UK_REVOLVERS = 10.1 * 1_000_000

print(
    "FCA UK credit-card revolvers:",
    FCA_UK_REVOLVERS
)


# ------------------------------------------------------------
# 3. Average outstanding balance per UK revolver
# ------------------------------------------------------------

average_revolver_balance = (
    boe_target_uk /
    FCA_UK_REVOLVERS
)

print(
    "Estimated average balance per UK revolver (£):",
    round(average_revolver_balance, 2)
)


# ------------------------------------------------------------
# 4. Synthetic revolver population
# ------------------------------------------------------------

synthetic_revolver_count = (
    synthetic_income["Credit_Card_Revolver"].sum()
)

print(
    "Synthetic revolvers:",
    synthetic_revolver_count
)


# ------------------------------------------------------------
# 5. Target synthetic aggregate
# ------------------------------------------------------------

synthetic_target = (
    average_revolver_balance
    * synthetic_revolver_count
)

print(
    "Synthetic target credit-card balance (£):",
    round(synthetic_target, 2)
)


# ------------------------------------------------------------
# 6. Current raw synthetic total
# ------------------------------------------------------------

synthetic_raw_total = (
    credit_card_revolvers[
        "Credit_Card_2024_Raw"
    ].sum()
)

print(
    "Raw synthetic total (£):",
    round(synthetic_raw_total, 2)
)


# ------------------------------------------------------------
# 7. Calibration factor
# ------------------------------------------------------------

calibration_factor = (
    synthetic_target /
    synthetic_raw_total
)

print(
    "Calibration factor:",
    round(calibration_factor, 6)
)


# ------------------------------------------------------------
# 8. Apply calibration
# ------------------------------------------------------------

credit_card_revolvers["Credit_Card_2024"] = (
    credit_card_revolvers["Credit_Card_2024_Raw"]
    * calibration_factor
).round(2)


# ------------------------------------------------------------
# 9. Validate
# ------------------------------------------------------------

calibrated_total = (
    credit_card_revolvers[
        "Credit_Card_2024"
    ].sum()
)

print(
    "Calibrated synthetic total (£):",
    round(calibrated_total, 2)
)

print(
    "Target (£):",
    round(synthetic_target, 2)
)


# ------------------------------------------------------------
# 10. Display
# ------------------------------------------------------------

display(
    credit_card_revolvers[
        [
            "Customer_ID",
            "Annual_Income_2024",
            "Credit_Card_2024_Raw",
            "Credit_Card_2024"
        ]
    ].head(20)
)

BoE UK credit-card outstanding (£): 52675000000
FCA UK credit-card revolvers: 10100000.0
Estimated average balance per UK revolver (£): 5215.35
Synthetic revolvers: 1861
Synthetic target credit-card balance (£): 9705759.9
Raw synthetic total (£): 6544318.04
Calibration factor: 1.483082
Calibrated synthetic total (£): 9705760.05
Target (£): 9705759.9


,Customer_ID,Annual_Income_2024,Credit_Card_2024_Raw,Credit_Card_2024
16,17,24999.5,3126.118379,4636.29
22,23,24999.5,2349.145855,3483.98
26,27,17499.5,2799.354391,4151.67
28,29,34999.5,5870.108647,8705.85
30,31,34999.5,2662.167628,3948.21
33,34,124999.5,5031.086966,7461.51
39,40,24999.5,5088.161153,7546.16
45,46,44999.5,4737.521633,7026.13
46,47,24999.5,2023.861374,3001.55
48,49,34999.5,3775.986703,5600.10


In [172]:
# ============================================================
# CELL 73C — ADD CALIBRATED CREDIT CARD BALANCES
# ============================================================

synthetic_income["Credit_Card_2024"] = 0.0

synthetic_income.loc[
    credit_card_revolvers.index,
    "Credit_Card_2024"
] = credit_card_revolvers["Credit_Card_2024"]

print(
    "Credit-card holders:",
    synthetic_income["Has_Credit_Card"].sum()
)

print(
    "Credit-card revolvers:",
    synthetic_income["Credit_Card_Revolver"].sum()
)

print(
    "Customers with positive balance:",
    (synthetic_income["Credit_Card_2024"] > 0).sum()
)

print(
    "Total synthetic credit-card balance (£):",
    round(
        synthetic_income["Credit_Card_2024"].sum(),
        2
    )
)

display(
    synthetic_income[
        [
            "Customer_ID",
            "Annual_Income_2024",
            "Has_Credit_Card",
            "Credit_Card_Revolver",
            "Credit_Card_2024"
        ]
    ].head(20)
)

Credit-card holders: 6506
Credit-card revolvers: 1861
Customers with positive balance: 1861
Total synthetic credit-card balance (£): 9705760.05


,Customer_ID,Annual_Income_2024,Has_Credit_Card,Credit_Card_Revolver,Credit_Card_2024
0,1,24999.5,1,0,0.00
1,2,199999.5,1,0,0.00
2,3,44999.5,0,0,0.00
3,4,34999.5,1,0,0.00
4,5,12499.5,1,0,0.00
5,6,12499.5,0,0,0.00
6,7,7499.5,0,0,0.00
7,8,84999.5,1,0,0.00
8,9,34999.5,0,0,0.00
9,10,44999.5,1,0,0.00


In [223]:
# ============================================================
# CELL 74A — ADD ACTUAL DECEMBER 2025 CREDIT CARD DATA
# ============================================================

dec_2025_credit_card = pd.DataFrame({
    "Date": [
        pd.Timestamp("2025-12-31")
    ],
    "Credit_Card_Outstanding": [
        57066
    ]
})

# ------------------------------------------------------------
# Calculate December 2025 growth from November 2025
# ------------------------------------------------------------

nov_2025_value = credit_card_boe.loc[
    credit_card_boe["Date"] == pd.Timestamp("2025-11-30"),
    "Credit_Card_Outstanding"
].iloc[0]

dec_2025_value = 57066

dec_2025_growth = (
    dec_2025_value / nov_2025_value
) - 1

dec_2025_row = pd.DataFrame({
    "Date": [
        pd.Timestamp("2025-12-31")
    ],
    "Credit_Card_Outstanding": [
        dec_2025_value
    ],
    "Credit_Card_Monthly_Growth": [
        dec_2025_growth
    ]
})

# ------------------------------------------------------------
# Add to BoE dataset
# ------------------------------------------------------------

credit_card_boe = pd.concat(
    [
        credit_card_boe,
        dec_2025_row
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# Sort and validate
# ------------------------------------------------------------

credit_card_boe = (
    credit_card_boe
    .sort_values("Date")
    .reset_index(drop=True)
)

print(
    "Credit-card period:",
    credit_card_boe["Date"].min(),
    "to",
    credit_card_boe["Date"].max()
)

print(
    "Credit-card observations:",
    len(credit_card_boe)
)

display(
    credit_card_boe.tail(3)
)

Credit-card period: 2008-01-31 00:00:00 to 2025-12-31 00:00:00
Credit-card observations: 217


,Date,Credit_Card_Outstanding,Credit_Card_Monthly_Growth
214,2025-11-30,56567,0.006065
215,2025-12-31,57066,0.008821
216,2025-12-31,57066,0.008821


In [224]:
# ============================================================
# CELL 74A — REMOVE DUPLICATE CREDIT CARD DATES
# ============================================================

# Make sure Date is datetime
credit_card_boe["Date"] = pd.to_datetime(
    credit_card_boe["Date"]
)

# Check duplicates BEFORE removing them
duplicate_dates = (
    credit_card_boe[
        credit_card_boe["Date"].duplicated(keep=False)
    ]
    .sort_values("Date")
)

print("Duplicate dates:")
display(duplicate_dates)


# ------------------------------------------------------------
# Keep only one observation per Date
# ------------------------------------------------------------

credit_card_boe = (
    credit_card_boe
    .sort_values("Date")
    .drop_duplicates(
        subset=["Date"],
        keep="last"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

print("=" * 70)

print(
    "Credit-card observations:",
    len(credit_card_boe)
)

print(
    "Unique dates:",
    credit_card_boe["Date"].nunique()
)

print(
    "Start:",
    credit_card_boe["Date"].min()
)

print(
    "End:",
    credit_card_boe["Date"].max()
)

print("=" * 70)

display(
    credit_card_boe.tail(5)
)

Duplicate dates:


,Date,Credit_Card_Outstanding,Credit_Card_Monthly_Growth
215,2025-12-31,57066,0.008821
216,2025-12-31,57066,0.008821


Credit-card observations: 216
Unique dates: 216
Start: 2008-01-31 00:00:00
End: 2025-12-31 00:00:00


,Date,Credit_Card_Outstanding,Credit_Card_Monthly_Growth
211,2025-08-31,55569,0.006721
212,2025-09-30,55824,0.004589
213,2025-10-31,56226,0.007201
214,2025-11-30,56567,0.006065
215,2025-12-31,57066,0.008821


In [225]:
# ============================================================
# CELL 74 — GENERATE CREDIT CARD HISTORY 2008–2025
# ============================================================

# ------------------------------------------------------------
# 1. CLEAN / VALIDATE BoE CREDIT CARD SERIES
# ------------------------------------------------------------

credit_card_boe["Date"] = pd.to_datetime(
    credit_card_boe["Date"]
)

# Remove duplicate dates
credit_card_boe = (
    credit_card_boe
    .sort_values("Date")
    .drop_duplicates(
        subset=["Date"],
        keep="last"
    )
    .reset_index(drop=True)
)

print("=" * 70)
print("BoE CREDIT CARD SERIES")
print("=" * 70)

print(
    "Period:",
    credit_card_boe["Date"].min(),
    "to",
    credit_card_boe["Date"].max()
)

print(
    "Observations:",
    len(credit_card_boe)
)

print(
    "Unique dates:",
    credit_card_boe["Date"].nunique()
)

display(
    credit_card_boe.tail(5)
)


# ============================================================
# 2. CREDIT CARD REVOLVER CUSTOMERS
# ============================================================

credit_card_customers = synthetic_income[
    synthetic_income["Credit_Card_Revolver"] == 1
][
    [
        "Customer_ID",
        "Annual_Income_2024",
        "Age_2024",
        "Credit_Card_Revolver",
        "Credit_Card_2024"
    ]
].copy()

print("=" * 70)

print(
    "Credit-card revolvers:",
    len(credit_card_customers)
)


# ============================================================
# 3. BASELINE
# ============================================================

baseline_date = pd.Timestamp(
    "2024-12-31"
)


# ============================================================
# 4. SPLIT BOE DATA
# ============================================================

pre_2024 = credit_card_boe[
    credit_card_boe["Date"] < baseline_date
].copy()

post_2024 = credit_card_boe[
    credit_card_boe["Date"] > baseline_date
].copy()


# ============================================================
# 5. GENERATE CUSTOMER HISTORY
# ============================================================

credit_card_rows = []


for _, customer in credit_card_customers.iterrows():

    customer_id = customer["Customer_ID"]

    balance_2024 = customer["Credit_Card_2024"]

    customer_history = {}


    # --------------------------------------------------------
    # DECEMBER 2024 BASELINE
    # --------------------------------------------------------

    customer_history[
        baseline_date
    ] = balance_2024


    # ========================================================
    # BACKWARD: DEC-2024 → JAN-2008
    # ========================================================

    current_balance = balance_2024

    dates_backward = (
        pre_2024["Date"]
        .sort_values(
            ascending=False
        )
        .tolist()
    )

    for date in dates_backward:

        next_date = (
            date +
            pd.offsets.MonthEnd(1)
        )

        growth_row = credit_card_boe[
            credit_card_boe["Date"] == next_date
        ]

        if len(growth_row) > 0:

            growth = growth_row[
                "Credit_Card_Monthly_Growth"
            ].iloc[0]

            if pd.notna(growth):

                current_balance = (
                    current_balance /
                    (1 + growth)
                )

        customer_history[
            date
        ] = current_balance


    # ========================================================
    # FORWARD: DEC-2024 → DEC-2025
    # ========================================================

    current_balance = balance_2024

    for _, row in post_2024.sort_values(
        "Date"
    ).iterrows():

        growth = row[
            "Credit_Card_Monthly_Growth"
        ]

        if pd.notna(growth):

            current_balance = (
                current_balance *
                (1 + growth)
            )

        customer_history[
            row["Date"]
        ] = current_balance


    # ========================================================
    # STORE CUSTOMER HISTORY
    # ========================================================

    for date, balance in sorted(
        customer_history.items()
    ):

        credit_card_rows.append({

            "Customer_ID":
                customer_id,

            "Date":
                date,

            "Credit_Card_Balance":
                balance

        })


# ============================================================
# 6. CREATE DATAFRAME
# ============================================================

credit_card_history = pd.DataFrame(
    credit_card_rows
)


# ============================================================
# 7. VALIDATION
# ============================================================

print("=" * 70)
print("CREDIT CARD HISTORY VALIDATION")
print("=" * 70)

print(
    "Credit-card revolvers:",
    len(credit_card_customers)
)

print(
    "Credit-card history rows:",
    len(credit_card_history)
)

print(
    "Expected rows:",
    len(credit_card_customers) * 216
)

print(
    "Unique customers:",
    credit_card_history[
        "Customer_ID"
    ].nunique()
)

print(
    "Unique months:",
    credit_card_history[
        "Date"
    ].nunique()
)

print(
    "Start:",
    credit_card_history[
        "Date"
    ].min()
)

print(
    "End:",
    credit_card_history[
        "Date"
    ].max()
)

print("=" * 70)

display(
    credit_card_history.head(20)
)

print("\nLast 5 dates:")

display(
    credit_card_history[
        credit_card_history["Customer_ID"]
        == credit_card_customers[
            "Customer_ID"
        ].iloc[0]
    ].tail(5)
)

BoE CREDIT CARD SERIES
Period: 2008-01-31 00:00:00 to 2025-12-31 00:00:00
Observations: 216
Unique dates: 216


,Date,Credit_Card_Outstanding,Credit_Card_Monthly_Growth
211,2025-08-31,55569,0.006721
212,2025-09-30,55824,0.004589
213,2025-10-31,56226,0.007201
214,2025-11-30,56567,0.006065
215,2025-12-31,57066,0.008821


Credit-card revolvers: 1861
CREDIT CARD HISTORY VALIDATION
Credit-card revolvers: 1861
Credit-card history rows: 401976
Expected rows: 401976
Unique customers: 1861
Unique months: 216
Start: 2008-01-31 00:00:00
End: 2025-12-31 00:00:00


,Customer_ID,Date,Credit_Card_Balance
0,17.0,2008-01-31,4773.332307
1,17.0,2008-02-29,4785.918723
2,17.0,2008-03-31,4794.368345
3,17.0,2008-04-30,4788.471213
4,17.0,2008-05-31,4859.500848
5,17.0,2008-06-30,4836.968523
6,17.0,2008-07-31,4838.904895
7,17.0,2008-08-31,4878.952582
8,17.0,2008-09-30,4975.947202
9,17.0,2008-10-31,4743.582596



Last 5 dates:


,Customer_ID,Date,Credit_Card_Balance
211,17.0,2025-08-31,4891.010897
212,17.0,2025-09-30,4913.455206
213,17.0,2025-10-31,4948.837998
214,17.0,2025-11-30,4978.851759
215,17.0,2025-12-31,5022.772191


In [226]:
# ============================================================
# CELL 75 — VALIDATE CREDIT CARD HISTORY
# ============================================================

synthetic_cc_monthly = (
    credit_card_history
    .groupby("Date")["Credit_Card_Balance"]
    .sum()
    .reset_index()
)

synthetic_cc_monthly.rename(
    columns={
        "Credit_Card_Balance": "Synthetic_Credit_Card"
    },
    inplace=True
)

cc_validation = pd.merge(
    synthetic_cc_monthly,
    credit_card_boe[
        [
            "Date",
            "Credit_Card_Outstanding",
            "Credit_Card_Monthly_Growth"
        ]
    ],
    on="Date",
    how="left"
)

cc_validation["Synthetic_Growth"] = (
    cc_validation["Synthetic_Credit_Card"]
    .pct_change()
)

display(
    cc_validation.head(12)
)

,Date,Synthetic_Credit_Card,Credit_Card_Outstanding,Credit_Card_Monthly_Growth,Synthetic_Growth
0,2008-01-31,9.992649e+06,54232,NaN,NaN
1,2008-02-29,1.001900e+07,54375,0.002637,0.002637
2,2008-03-31,1.003669e+07,54471,0.001766,0.001766
3,2008-04-30,1.002434e+07,54404,-0.001230,-0.001230
4,2008-05-31,1.017304e+07,55211,0.014833,0.014833
5,2008-06-30,1.012587e+07,54955,-0.004637,-0.004637
6,2008-07-31,1.012992e+07,54977,0.000400,0.000400
7,2008-08-31,1.021376e+07,55432,0.008276,0.008276
8,2008-09-30,1.041681e+07,56534,0.019880,0.019880
9,2008-10-31,9.930370e+06,53894,-0.046698,-0.046698


In [239]:
# ============================================================
# CELL 76 — INSPECT BOE PERSONAL / NON-CARD CONSUMER CREDIT
# ============================================================

# Find relevant BoE columns
consumer_credit_columns = [
    col for col in boe_df.columns
    if (
        "LPMBC54" in col
        or "LPMZ5G7" in col
    )
]

print("Relevant BoE columns:")
for col in consumer_credit_columns:
    print("\n", col)


# Show 2024 values
display(
    boe_df[
        boe_df["Date"].astype(str).str.startswith("2024")
    ][
        ["Date"] + consumer_credit_columns
    ]
)

Relevant BoE columns:

 Monthly amounts outstanding of monetary financial institutions' sterling consumer credit (excluding credit card) excluding securitisations to individuals (in sterling millions) seasonally adjusted  						[x] [a] [m] [n] [y] [q] [z] [r] [r] [d] [1] [t] [l] 						LPMBC54

 Monthly amounts outstanding of monetary financial institutions' sterling net consumer credit loans (excluding credit card and overdrafts) excluding securitisations to individuals (in sterling millions) seasonally adjusted  						[a] [57] [d] [1] [t] 						LPMZ5G7


,Date,Monthly amounts outstanding of monetary financial institutions' sterling consumer credit (excluding credit card) excluding securitisations to individuals (in sterling millions) seasonally adjusted \t\t\t\t\t\t[x] [a] [m] [n] [y] [q] [z] [r] [r] [d] [1] [t] [l] \t\t\t\t\t\tLPMBC54,Monthly amounts outstanding of monetary financial institutions' sterling net consumer credit loans (excluding credit card and overdrafts) excluding securitisations to individuals (in sterling millions) seasonally adjusted \t\t\t\t\t\t[a] [57] [d] [1] [t] \t\t\t\t\t\tLPMZ5G7
192,2024-12-31,73657,67690
193,2024-11-30,72845,67153
194,2024-10-31,73060,67031
195,2024-09-30,73683,67802
196,2024-08-31,73098,67191
197,2024-07-31,72813,66917
198,2024-06-30,72322,66469
199,2024-05-31,71961,66048
200,2024-04-30,71077,65224
201,2024-03-31,70483,64868


In [240]:
# ============================================================
# CELL 77A — ADD ACTUAL DECEMBER 2025 PERSONAL LOAN DATA
# ============================================================

# Make sure Date is datetime
personal_loan_boe["Date"] = pd.to_datetime(
    personal_loan_boe["Date"]
)

# ------------------------------------------------------------
# Check November 2025
# ------------------------------------------------------------

nov_2025_value = personal_loan_boe.loc[
    personal_loan_boe["Date"] == pd.Timestamp("2025-11-30"),
    "Personal_Loan_Outstanding"
].iloc[0]

print(
    "November 2025 Personal Loan:",
    nov_2025_value
)


# ------------------------------------------------------------
# Actual December 2025 BoE value
# LPMZ5G7 = £72,639 million
# ------------------------------------------------------------

dec_2025_value = 72639


# ------------------------------------------------------------
# Calculate December growth
# ------------------------------------------------------------

dec_2025_growth = (
    dec_2025_value / nov_2025_value
) - 1


print(
    "December 2025 Personal Loan:",
    dec_2025_value
)

print(
    "December 2025 growth:",
    dec_2025_growth
)


# ------------------------------------------------------------
# Create December row
# ------------------------------------------------------------

dec_2025_row = pd.DataFrame({

    "Date": [
        pd.Timestamp("2025-12-31")
    ],

    "Personal_Loan_Outstanding": [
        dec_2025_value
    ],

    "Personal_Loan_Monthly_Growth": [
        dec_2025_growth
    ]

})


# ------------------------------------------------------------
# Append December
# ------------------------------------------------------------

personal_loan_boe = pd.concat(
    [
        personal_loan_boe,
        dec_2025_row
    ],
    ignore_index=True
)


# ------------------------------------------------------------
# Remove duplicate dates if any
# ------------------------------------------------------------

personal_loan_boe = (
    personal_loan_boe
    .sort_values("Date")
    .drop_duplicates(
        subset=["Date"],
        keep="last"
    )
    .reset_index(drop=True)
)


# ============================================================
# VALIDATION
# ============================================================

print("=" * 70)

print(
    "Personal Loan observations:",
    len(personal_loan_boe)
)

print(
    "Unique dates:",
    personal_loan_boe["Date"].nunique()
)

print(
    "Start:",
    personal_loan_boe["Date"].min()
)

print(
    "End:",
    personal_loan_boe["Date"].max()
)

print("=" * 70)

display(
    personal_loan_boe.tail(3)
)

November 2025 Personal Loan: 71623
December 2025 Personal Loan: 72639
December 2025 growth: 0.01418538737556374
Personal Loan observations: 216
Unique dates: 216
Start: 2008-01-31 00:00:00
End: 2025-12-31 00:00:00


,Date,Personal_Loan_Outstanding,Personal_Loan_Monthly_Growth
213,2025-10-31,71018,0.008020
214,2025-11-30,71623,0.008519
215,2025-12-31,72639,0.014185


In [241]:
# ============================================================
# CELL 76 — PREPARE BOE NON-CARD CONSUMER CREDIT LOANS
# ============================================================

# ------------------------------------------------------------
# 1. FIND LPMZ5G7 COLUMN
# ------------------------------------------------------------

personal_loan_column = [
    col for col in boe_df.columns
    if "LPMZ5G7" in str(col)
][0]

print("BoE loan series found:")
print(personal_loan_column)


# ------------------------------------------------------------
# 2. CREATE CLEAN DATAFRAME
# ------------------------------------------------------------

personal_loan_boe = boe_df[
    [
        "Date",
        personal_loan_column
    ]
].copy()


# ------------------------------------------------------------
# 3. CONVERT DATA TYPES
# ------------------------------------------------------------

personal_loan_boe["Date"] = pd.to_datetime(
    personal_loan_boe["Date"],
    errors="coerce"
)

personal_loan_boe["Personal_Loan_Outstanding"] = pd.to_numeric(
    personal_loan_boe[personal_loan_column],
    errors="coerce"
)


# ------------------------------------------------------------
# 4. KEEP ONLY REQUIRED COLUMNS
# ------------------------------------------------------------

personal_loan_boe = personal_loan_boe[
    [
        "Date",
        "Personal_Loan_Outstanding"
    ]
].copy()


# ------------------------------------------------------------
# 5. REMOVE INVALID DATES / VALUES
# ------------------------------------------------------------

personal_loan_boe = personal_loan_boe.dropna(
    subset=[
        "Date",
        "Personal_Loan_Outstanding"
    ]
)


# ------------------------------------------------------------
# 6. SORT CHRONOLOGICALLY
# ------------------------------------------------------------

personal_loan_boe = (
    personal_loan_boe
    .sort_values("Date")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 7. KEEP JAN-2008 TO DEC-2025
# ------------------------------------------------------------

personal_loan_boe = personal_loan_boe[
    (personal_loan_boe["Date"] >= "2008-01-31") &
    (personal_loan_boe["Date"] <= "2025-12-31")
].copy()


# ------------------------------------------------------------
# 8. REMOVE DUPLICATE DATES
# ------------------------------------------------------------

personal_loan_boe = (
    personal_loan_boe
    .drop_duplicates(
        subset=["Date"],
        keep="last"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 9. CALCULATE MONTHLY GROWTH
# ------------------------------------------------------------

personal_loan_boe["Personal_Loan_Monthly_Growth"] = (
    personal_loan_boe[
        "Personal_Loan_Outstanding"
    ].pct_change()
)


# ============================================================
# 10. VALIDATION
# ============================================================

print("=" * 70)

print(
    "Period:",
    personal_loan_boe["Date"].min(),
    "to",
    personal_loan_boe["Date"].max()
)

print(
    "Observations:",
    len(personal_loan_boe)
)

print(
    "Unique dates:",
    personal_loan_boe["Date"].nunique()
)

print(
    "Missing outstanding values:",
    personal_loan_boe[
        "Personal_Loan_Outstanding"
    ].isna().sum()
)

print("=" * 70)


# ------------------------------------------------------------
# 11. CHECK LAST MONTHS
# ------------------------------------------------------------

display(
    personal_loan_boe.tail(5)
)

BoE loan series found:
Monthly amounts outstanding of monetary financial institutions' sterling net consumer credit loans (excluding credit card and overdrafts) excluding securitisations to individuals (in sterling millions) seasonally adjusted  						[a] [57] [d] [1] [t] 						LPMZ5G7
Period: 2008-01-31 00:00:00 to 2025-12-31 00:00:00
Observations: 216
Unique dates: 216
Missing outstanding values: 0


,Date,Personal_Loan_Outstanding,Personal_Loan_Monthly_Growth
211,2025-08-31,70120,0.005766
212,2025-09-30,70453,0.004749
213,2025-10-31,71018,0.008020
214,2025-11-30,71623,0.008519
215,2025-12-31,72639,0.014185


In [242]:
# ============================================================
# CELL 77 — INITIAL 2024 PERSONAL LOAN BALANCES
# ============================================================

np.random.seed(42)

personal_loan_customers = synthetic_income[
    synthetic_income["Has_Personal_Loan"] == 1
].copy()

print(
    "Personal loan customers:",
    len(personal_loan_customers)
)

# ------------------------------------------------------------
# Income as customer-level driver
# ------------------------------------------------------------

income = personal_loan_customers[
    "Annual_Income_2024"
]

income_factor = np.sqrt(
    income / income.median()
)

# Individual variation
random_factor = np.random.lognormal(
    mean=0,
    sigma=0.50,
    size=len(personal_loan_customers)
)

# Raw loan balance
personal_loan_customers[
    "Personal_Loan_2024_Raw"
] = (
    income_factor
    * random_factor
    * income.median()
    * 0.20
)

# Avoid unrealistic zero/negative balances
personal_loan_customers[
    "Personal_Loan_2024_Raw"
] = (
    personal_loan_customers[
        "Personal_Loan_2024_Raw"
    ]
    .clip(lower=500)
)

display(
    personal_loan_customers[
        [
            "Customer_ID",
            "Annual_Income_2024",
            "Personal_Loan_2024_Raw"
        ]
    ].head(20)
)

Personal loan customers: 1413


,Customer_ID,Annual_Income_2024,Personal_Loan_2024_Raw
1,2,199999.5,21450.404118
2,3,44999.5,7406.977891
18,19,24999.5,8178.437820
20,21,34999.5,14990.411921
32,33,7499.5,2882.247765
44,45,17499.5,4402.826194
51,52,54999.5,19326.890770
53,54,84999.5,16010.888580
57,58,17499.5,3914.070719
62,63,64999.5,12512.132655


In [243]:
# ============================================================
# CELL 77B — PERSONAL LOAN 2024 CALIBRATION
# ============================================================

# ------------------------------------------------------------
# Get December 2024 directly from personal_loan_boe
# ------------------------------------------------------------

personal_loan_2024_row = personal_loan_boe[
    personal_loan_boe["Date"] == "2024-12-31"
].copy()

display(personal_loan_2024_row)


# ------------------------------------------------------------
# BoE December 2024 outstanding amount
# LPMZ5G7
# Values are in £ millions
# ------------------------------------------------------------

boe_personal_loan_target = (
    personal_loan_2024_row[
        "Personal_Loan_Outstanding"
    ].iloc[0]
    * 1_000_000
)

print(
    "BoE Dec-2024 non-card consumer-credit loans (£):",
    round(boe_personal_loan_target, 2)
)


# ------------------------------------------------------------
# FCA 2024 personal-loan population proxy
# Approximately 7.3 million adults
# ------------------------------------------------------------

FCA_PERSONAL_LOAN_HOLDERS = 7.3 * 1_000_000

print(
    "FCA personal-loan population :",
    FCA_PERSONAL_LOAN_HOLDERS
)


# ------------------------------------------------------------
# Synthetic personal-loan customers
# ------------------------------------------------------------

synthetic_loan_count = (
    synthetic_income["Has_Personal_Loan"].sum()
)

print(
    "Synthetic personal-loan customers:",
    synthetic_loan_count
)


# ------------------------------------------------------------
# Estimated average outstanding balance
# ------------------------------------------------------------

uk_average_loan_balance = (
    boe_personal_loan_target
    / FCA_PERSONAL_LOAN_HOLDERS
)

print(
    "Estimated average loan balance (£):",
    round(uk_average_loan_balance, 2)
)


# ------------------------------------------------------------
# Synthetic target
# ------------------------------------------------------------

synthetic_personal_loan_target = (
    uk_average_loan_balance
    * synthetic_loan_count
)

print(
    "Synthetic target balance (£):",
    round(
        synthetic_personal_loan_target,
        2
    )
)

,Date,Personal_Loan_Outstanding,Personal_Loan_Monthly_Growth
203,2024-12-31,67690,0.007997


BoE Dec-2024 non-card consumer-credit loans (£): 67690000000
FCA personal-loan population : 7300000.0
Synthetic personal-loan customers: 1413
Estimated average loan balance (£): 9272.6
Synthetic target balance (£): 13102187.67


In [244]:
# ============================================================
# CELL 77C — CALIBRATE PERSONAL LOAN BALANCES
# ============================================================

raw_total = (
    personal_loan_customers[
        "Personal_Loan_2024_Raw"
    ].sum()
)

calibration_factor = (
    synthetic_personal_loan_target
    / raw_total
)

personal_loan_customers[
    "Personal_Loan_2024"
] = (
    personal_loan_customers[
        "Personal_Loan_2024_Raw"
    ]
    * calibration_factor
).round(2)


# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

calibrated_total = (
    personal_loan_customers[
        "Personal_Loan_2024"
    ].sum()
)

print("Raw total (£):", round(raw_total, 2))
print("Calibration factor:", round(calibration_factor, 6))
print("Calibrated total (£):", round(calibrated_total, 2))
print("Target (£):", round(synthetic_personal_loan_target, 2))

print(
    "Difference (£):",
    round(
        calibrated_total - synthetic_personal_loan_target,
        2
    )
)

display(
    personal_loan_customers[
        [
            "Customer_ID",
            "Annual_Income_2024",
            "Personal_Loan_2024_Raw",
            "Personal_Loan_2024"
        ]
    ].head(20)
)

Raw total (£): 11795396.74
Calibration factor: 1.110788
Calibrated total (£): 13102187.82
Target (£): 13102187.67
Difference (£): 0.15


,Customer_ID,Annual_Income_2024,Personal_Loan_2024_Raw,Personal_Loan_2024
1,2,199999.5,21450.404118,23826.86
2,3,44999.5,7406.977891,8227.58
18,19,24999.5,8178.437820,9084.51
20,21,34999.5,14990.411921,16651.17
32,33,7499.5,2882.247765,3201.57
44,45,17499.5,4402.826194,4890.61
51,52,54999.5,19326.890770,21468.08
53,54,84999.5,16010.888580,17784.71
57,58,17499.5,3914.070719,4347.70
62,63,64999.5,12512.132655,13898.33


In [245]:
# ============================================================
# CELL 77D — ADD CALIBRATED PERSONAL LOAN TO CUSTOMER MASTER
# ============================================================

# Create the column for all customers
synthetic_income["Personal_Loan_2024"] = 0.0

# Copy calibrated balances to the customers who have loans
synthetic_income.loc[
    personal_loan_customers.index,
    "Personal_Loan_2024"
] = personal_loan_customers[
    "Personal_Loan_2024"
]

# ------------------------------------------------------------
# CHECK
# ------------------------------------------------------------

print(
    "Total customers:",
    len(synthetic_income)
)

print(
    "Personal-loan customers:",
    synthetic_income["Has_Personal_Loan"].sum()
)

print(
    "Customers with positive loan balance:",
    (
        synthetic_income["Personal_Loan_2024"] > 0
    ).sum()
)

print(
    "Total synthetic 2024 personal-loan balance (£):",
    round(
        synthetic_income["Personal_Loan_2024"].sum(),
        2
    )
)

display(
    synthetic_income[
        [
            "Customer_ID",
            "Annual_Income_2024",
            "Has_Personal_Loan",
            "Personal_Loan_2024"
        ]
    ].head(20)
)

Total customers: 10000
Personal-loan customers: 1413
Customers with positive loan balance: 1413
Total synthetic 2024 personal-loan balance (£): 13102187.82


,Customer_ID,Annual_Income_2024,Has_Personal_Loan,Personal_Loan_2024
0,1,24999.5,0,0.00
1,2,199999.5,1,23826.86
2,3,44999.5,1,8227.58
3,4,34999.5,0,0.00
4,5,12499.5,0,0.00
5,6,12499.5,0,0.00
6,7,7499.5,0,0.00
7,8,84999.5,0,0.00
8,9,34999.5,0,0.00
9,10,44999.5,0,0.00


In [246]:
# ============================================================
# CELL 78 — GENERATE PERSONAL LOAN HISTORY 2008–2025
# ============================================================

# ------------------------------------------------------------
# 1. PERSONAL LOAN CUSTOMERS
# ------------------------------------------------------------

personal_loan_customers_history = synthetic_income[
    synthetic_income["Has_Personal_Loan"] == 1
][
    [
        "Customer_ID",
        "Annual_Income_2024",
        "Age_2024",
        "Has_Personal_Loan",
        "Personal_Loan_2024"
    ]
].copy()

print(
    "Personal-loan customers:",
    len(personal_loan_customers_history)
)


# ============================================================
# 2. BASELINE
# ============================================================

baseline_date = pd.Timestamp("2024-12-31")


# ============================================================
# 3. PREPARE BOE DATA
# ============================================================

personal_loan_boe["Date"] = pd.to_datetime(
    personal_loan_boe["Date"]
)

personal_loan_boe = (
    personal_loan_boe
    .sort_values("Date")
    .drop_duplicates(
        subset=["Date"],
        keep="last"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Before December 2024
# ------------------------------------------------------------

pre_2024 = personal_loan_boe[
    personal_loan_boe["Date"] < baseline_date
].copy()


# ------------------------------------------------------------
# After December 2024
# ------------------------------------------------------------

post_2024 = personal_loan_boe[
    personal_loan_boe["Date"] > baseline_date
].copy()


# ============================================================
# 4. CHECK FORWARD PERIOD
# ============================================================

print("=" * 70)
print("PERSONAL LOAN BOE FORWARD CHECK")
print("=" * 70)

print(
    "Forward observations:",
    len(post_2024)
)

print(
    "Forward start:",
    post_2024["Date"].min()
)

print(
    "Forward end:",
    post_2024["Date"].max()
)

print(
    "December 2025 present:",
    (
        post_2024["Date"] ==
        pd.Timestamp("2025-12-31")
    ).any()
)

print("=" * 70)


# ============================================================
# 5. GENERATE HISTORY
# ============================================================

personal_loan_rows = []


for _, customer in personal_loan_customers_history.iterrows():

    customer_id = customer["Customer_ID"]

    balance_2024 = customer["Personal_Loan_2024"]

    customer_history = {}


    # --------------------------------------------------------
    # DECEMBER 2024 BASELINE
    # --------------------------------------------------------

    customer_history[
        baseline_date
    ] = balance_2024


    # ========================================================
    # BACKWARD: DEC-2024 → JAN-2008
    # ========================================================

    current_balance = balance_2024

    dates_backward = (
        pre_2024["Date"]
        .sort_values(
            ascending=False
        )
        .tolist()
    )

    for date in dates_backward:

        next_date = (
            date +
            pd.offsets.MonthEnd(1)
        )

        growth_row = personal_loan_boe[
            personal_loan_boe["Date"] == next_date
        ]

        if len(growth_row) > 0:

            growth = growth_row[
                "Personal_Loan_Monthly_Growth"
            ].iloc[0]

            if pd.notna(growth):

                current_balance = (
                    current_balance /
                    (1 + growth)
                )

        customer_history[
            date
        ] = current_balance


    # ========================================================
    # FORWARD: DEC-2024 → DEC-2025
    # ========================================================

    current_balance = balance_2024

    for _, row in post_2024.sort_values(
        "Date"
    ).iterrows():

        growth = row[
            "Personal_Loan_Monthly_Growth"
        ]

        if pd.notna(growth):

            current_balance = (
                current_balance *
                (1 + growth)
            )

        customer_history[
            row["Date"]
        ] = current_balance


    # ========================================================
    # STORE CUSTOMER HISTORY
    # ========================================================

    for date, balance in sorted(
        customer_history.items()
    ):

        personal_loan_rows.append({

            "Customer_ID":
                customer_id,

            "Date":
                date,

            "Personal_Loan_Balance":
                balance

        })


# ============================================================
# 6. CREATE DATAFRAME
# ============================================================

personal_loan_history = pd.DataFrame(
    personal_loan_rows
)


# ============================================================
# 7. VALIDATION
# ============================================================

print("=" * 70)
print("PERSONAL LOAN HISTORY VALIDATION")
print("=" * 70)

print(
    "Personal-loan customers:",
    len(personal_loan_customers_history)
)

print(
    "History rows:",
    len(personal_loan_history)
)

print(
    "Expected rows:",
    len(personal_loan_customers_history) * 216
)

print(
    "Unique customers:",
    personal_loan_history[
        "Customer_ID"
    ].nunique()
)

print(
    "Unique months:",
    personal_loan_history[
        "Date"
    ].nunique()
)

print(
    "Start:",
    personal_loan_history[
        "Date"
    ].min()
)

print(
    "End:",
    personal_loan_history[
        "Date"
    ].max()
)

print(
    "Zero balances:",
    (
        personal_loan_history[
            "Personal_Loan_Balance"
        ] <= 0
    ).sum()
)

print("=" * 70)

display(
    personal_loan_history.head(20)
)

print("\nLast 5 months for first customer:")

first_customer = (
    personal_loan_customers_history[
        "Customer_ID"
    ].iloc[0]
)

display(
    personal_loan_history[
        personal_loan_history["Customer_ID"]
        == first_customer
    ].tail(5)
)

Personal-loan customers: 1413
PERSONAL LOAN BOE FORWARD CHECK
Forward observations: 12
Forward start: 2025-01-31 00:00:00
Forward end: 2025-12-31 00:00:00
December 2025 present: True
PERSONAL LOAN HISTORY VALIDATION
Personal-loan customers: 1413
History rows: 305208
Expected rows: 305208
Unique customers: 1413
Unique months: 216
Start: 2008-01-31 00:00:00
End: 2025-12-31 00:00:00
Zero balances: 0


,Customer_ID,Date,Personal_Loan_Balance
0,2.0,2008-01-31,31654.629429
1,2.0,2008-02-29,31677.157411
2,2.0,2008-03-31,31624.709455
3,2.0,2008-04-30,31574.725497
4,2.0,2008-05-31,31477.573578
5,2.0,2008-06-30,31242.085776
6,2.0,2008-07-31,31122.757876
7,2.0,2008-08-31,31024.549958
8,2.0,2008-09-30,30873.190085
9,2.0,2008-10-31,30647.558275



Last 5 months for first customer:


,Customer_ID,Date,Personal_Loan_Balance
211,2.0,2025-08-31,24682.219282
212,2.0,2025-09-30,24799.435184
213,2.0,2025-10-31,24998.315017
214,2.0,2025-11-30,25211.274838
215,2.0,2025-12-31,25568.906538


In [188]:
# ============================================================
# CELL 79 — VALIDATE PERSONAL LOAN GROWTH
# ============================================================

synthetic_loan_monthly = (
    personal_loan_history
    .groupby("Date")["Personal_Loan_Balance"]
    .sum()
    .reset_index()
)

synthetic_loan_monthly.rename(
    columns={
        "Personal_Loan_Balance":
        "Synthetic_Personal_Loan"
    },
    inplace=True
)


loan_validation = pd.merge(
    synthetic_loan_monthly,
    personal_loan_boe[
        [
            "Date",
            "Personal_Loan_Outstanding",
            "Personal_Loan_Monthly_Growth"
        ]
    ],
    on="Date",
    how="left"
)


loan_validation["Synthetic_Growth"] = (
    loan_validation[
        "Synthetic_Personal_Loan"
    ].pct_change()
)


display(
    loan_validation.head(12)
)

,Date,Synthetic_Personal_Loan,Personal_Loan_Outstanding,Personal_Loan_Monthly_Growth,Synthetic_Growth
0,2008-01-31,1.740661e+07,89928,NaN,NaN
1,2008-02-29,1.741900e+07,89992,0.000712,0.000712
2,2008-03-31,1.739016e+07,89843,-0.001656,-0.001656
3,2008-04-30,1.736267e+07,89701,-0.001581,-0.001581
4,2008-05-31,1.730925e+07,89425,-0.003077,-0.003077
5,2008-06-30,1.717976e+07,88756,-0.007481,-0.007481
6,2008-07-31,1.711414e+07,88417,-0.003819,-0.003819
7,2008-08-31,1.706014e+07,88138,-0.003156,-0.003156
8,2008-09-30,1.697690e+07,87708,-0.004879,-0.004879
9,2008-10-31,1.685283e+07,87067,-0.007308,-0.007308


In [191]:
# ============================================================
# CELL 79B — VALIDATE GROWTH
# ============================================================

loan_growth_correlation = (
    loan_validation[
        [
            "Synthetic_Growth",
            "Personal_Loan_Monthly_Growth"
        ]
    ]
    .corr()
    .iloc[0, 1]
)

print(
    "Personal-loan growth correlation:",
    round(loan_growth_correlation, 4)
)

Personal-loan growth correlation: 1.0


In [192]:
# ============================================================
# CELL 79C — CHECK HISTORY
# ============================================================

print("History rows:", len(personal_loan_history))
print("Expected rows:", 1413 * 215)

print(
    "Unique customers:",
    personal_loan_history["Customer_ID"].nunique()
)

print(
    "Unique months:",
    personal_loan_history["Date"].nunique()
)

print(
    "Start:",
    personal_loan_history["Date"].min()
)

print(
    "End:",
    personal_loan_history["Date"].max()
)

History rows: 303795
Expected rows: 303795
Unique customers: 1413
Unique months: 215
Start: 2008-01-31 00:00:00
End: 2025-11-30 00:00:00


In [193]:
# ============================================================
# CELL 80 — PREPARE BOE SECURED LENDING
# ============================================================

secured_lending_column = [
    col for col in boe_df.columns
    if "LPMBC55" in col
][0]

print("Secured lending series found:")
print(secured_lending_column)


mortgage_boe = boe_df[
    [
        "Date",
        secured_lending_column
    ]
].copy()


mortgage_boe["Date"] = pd.to_datetime(
    mortgage_boe["Date"],
    errors="coerce"
)

mortgage_boe["Mortgage_Outstanding"] = pd.to_numeric(
    mortgage_boe[secured_lending_column],
    errors="coerce"
)


mortgage_boe = mortgage_boe[
    [
        "Date",
        "Mortgage_Outstanding"
    ]
].sort_values("Date").reset_index(drop=True)


mortgage_boe = mortgage_boe[
    (mortgage_boe["Date"] >= "2008-01-01") &
    (mortgage_boe["Date"] <= "2025-12-01")
].copy()


mortgage_boe["Mortgage_Monthly_Growth"] = (
    mortgage_boe["Mortgage_Outstanding"].pct_change()
)


print(
    "Period:",
    mortgage_boe["Date"].min(),
    "to",
    mortgage_boe["Date"].max()
)

print(
    "Observations:",
    len(mortgage_boe)
)

print(
    "Missing:",
    mortgage_boe["Mortgage_Outstanding"].isna().sum()
)

display(
    mortgage_boe.head(12)
)

Secured lending series found:
Monthly amounts outstanding of monetary financial institutions' sterling net secured lending to individuals (in sterling millions) seasonally adjusted  						[a] [m] [n] [2] [3] [4] [5] [6] [6] [c] [h] [i] [j] 						LPMBC55
Period: 2008-01-31 00:00:00 to 2025-11-30 00:00:00
Observations: 215
Missing: 0


,Date,Mortgage_Outstanding,Mortgage_Monthly_Growth
0,2008-01-31,1030877,NaN
1,2008-02-29,1038577,0.007469
2,2008-03-31,1045286,0.006460
3,2008-04-30,1051421,0.005869
4,2008-05-31,1056457,0.004790
5,2008-06-30,1062011,0.005257
6,2008-07-31,1068940,0.006524
7,2008-08-31,1070549,0.001505
8,2008-09-30,1073928,0.003156
9,2008-10-31,1077822,0.003626


In [194]:
# ============================================================
# CELL 81 — DECEMBER 2024 BOE SECURED LENDING
# ============================================================

mortgage_2024_boe = mortgage_boe[
    mortgage_boe["Date"] == "2024-12-31"
].copy()

display(mortgage_2024_boe)

boe_mortgage_target = (
    mortgage_2024_boe[
        "Mortgage_Outstanding"
    ].iloc[0]
    * 1_000_000
)

print(
    "BoE Dec-2024 secured lending (£):",
    round(boe_mortgage_target, 2)
)

print(
    "BoE Dec-2024 secured lending (£bn):",
    round(
        boe_mortgage_target / 1_000_000_000,
        2
    )
)

,Date,Mortgage_Outstanding,Mortgage_Monthly_Growth
203,2024-12-31,1472728,-0.000191


BoE Dec-2024 secured lending (£): 1472728000000
BoE Dec-2024 secured lending (£bn): 1472.73


In [195]:
# ============================================================
# CELL 83 — MORTGAGE 2024 CALIBRATION TARGET
# ============================================================

# FCA 2024 residential mortgage holders
FCA_UK_MORTGAGE_HOLDERS = 14.8 * 1_000_000

# Synthetic mortgage customers
synthetic_mortgage_count = (
    synthetic_income["Has_Mortgage"].sum()
)

# BoE December 2024 secured lending
boe_mortgage_target = (
    mortgage_2024_boe[
        "Mortgage_Outstanding"
    ].iloc[0]
    * 1_000_000
)

print(
    "BoE Dec-2024 secured lending (£):",
    f"{boe_mortgage_target:,.2f}"
)

print(
    "FCA UK residential mortgage holders:",
    f"{FCA_UK_MORTGAGE_HOLDERS:,.0f}"
)

print(
    "Synthetic mortgage customers:",
    synthetic_mortgage_count
)


# ------------------------------------------------------------
# Estimated average balance
# ------------------------------------------------------------

uk_average_mortgage_balance = (
    boe_mortgage_target /
    FCA_UK_MORTGAGE_HOLDERS
)

print(
    "Estimated average mortgage balance (£):",
    f"{uk_average_mortgage_balance:,.2f}"
)


# ------------------------------------------------------------
# Synthetic target
# ------------------------------------------------------------

synthetic_mortgage_target = (
    uk_average_mortgage_balance *
    synthetic_mortgage_count
)

print(
    "Synthetic mortgage target (£):",
    f"{synthetic_mortgage_target:,.2f}"
)

BoE Dec-2024 secured lending (£): 1,472,728,000,000.00
FCA UK residential mortgage holders: 14,800,000
Synthetic mortgage customers: 2621
Estimated average mortgage balance (£): 99,508.65
Synthetic mortgage target (£): 260,812,168.11


In [197]:
# ============================================================
# CELL 84 — REVISED RAW 2024 MORTGAGE BALANCES
# ============================================================

np.random.seed(42)

mortgage_customers = synthetic_income[
    synthetic_income["Has_Mortgage"] == 1
].copy()

print(
    "Mortgage customers:",
    len(mortgage_customers)
)


# ------------------------------------------------------------
# Income
# ------------------------------------------------------------

income = mortgage_customers[
    "Annual_Income_2024"
].copy()

# Prevent extremely low synthetic incomes from generating
# unrealistic mortgage capacity.
income_for_mortgage = income.clip(lower=15_000)


# ------------------------------------------------------------
# Income relationship
# ------------------------------------------------------------

income_factor = (
    income_for_mortgage /
    income_for_mortgage.median()
) ** 0.65


# ------------------------------------------------------------
# Age factor
# ------------------------------------------------------------

age = mortgage_customers["Age_2024"]

age_factor = np.select(
    [
        age < 25,
        (age >= 25) & (age < 35),
        (age >= 35) & (age < 50),
        (age >= 50) & (age < 65),
        age >= 65
    ],
    [
        0.75,
        0.90,
        1.00,
        1.05,
        0.85
    ],
    default=1.00
)


# ------------------------------------------------------------
# Individual variation
# ------------------------------------------------------------

random_factor = np.random.lognormal(
    mean=0,
    sigma=0.25,
    size=len(mortgage_customers)
)


# ------------------------------------------------------------
# Raw mortgage balance
# ------------------------------------------------------------

mortgage_customers[
    "Mortgage_2024_Raw"
] = (
    income_factor
    * age_factor
    * random_factor
    * income_for_mortgage
    * 2.2
)


# ------------------------------------------------------------
# Mortgage-to-income constraint
#
# We don't want very low-income customers to receive
# extremely large mortgages.
# ------------------------------------------------------------

mortgage_customers[
    "Mortgage_2024_Raw"
] = np.minimum(
    mortgage_customers[
        "Mortgage_2024_Raw"
    ],
    income_for_mortgage * 4.5
)


# ------------------------------------------------------------
# Minimum mortgage
# ------------------------------------------------------------

mortgage_customers[
    "Mortgage_2024_Raw"
] = (
    mortgage_customers[
        "Mortgage_2024_Raw"
    ].clip(lower=20_000)
)


# ------------------------------------------------------------
# Calculate LTI
# ------------------------------------------------------------

mortgage_customers[
    "Mortgage_to_Income"
] = (
    mortgage_customers["Mortgage_2024_Raw"]
    / income_for_mortgage
)


# ------------------------------------------------------------
# CHECK
# ------------------------------------------------------------

print(
    "Median mortgage (£):",
    round(
        mortgage_customers[
            "Mortgage_2024_Raw"
        ].median(),
        2
    )
)

print(
    "Median mortgage-to-income:",
    round(
        mortgage_customers[
            "Mortgage_to_Income"
        ].median(),
        2
    )
)

print(
    "Maximum mortgage-to-income:",
    round(
        mortgage_customers[
            "Mortgage_to_Income"
        ].max(),
        2
    )
)

print(
    "Customers above 4x LTI:",
    (
        mortgage_customers[
            "Mortgage_to_Income"
        ] >= 4
    ).sum()
)

print(
    "Customers above 4.5x LTI:",
    (
        mortgage_customers[
            "Mortgage_to_Income"
        ] >= 4.5
    ).sum()
)

display(
    mortgage_customers[
        [
            "Customer_ID",
            "Age_2024",
            "Annual_Income_2024",
            "Mortgage_2024_Raw",
            "Mortgage_to_Income"
        ]
    ].head(20)
)

Mortgage customers: 2621
Median mortgage (£): 84626.43
Median mortgage-to-income: 2.27
Maximum mortgage-to-income: 4.5
Customers above 4x LTI: 286
Customers above 4.5x LTI: 203


,Customer_ID,Age_2024,Annual_Income_2024,Mortgage_2024_Raw,Mortgage_to_Income
2,3,58,44999.5,138578.074733,3.079547
6,7,24,7499.5,20000.000000,1.333333
9,10,55,44999.5,143908.471018,3.198002
14,15,30,12499.5,25057.424891,1.670495
24,25,49,24999.5,41681.782932,1.667305
28,29,51,34999.5,76252.260502,2.178667
36,37,41,24999.5,65588.850575,2.623606
38,39,60,44999.5,148281.727329,3.295186
42,43,22,2500.0,20000.000000,1.333333
61,62,37,17499.5,28098.458778,1.605672


In [198]:
# ============================================================
# CELL 84C — MORTGAGE LTI DISTRIBUTION CHECK
# ============================================================

print("Mortgage customers:", len(mortgage_customers))

print("\nLTI statistics:")
print(
    mortgage_customers[
        "Mortgage_to_Income"
    ].describe()
)

print("\nLTI >= 4.0:")
print(
    (
        mortgage_customers["Mortgage_to_Income"] >= 4.0
    ).sum()
)

print("\nLTI >= 4.5:")
print(
    (
        mortgage_customers["Mortgage_to_Income"] >= 4.5
    ).sum()
)

print("\nLTI percentage >= 4.0:")
print(
    round(
        (
            mortgage_customers["Mortgage_to_Income"] >= 4.0
        ).mean() * 100,
        2
    ),
    "%"
)

print("\nIncome below £15,000:")
print(
    (
        mortgage_customers["Annual_Income_2024"] < 15_000
    ).sum()
)

display(
    mortgage_customers[
        mortgage_customers["Annual_Income_2024"] < 15_000
    ][
        [
            "Customer_ID",
            "Age_2024",
            "Annual_Income_2024",
            "Mortgage_2024_Raw",
            "Mortgage_to_Income"
        ]
    ]
)

Mortgage customers: 2621

LTI statistics:
count    2621.000000
mean        2.443094
std         0.985470
min         0.872545
25%         1.607494
50%         2.267821
75%         3.028327
max         4.500000
Name: Mortgage_to_Income, dtype: float64

LTI >= 4.0:
286

LTI >= 4.5:
203

LTI percentage >= 4.0:
10.91 %

Income below £15,000:
263


,Customer_ID,Age_2024,Annual_Income_2024,Mortgage_2024_Raw,Mortgage_to_Income
6,7,24,7499.5,20000.000000,1.333333
14,15,30,12499.5,25057.424891,1.670495
42,43,22,2500.0,20000.000000,1.333333
98,99,23,2500.0,20000.000000,1.333333
111,112,32,12499.5,20000.000000,1.333333
...,...,...,...,...,...
9859,9860,29,12499.5,20000.000000,1.333333
9895,9896,27,7499.5,20000.000000,1.333333
9907,9908,23,2500.0,20000.000000,1.333333
9909,9910,19,2500.0,20000.000000,1.333333


In [199]:
# ============================================================
# CELL 84D — FIX MORTGAGE ELIGIBILITY
# ============================================================

np.random.seed(42)

TARGET_MORTGAGE_CUSTOMERS = 2621

# ------------------------------------------------------------
# Eligible customers
# ------------------------------------------------------------

eligible_mortgage_customers = synthetic_income[
    synthetic_income["Annual_Income_2024"] >= 15_000
].copy()

print(
    "Eligible customers:",
    len(eligible_mortgage_customers)
)

print(
    "Required mortgage customers:",
    TARGET_MORTGAGE_CUSTOMERS
)


# ------------------------------------------------------------
# Select 2,621 eligible customers
#
# Use income + age to make selection more realistic.
# Higher income and adult age increase selection score.
# ------------------------------------------------------------

income_score = (
    eligible_mortgage_customers["Annual_Income_2024"]
    / eligible_mortgage_customers["Annual_Income_2024"].median()
)

age = eligible_mortgage_customers["Age_2024"]

age_score = np.where(
    (age >= 25) & (age <= 65),
    1.0,
    0.6
)

selection_score = (
    income_score ** 0.5
) * age_score


# ------------------------------------------------------------
# Weighted selection
# ------------------------------------------------------------

selected_indices = np.random.choice(
    eligible_mortgage_customers.index,
    size=TARGET_MORTGAGE_CUSTOMERS,
    replace=False,
    p=selection_score / selection_score.sum()
)


# ------------------------------------------------------------
# Reset mortgage status
# ------------------------------------------------------------

synthetic_income["Has_Mortgage"] = 0

synthetic_income.loc[
    selected_indices,
    "Has_Mortgage"
] = 1


# ------------------------------------------------------------
# Recreate mortgage customer dataframe
# ------------------------------------------------------------

mortgage_customers = synthetic_income[
    synthetic_income["Has_Mortgage"] == 1
].copy()


# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

print("=" * 60)

print(
    "Mortgage customers:",
    len(mortgage_customers)
)

print(
    "Mortgage proportion:",
    round(
        len(mortgage_customers) /
        len(synthetic_income),
        4
    )
)

print(
    "Mortgage customers income < £15k:",
    (
        mortgage_customers[
            "Annual_Income_2024"
        ] < 15_000
    ).sum()
)

print("=" * 60)

display(
    mortgage_customers[
        [
            "Customer_ID",
            "Age_2024",
            "Annual_Income_2024",
            "Has_Mortgage"
        ]
    ].head(20)
)

Eligible customers: 8083
Required mortgage customers: 2621
Mortgage customers: 2621
Mortgage proportion: 0.2621
Mortgage customers income < £15k: 0


,Customer_ID,Age_2024,Annual_Income_2024,Has_Mortgage
0,1,41,24999.5,1
1,2,90,199999.5,1
12,13,68,64999.5,1
18,19,36,24999.5,1
36,37,41,24999.5,1
39,40,53,24999.5,1
44,45,25,17499.5,1
52,53,80,124999.5,1
53,54,65,84999.5,1
54,55,45,34999.5,1


In [200]:
# ============================================================
# CELL 84E — GENERATE REVISED RAW 2024 MORTGAGE BALANCES
# ============================================================

np.random.seed(42)

income = mortgage_customers[
    "Annual_Income_2024"
]

age = mortgage_customers[
    "Age_2024"
]


# ------------------------------------------------------------
# Income factor
# ------------------------------------------------------------

income_factor = (
    income / income.median()
) ** 0.65


# ------------------------------------------------------------
# Age factor
# ------------------------------------------------------------

age_factor = np.select(
    [
        age < 25,
        (age >= 25) & (age < 35),
        (age >= 35) & (age < 50),
        (age >= 50) & (age < 65),
        age >= 65
    ],
    [
        0.75,
        0.90,
        1.00,
        1.05,
        0.85
    ],
    default=1.00
)


# ------------------------------------------------------------
# Individual variation
# ------------------------------------------------------------

random_factor = np.random.lognormal(
    mean=0,
    sigma=0.25,
    size=len(mortgage_customers)
)


# ------------------------------------------------------------
# Raw mortgage balance
# ------------------------------------------------------------

mortgage_customers[
    "Mortgage_2024_Raw"
] = (
    income_factor
    * age_factor
    * random_factor
    * income
    * 2.2
)


# ------------------------------------------------------------
# Maximum 4.5× income
# ------------------------------------------------------------

mortgage_customers[
    "Mortgage_2024_Raw"
] = np.minimum(
    mortgage_customers[
        "Mortgage_2024_Raw"
    ],
    income * 4.5
)


# ------------------------------------------------------------
# Minimum mortgage
# ------------------------------------------------------------

mortgage_customers[
    "Mortgage_2024_Raw"
] = (
    mortgage_customers[
        "Mortgage_2024_Raw"
    ].clip(lower=20_000)
)


# ------------------------------------------------------------
# Mortgage-to-income ratio
# ------------------------------------------------------------

mortgage_customers[
    "Mortgage_to_Income"
] = (
    mortgage_customers[
        "Mortgage_2024_Raw"
    ]
    / income
)


# ------------------------------------------------------------
# CHECK
# ------------------------------------------------------------

print(
    "Mortgage customers:",
    len(mortgage_customers)
)

print(
    "Median mortgage (£):",
    round(
        mortgage_customers[
            "Mortgage_2024_Raw"
        ].median(),
        2
    )
)

print(
    "Median mortgage-to-income:",
    round(
        mortgage_customers[
            "Mortgage_to_Income"
        ].median(),
        2
    )
)

print(
    "Maximum mortgage-to-income:",
    round(
        mortgage_customers[
            "Mortgage_to_Income"
        ].max(),
        2
    )
)

print(
    "Customers >= 4x LTI:",
    (
        mortgage_customers[
            "Mortgage_to_Income"
        ] >= 4
    ).sum()
)

print(
    "Customers >= 4.5x LTI:",
    (
        mortgage_customers[
            "Mortgage_to_Income"
        ] >= 4.5
    ).sum()
)

display(
    mortgage_customers[
        [
            "Customer_ID",
            "Age_2024",
            "Annual_Income_2024",
            "Mortgage_2024_Raw",
            "Mortgage_to_Income"
        ]
    ].head(20)
)

Mortgage customers: 2621
Median mortgage (£): 92614.83
Median mortgage-to-income: 2.36
Maximum mortgage-to-income: 4.5
Customers >= 4x LTI: 448
Customers >= 4.5x LTI: 351


,Customer_ID,Age_2024,Annual_Income_2024,Mortgage_2024_Raw,Mortgage_to_Income
0,1,41,24999.5,5.003792e+04,2.001557
1,2,90,199999.5,8.999978e+05,4.500000
12,13,68,64999.5,2.137100e+05,3.287871
18,19,36,24999.5,6.467405e+04,2.587014
36,37,41,24999.5,4.168178e+04,1.667305
39,40,53,24999.5,4.376605e+04,1.750677
44,45,25,17499.5,3.277013e+04,1.872632
52,53,80,124999.5,5.624978e+05,4.500000
53,54,65,84999.5,2.516320e+05,2.960394
54,55,45,34999.5,8.818449e+04,2.519593


In [201]:
# ============================================================
# CELL 84F — CALIBRATE MORTGAGE BALANCES
# ============================================================

raw_mortgage_total = (
    mortgage_customers[
        "Mortgage_2024_Raw"
    ].sum()
)

mortgage_calibration_factor = (
    synthetic_mortgage_target
    / raw_mortgage_total
)

mortgage_customers[
    "Mortgage_2024"
] = (
    mortgage_customers[
        "Mortgage_2024_Raw"
    ]
    * mortgage_calibration_factor
).round(2)


# ============================================================
# VALIDATION
# ============================================================

calibrated_mortgage_total = (
    mortgage_customers[
        "Mortgage_2024"
    ].sum()
)

print("=" * 70)

print(
    "Raw mortgage total (£):",
    f"{raw_mortgage_total:,.2f}"
)

print(
    "Calibration factor:",
    round(
        mortgage_calibration_factor,
        6
    )
)

print(
    "Calibrated mortgage total (£):",
    f"{calibrated_mortgage_total:,.2f}"
)

print(
    "Target (£):",
    f"{synthetic_mortgage_target:,.2f}"
)

print(
    "Difference (£):",
    f"{calibrated_mortgage_total - synthetic_mortgage_target:,.2f}"
)

print("=" * 70)


display(
    mortgage_customers[
        [
            "Customer_ID",
            "Age_2024",
            "Annual_Income_2024",
            "Mortgage_2024_Raw",
            "Mortgage_2024"
        ]
    ].head(20)
)

Raw mortgage total (£): 816,617,341.30
Calibration factor: 0.319381
Calibrated mortgage total (£): 260,812,169.09
Target (£): 260,812,168.11
Difference (£): 0.98


,Customer_ID,Age_2024,Annual_Income_2024,Mortgage_2024_Raw,Mortgage_2024
0,1,41,24999.5,5.003792e+04,15981.17
1,2,90,199999.5,8.999978e+05,287442.30
12,13,68,64999.5,2.137100e+05,68254.94
18,19,36,24999.5,6.467405e+04,20655.67
36,37,41,24999.5,4.168178e+04,13312.37
39,40,53,24999.5,4.376605e+04,13978.05
44,45,25,17499.5,3.277013e+04,10466.16
52,53,80,124999.5,5.624978e+05,179651.17
53,54,65,84999.5,2.516320e+05,80366.52
54,55,45,34999.5,8.818449e+04,28164.46


In [205]:
# ============================================================
# CELL 85A — CHECK DECEMBER 2025
# ============================================================

print(
    mortgage_boe[
        mortgage_boe["Date"] >= "2025-11-01"
    ]
)

          Date  Mortgage_Outstanding  Mortgage_Monthly_Growth
214 2025-11-30          1.519710e+06                  0.00172
215 2025-12-31          1.522324e+06                  0.00172


In [206]:
# ============================================================
# CELL 85 — GENERATE MORTGAGE HISTORY 2008–2025
# 216 MONTHS
# ============================================================

mortgage_customers_history = mortgage_customers[
    [
        "Customer_ID",
        "Annual_Income_2024",
        "Age_2024",
        "Has_Mortgage",
        "Mortgage_2024"
    ]
].copy()

baseline_date = pd.Timestamp("2024-12-31")

pre_2024 = mortgage_boe[
    mortgage_boe["Date"] < baseline_date
].copy()

post_2024 = mortgage_boe[
    mortgage_boe["Date"] > baseline_date
].copy()

mortgage_rows = []


# ============================================================
# CUSTOMER LOOP
# ============================================================

for _, customer in mortgage_customers_history.iterrows():

    customer_id = customer["Customer_ID"]

    balance_2024 = customer["Mortgage_2024"]

    customer_history = {}

    # --------------------------------------------------------
    # DECEMBER 2024 BASELINE
    # --------------------------------------------------------

    customer_history[baseline_date] = balance_2024


    # ========================================================
    # BACKWARD: DEC-2024 → JAN-2008
    # ========================================================

    current_balance = balance_2024

    dates_backward = (
        pre_2024["Date"]
        .sort_values(ascending=False)
        .tolist()
    )

    for date in dates_backward:

        next_date = date + pd.offsets.MonthEnd(1)

        growth_row = mortgage_boe[
            mortgage_boe["Date"] == next_date
        ]

        if len(growth_row) > 0:

            growth = growth_row[
                "Mortgage_Monthly_Growth"
            ].iloc[0]

            if pd.notna(growth):

                current_balance = (
                    current_balance / (1 + growth)
                )

        customer_history[date] = current_balance


    # ========================================================
    # FORWARD: DEC-2024 → DEC-2025
    # ========================================================

    current_balance = balance_2024

    for _, row in post_2024.sort_values("Date").iterrows():

        growth = row[
            "Mortgage_Monthly_Growth"
        ]

        if pd.notna(growth):

            current_balance = (
                current_balance * (1 + growth)
            )

        customer_history[row["Date"]] = current_balance


    # ========================================================
    # STORE CUSTOMER HISTORY
    # ========================================================

    for date, balance in sorted(
        customer_history.items()
    ):

        mortgage_rows.append({

            "Customer_ID": customer_id,

            "Date": date,

            "Mortgage_Balance": balance

        })


# ============================================================
# CREATE DATAFRAME
# ============================================================

mortgage_history = pd.DataFrame(
    mortgage_rows
)


# ============================================================
# VALIDATION
# ============================================================

print("=" * 70)

print(
    "Mortgage history rows:",
    len(mortgage_history)
)

print(
    "Expected rows:",
    2621 * 216
)

print(
    "Unique customers:",
    mortgage_history[
        "Customer_ID"
    ].nunique()
)

print(
    "Unique months:",
    mortgage_history[
        "Date"
    ].nunique()
)

print(
    "Start:",
    mortgage_history[
        "Date"
    ].min()
)

print(
    "End:",
    mortgage_history[
        "Date"
    ].max()
)

print("=" * 70)

display(
    mortgage_history.head(20)
)

Mortgage history rows: 566136
Expected rows: 566136
Unique customers: 2621
Unique months: 216
Start: 2008-01-31 00:00:00
End: 2025-12-31 00:00:00


,Customer_ID,Date,Mortgage_Balance
0,1.0,2008-01-31,11186.465244
1,1.0,2008-02-29,11270.021073
2,1.0,2008-03-31,11342.823159
3,1.0,2008-04-30,11409.396537
4,1.0,2008-05-31,11464.044219
5,1.0,2008-06-30,11524.312930
6,1.0,2008-07-31,11599.502325
7,1.0,2008-08-31,11616.962238
8,1.0,2008-09-30,11653.629140
9,1.0,2008-10-31,11695.884516


In [207]:
# ============================================================
# CELL 86 — MORTGAGE GROWTH VALIDATION
# ============================================================

# Aggregate synthetic mortgage balances by month
synthetic_mortgage_monthly = (
    mortgage_history
    .groupby("Date")["Mortgage_Balance"]
    .sum()
    .reset_index()
)

synthetic_mortgage_monthly.rename(
    columns={
        "Mortgage_Balance":
        "Synthetic_Mortgage"
    },
    inplace=True
)


# ------------------------------------------------------------
# Merge with BoE
# ------------------------------------------------------------

mortgage_validation = pd.merge(
    synthetic_mortgage_monthly,
    mortgage_boe[
        [
            "Date",
            "Mortgage_Outstanding",
            "Mortgage_Monthly_Growth"
        ]
    ],
    on="Date",
    how="left"
)


# ------------------------------------------------------------
# Synthetic growth
# ------------------------------------------------------------

mortgage_validation[
    "Synthetic_Growth"
] = (
    mortgage_validation[
        "Synthetic_Mortgage"
    ].pct_change()
)


# ------------------------------------------------------------
# Correlation
# ------------------------------------------------------------

mortgage_growth_correlation = (
    mortgage_validation[
        [
            "Synthetic_Growth",
            "Mortgage_Monthly_Growth"
        ]
    ]
    .corr()
    .iloc[0, 1]
)


print("=" * 70)

print(
    "Mortgage growth correlation:",
    round(
        mortgage_growth_correlation,
        6
    )
)

print("=" * 70)

display(
    mortgage_validation.head(12)
)

Mortgage growth correlation: 1.0


,Date,Synthetic_Mortgage,Mortgage_Outstanding,Mortgage_Monthly_Growth,Synthetic_Growth
0,2008-01-31,1.825627e+08,1030877.0,NaN,NaN
1,2008-02-29,1.839264e+08,1038577.0,0.007469,0.007469
2,2008-03-31,1.851145e+08,1045286.0,0.006460,0.006460
3,2008-04-30,1.862010e+08,1051421.0,0.005869,0.005869
4,2008-05-31,1.870928e+08,1056457.0,0.004790,0.004790
5,2008-06-30,1.880764e+08,1062011.0,0.005257,0.005257
6,2008-07-31,1.893035e+08,1068940.0,0.006524,0.006524
7,2008-08-31,1.895884e+08,1070549.0,0.001505,0.001505
8,2008-09-30,1.901868e+08,1073928.0,0.003156,0.003156
9,2008-10-31,1.908765e+08,1077822.0,0.003626,0.003626


In [208]:
# ============================================================
# CELL 86B — FINAL MORTGAGE BASELINE CHECK
# ============================================================

dec_2024_mortgage = mortgage_validation[
    mortgage_validation["Date"] == "2024-12-31"
]

synthetic_dec_2024 = (
    dec_2024_mortgage[
        "Synthetic_Mortgage"
    ].iloc[0]
)

print(
    "Synthetic Dec-2024 mortgage (£):",
    f"{synthetic_dec_2024:,.2f}"
)

print(
    "Target Dec-2024 mortgage (£):",
    f"{synthetic_mortgage_target:,.2f}"
)

print(
    "Difference (£):",
    f"{synthetic_dec_2024 - synthetic_mortgage_target:,.2f}"
)

Synthetic Dec-2024 mortgage (£): 260,812,169.09
Target Dec-2024 mortgage (£): 260,812,168.11
Difference (£): 0.98


In [215]:

# ============================================================
# CELL 87A — STANDARDISE CURRENT ACCOUNT AND SAVINGS DATES
# ============================================================

current_account_history["Date"] = (
    pd.to_datetime(
        current_account_history["Date"]
    ) + pd.offsets.MonthEnd(0)
)

savings_history["Date"] = (
    pd.to_datetime(
        savings_history["Date"]
    ) + pd.offsets.MonthEnd(0)
)


# ============================================================
# VALIDATION
# ============================================================

print("CURRENT ACCOUNT")
print(
    current_account_history["Date"].min(),
    "→",
    current_account_history["Date"].max()
)

print(
    "Rows:",
    len(current_account_history)
)

print("\nSAVINGS")
print(
    savings_history["Date"].min(),
    "→",
    savings_history["Date"].max()
)

print(
    "Rows:",
    len(savings_history)
)

CURRENT ACCOUNT
2008-01-31 00:00:00 → 2025-12-31 00:00:00
Rows: 2102976

SAVINGS
2008-01-31 00:00:00 → 2025-12-31 00:00:00
Rows: 1926504


In [249]:
# ============================================================
# CELL 85 — BUILD FINAL DATASET
# ============================================================

# ------------------------------------------------------------
# 1. CUSTOMER MASTER
# ------------------------------------------------------------

customer_master = synthetic_income[
    [
        "Customer_ID",
        "Age_2024",
        "Annual_Income_2024",
        "Has_Current_Account",
        "Has_Savings",
        "Has_Credit_Card",
        "Has_Personal_Loan",
        "Has_Mortgage"
    ]
].copy()


# ------------------------------------------------------------
# 2. CREATE 216-MONTH CALENDAR
# ------------------------------------------------------------

dates = pd.date_range(
    start="2008-01-31",
    end="2025-12-31",
    freq="ME"
)

date_df = pd.DataFrame({
    "Date": dates
})


# ------------------------------------------------------------
# 3. CUSTOMER × MONTH GRID
# ------------------------------------------------------------

customer_master["_key"] = 1
date_df["_key"] = 1

final_dataset = customer_master.merge(
    date_df,
    on="_key",
    how="inner"
).drop(columns="_key")


# ------------------------------------------------------------
# 4. MERGE CURRENT ACCOUNT
# ------------------------------------------------------------

final_dataset = final_dataset.merge(
    current_account_history[
        [
            "Customer_ID",
            "Date",
            "Current_Account_Balance"
        ]
    ],
    on=["Customer_ID", "Date"],
    how="left"
)


# ------------------------------------------------------------
# 5. MERGE SAVINGS
# ------------------------------------------------------------

final_dataset = final_dataset.merge(
    savings_history[
        [
            "Customer_ID",
            "Date",
            "Savings_Balance"
        ]
    ],
    on=["Customer_ID", "Date"],
    how="left"
)


# ------------------------------------------------------------
# 6. MERGE CREDIT CARD
# ------------------------------------------------------------

final_dataset = final_dataset.merge(
    credit_card_history[
        [
            "Customer_ID",
            "Date",
            "Credit_Card_Balance"
        ]
    ],
    on=["Customer_ID", "Date"],
    how="left"
)


# ------------------------------------------------------------
# 7. MERGE PERSONAL LOAN
# ------------------------------------------------------------

final_dataset = final_dataset.merge(
    personal_loan_history[
        [
            "Customer_ID",
            "Date",
            "Personal_Loan_Balance"
        ]
    ],
    on=["Customer_ID", "Date"],
    how="left"
)


# ------------------------------------------------------------
# 8. MERGE MORTGAGE
# ------------------------------------------------------------

final_dataset = final_dataset.merge(
    mortgage_history[
        [
            "Customer_ID",
            "Date",
            "Mortgage_Balance"
        ]
    ],
    on=["Customer_ID", "Date"],
    how="left"
)


# ------------------------------------------------------------
# 9. NON-HOLDER BALANCES = 0
# ------------------------------------------------------------

final_dataset.loc[
    final_dataset["Has_Current_Account"] == 0,
    "Current_Account_Balance"
] = 0

final_dataset.loc[
    final_dataset["Has_Savings"] == 0,
    "Savings_Balance"
] = 0

final_dataset.loc[
    final_dataset["Has_Credit_Card"] == 0,
    "Credit_Card_Balance"
] = 0

final_dataset.loc[
    final_dataset["Has_Personal_Loan"] == 0,
    "Personal_Loan_Balance"
] = 0

final_dataset.loc[
    final_dataset["Has_Mortgage"] == 0,
    "Mortgage_Balance"
] = 0


# ------------------------------------------------------------
# 10. SORT
# ------------------------------------------------------------

final_dataset = (
    final_dataset
    .sort_values(
        ["Customer_ID", "Date"]
    )
    .reset_index(drop=True)
)


# ============================================================
# 11. BASIC VALIDATION
# ============================================================

print("=" * 70)
print("FINAL DATASET")
print("=" * 70)

print(
    "Rows:",
    len(final_dataset)
)

print(
    "Expected rows:",
    10000 * 216
)

print(
    "Unique customers:",
    final_dataset["Customer_ID"].nunique()
)

print(
    "Unique months:",
    final_dataset["Date"].nunique()
)

print(
    "Start:",
    final_dataset["Date"].min()
)

print(
    "End:",
    final_dataset["Date"].max()
)

print("=" * 70)

FINAL DATASET
Rows: 2160000
Expected rows: 2160000
Unique customers: 10000
Unique months: 216
Start: 2008-01-31 00:00:00
End: 2025-12-31 00:00:00


In [252]:
# ============================================================
# CELL 87 — FIX CREDIT CARD NON-REVOLVER BALANCES
# ============================================================

# ------------------------------------------------------------
# Add revolver flag from synthetic_income
# ------------------------------------------------------------

credit_card_flags = synthetic_income[
    [
        "Customer_ID",
        "Credit_Card_Revolver"
    ]
].copy()


# ------------------------------------------------------------
# Merge revolver flag into final dataset
# ------------------------------------------------------------

final_dataset = final_dataset.merge(
    credit_card_flags,
    on="Customer_ID",
    how="left"
)


# ------------------------------------------------------------
# Non-revolvers have zero revolving credit-card balance
# ------------------------------------------------------------

final_dataset.loc[
    final_dataset["Credit_Card_Revolver"] == 0,
    "Credit_Card_Balance"
] = 0


# ------------------------------------------------------------
# Customers without credit cards also have zero balance
# ------------------------------------------------------------

final_dataset.loc[
    final_dataset["Has_Credit_Card"] == 0,
    "Credit_Card_Balance"
] = 0


# ============================================================
# VALIDATION
# ============================================================

print("=" * 70)
print("CREDIT CARD FINAL VALIDATION")
print("=" * 70)

print(
    "Credit-card holders:",
    (
        final_dataset["Has_Credit_Card"] == 1
    ).sum()
)

print(
    "Credit-card revolver rows:",
    (
        final_dataset["Credit_Card_Revolver"] == 1
    ).sum()
)

print(
    "Revolver customers:",
    final_dataset[
        final_dataset["Credit_Card_Revolver"] == 1
    ]["Customer_ID"].nunique()
)

print(
    "Credit-card balance missing:",
    final_dataset[
        final_dataset["Has_Credit_Card"] == 1
    ]["Credit_Card_Balance"].isna().sum()
)

print(
    "Non-card customers with balance:",
    (
        (final_dataset["Has_Credit_Card"] == 0) &
        (final_dataset["Credit_Card_Balance"] > 0)
    ).sum()
)

print(
    "Non-revolvers with positive balance:",
    (
        (final_dataset["Credit_Card_Revolver"] == 0) &
        (final_dataset["Credit_Card_Balance"] > 0)
    ).sum()
)


# ------------------------------------------------------------
# Sample
# ------------------------------------------------------------

display(
    final_dataset[
        [
            "Customer_ID",
            "Has_Credit_Card",
            "Credit_Card_Revolver",
            "Credit_Card_Balance"
        ]
    ].drop_duplicates("Customer_ID").head(20)
)

CREDIT CARD FINAL VALIDATION
Credit-card holders: 1405296
Credit-card revolver rows: 401976
Revolver customers: 1861
Credit-card balance missing: 0
Non-card customers with balance: 0
Non-revolvers with positive balance: 0


,Customer_ID,Has_Credit_Card,Credit_Card_Revolver,Credit_Card_Balance
0,1,1,0,0.000000
216,2,1,0,0.000000
432,3,0,0,0.000000
648,4,1,0,0.000000
864,5,1,0,0.000000
1080,6,0,0,0.000000
1296,7,0,0,0.000000
1512,8,1,0,0.000000
1728,9,0,0,0.000000
1944,10,1,0,0.000000


In [253]:
# ============================================================
# CELL 88 — FINAL DECEMBER 2024 PORTFOLIO RECONCILIATION
# ============================================================

dec_2024 = final_dataset[
    final_dataset["Date"] == pd.Timestamp("2024-12-31")
].copy()


print("=" * 70)
print("DECEMBER 2024 SYNTHETIC BANK RECONCILIATION")
print("=" * 70)


# ------------------------------------------------------------
# SYNTHETIC TOTALS
# ------------------------------------------------------------

synthetic_totals = {

    "Credit Card":
        dec_2024["Credit_Card_Balance"].sum(),

    "Personal Loan":
        dec_2024["Personal_Loan_Balance"].sum(),

    "Mortgage":
        dec_2024["Mortgage_Balance"].sum()
}


for product, total in synthetic_totals.items():

    print(
        f"{product} synthetic total (£): "
        f"{total:,.2f}"
    )


# ------------------------------------------------------------
# BOE TARGETS
# ------------------------------------------------------------

boe_targets = {

    "Credit Card":
        credit_card_boe[
            credit_card_boe["Date"] ==
            pd.Timestamp("2024-12-31")
        ]["Credit_Card_Outstanding"].iloc[0] * 1_000_000,

    "Personal Loan":
        personal_loan_boe[
            personal_loan_boe["Date"] ==
            pd.Timestamp("2024-12-31")
        ]["Personal_Loan_Outstanding"].iloc[0] * 1_000_000,

    "Mortgage":
        mortgage_boe[
            mortgage_boe["Date"] ==
            pd.Timestamp("2024-12-31")
        ]["Mortgage_Outstanding"].iloc[0] * 1_000_000
}


print("\n" + "=" * 70)
print("BOE TARGETS")
print("=" * 70)


for product, target in boe_targets.items():

    print(
        f"{product} BoE target (£): "
        f"{target:,.2f}"
    )


# ------------------------------------------------------------
# RECONCILIATION
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("RECONCILIATION")
print("=" * 70)


for product in synthetic_totals:

    synthetic = synthetic_totals[product]
    target = boe_targets[product]

    difference = synthetic - target

    percentage_difference = (
        difference / target
    ) * 100

    print(f"\n{product}")
    print("-" * 50)

    print(
        f"Synthetic (£):   {synthetic:,.2f}"
    )

    print(
        f"Target (£):      {target:,.2f}"
    )

    print(
        f"Difference (£):  {difference:,.2f}"
    )

    print(
        f"Difference (%):  {percentage_difference:.6f}%"
    )

DECEMBER 2024 SYNTHETIC BANK RECONCILIATION
Credit Card synthetic total (£): 9,705,760.05
Personal Loan synthetic total (£): 13,102,187.82
Mortgage synthetic total (£): 260,812,169.09

BOE TARGETS
Credit Card BoE target (£): 52,675,000,000.00
Personal Loan BoE target (£): 67,690,000,000.00
Mortgage BoE target (£): 1,472,728,000,000.00

RECONCILIATION

Credit Card
--------------------------------------------------
Synthetic (£):   9,705,760.05
Target (£):      52,675,000,000.00
Difference (£):  -52,665,294,239.95
Difference (%):  -99.981574%

Personal Loan
--------------------------------------------------
Synthetic (£):   13,102,187.82
Target (£):      67,690,000,000.00
Difference (£):  -67,676,897,812.18
Difference (%):  -99.980644%

Mortgage
--------------------------------------------------
Synthetic (£):   260,812,169.09
Target (£):      1,472,728,000,000.00
Difference (£):  -1,472,467,187,830.91
Difference (%):  -99.982291%


In [254]:
# ============================================================
# CELL 89 — POPULATION-SCALED BOE RECONCILIATION
# ============================================================

print("=" * 70)
print("POPULATION-SCALED BOE RECONCILIATION")
print("=" * 70)


# ============================================================
# SYNTHETIC CUSTOMER COUNTS
# ============================================================

synthetic_counts = {

    "Credit Card":
        final_dataset[
            final_dataset["Credit_Card_Revolver"] == 1
        ]["Customer_ID"].nunique(),

    "Personal Loan":
        final_dataset[
            final_dataset["Has_Personal_Loan"] == 1
        ]["Customer_ID"].nunique(),

    "Mortgage":
        final_dataset[
            final_dataset["Has_Mortgage"] == 1
        ]["Customer_ID"].nunique()
}


# ============================================================
# FCA POPULATION PROXIES
# ============================================================

fca_populations = {

    "Credit Card":
        10_100_000,

    "Personal Loan":
        7_300_000,

    "Mortgage":
        14_800_000
}


# ============================================================
# DECEMBER 2024 SYNTHETIC BALANCES
# ============================================================

dec_2024 = final_dataset[
    final_dataset["Date"] ==
    pd.Timestamp("2024-12-31")
].copy()


synthetic_totals = {

    "Credit Card":
        dec_2024[
            "Credit_Card_Balance"
        ].sum(),

    "Personal Loan":
        dec_2024[
            "Personal_Loan_Balance"
        ].sum(),

    "Mortgage":
        dec_2024[
            "Mortgage_Balance"
        ].sum()
}


# ============================================================
# BOE NATIONAL TARGETS
# ============================================================

boe_targets = {

    "Credit Card":
        credit_card_boe[
            credit_card_boe["Date"] ==
            pd.Timestamp("2024-12-31")
        ][
            "Credit_Card_Outstanding"
        ].iloc[0] * 1_000_000,

    "Personal Loan":
        personal_loan_boe[
            personal_loan_boe["Date"] ==
            pd.Timestamp("2024-12-31")
        ][
            "Personal_Loan_Outstanding"
        ].iloc[0] * 1_000_000,

    "Mortgage":
        mortgage_boe[
            mortgage_boe["Date"] ==
            pd.Timestamp("2024-12-31")
        ][
            "Mortgage_Outstanding"
        ].iloc[0] * 1_000_000
}


# ============================================================
# CALCULATE AVERAGE SYNTHETIC BALANCE
# ============================================================

for product in synthetic_totals:

    customers = synthetic_counts[product]

    synthetic_total = synthetic_totals[product]

    average_balance = (
        synthetic_total / customers
    )

    print("\n" + product)
    print("-" * 60)

    print(
        "Synthetic customers:",
        customers
    )

    print(
        "FCA population:",
        f"{fca_populations[product]:,.0f}"
    )

    print(
        "Synthetic total (£):",
        f"{synthetic_total:,.2f}"
    )

    print(
        "Average synthetic balance (£):",
        f"{average_balance:,.2f}"
    )


# ============================================================
# SCALE SYNTHETIC PORTFOLIO TO FCA POPULATION
# ============================================================

print("\n" + "=" * 70)
print("SCALED PORTFOLIO")
print("=" * 70)


for product in synthetic_totals:

    customers = synthetic_counts[product]

    synthetic_total = synthetic_totals[product]

    population = fca_populations[product]

    boe_target = boe_targets[product]


    # Average balance per synthetic customer
    average_balance = (
        synthetic_total / customers
    )


    # Reconstruct UK-wide portfolio
    scaled_total = (
        average_balance *
        population
    )


    difference = (
        scaled_total -
        boe_target
    )


    percentage_difference = (
        difference /
        boe_target
    ) * 100


    print("\n" + product)
    print("-" * 60)

    print(
        "Synthetic customers:",
        f"{customers:,}"
    )

    print(
        "Average synthetic balance (£):",
        f"{average_balance:,.2f}"
    )

    print(
        "FCA population:",
        f"{population:,.0f}"
    )

    print(
        "Scaled UK portfolio (£):",
        f"{scaled_total:,.2f}"
    )

    print(
        "BoE target (£):",
        f"{boe_target:,.2f}"
    )

    print(
        "Difference (£):",
        f"{difference:,.2f}"
    )

    print(
        "Difference (%):",
        f"{percentage_difference:.4f}%"
    )

POPULATION-SCALED BOE RECONCILIATION

Credit Card
------------------------------------------------------------
Synthetic customers: 1861
FCA population: 10,100,000
Synthetic total (£): 9,705,760.05
Average synthetic balance (£): 5,215.35

Personal Loan
------------------------------------------------------------
Synthetic customers: 1413
FCA population: 7,300,000
Synthetic total (£): 13,102,187.82
Average synthetic balance (£): 9,272.60

Mortgage
------------------------------------------------------------
Synthetic customers: 2621
FCA population: 14,800,000
Synthetic total (£): 260,812,169.09
Average synthetic balance (£): 99,508.65

SCALED PORTFOLIO

Credit Card
------------------------------------------------------------
Synthetic customers: 1,861
Average synthetic balance (£): 5,215.35
FCA population: 10,100,000
Scaled UK portfolio (£): 52,675,000,808.71
BoE target (£): 52,675,000,000.00
Difference (£): 808.71
Difference (%): 0.0000%

Personal Loan
---------------------------------

In [255]:
# ============================================================
# CELL 91 — CTGAN TRAINING DATA
# DECEMBER 2024 CUSTOMER SNAPSHOT
# ============================================================

ctgan_date = pd.Timestamp("2024-12-31")

ctgan_data = final_dataset[
    final_dataset["Date"] == ctgan_date
].copy()


print("=" * 70)
print("CTGAN TRAINING DATA")
print("=" * 70)

print("Date:", ctgan_date)
print("Rows:", len(ctgan_data))
print(
    "Unique customers:",
    ctgan_data["Customer_ID"].nunique()
)

print("\nColumns:")
print(ctgan_data.columns.tolist())

display(ctgan_data.head())

CTGAN TRAINING DATA
Date: 2024-12-31 00:00:00
Rows: 10000
Unique customers: 10000

Columns:
['Customer_ID', 'Age_2024', 'Annual_Income_2024', 'Has_Current_Account', 'Has_Savings', 'Has_Credit_Card', 'Has_Personal_Loan', 'Has_Mortgage', 'Date', 'Current_Account_Balance', 'Savings_Balance', 'Credit_Card_Balance', 'Personal_Loan_Balance', 'Mortgage_Balance', 'Credit_Card_Revolver']


,Customer_ID,Age_2024,Annual_Income_2024,Has_Current_Account,Has_Savings,Has_Credit_Card,Has_Personal_Loan,Has_Mortgage,Date,Current_Account_Balance,Savings_Balance,Credit_Card_Balance,Personal_Loan_Balance,Mortgage_Balance,Credit_Card_Revolver
203,1,41,24999.5,1,1,1,0,1,2024-12-31,1172.0,2772.0,0.0,0.00,15981.17,0
419,2,90,199999.5,1,1,1,1,1,2024-12-31,12254.0,98029.0,0.0,23826.86,287442.30,0
635,3,58,44999.5,1,1,0,1,0,2024-12-31,2511.0,9491.0,0.0,8227.58,0.00,0
851,4,46,34999.5,1,1,1,0,0,2024-12-31,1836.0,4722.0,0.0,0.00,0.00,0
1067,5,29,12499.5,1,1,1,0,0,2024-12-31,517.0,1237.0,0.0,0.00,0.00,0


In [256]:
# ============================================================
# CELL 92 — PREPARE CTGAN COLUMNS
# ============================================================

ctgan_train = ctgan_data[
    [
        "Age_2024",
        "Annual_Income_2024",

        "Has_Current_Account",
        "Has_Savings",
        "Has_Credit_Card",
        "Has_Personal_Loan",
        "Has_Mortgage",

        "Credit_Card_Revolver",

        "Current_Account_Balance",
        "Savings_Balance",
        "Credit_Card_Balance",
        "Personal_Loan_Balance",
        "Mortgage_Balance"
    ]
].copy()


print("=" * 70)
print("CTGAN TRAINING COLUMNS")
print("=" * 70)

print("Rows:", len(ctgan_train))
print("Columns:", len(ctgan_train.columns))

display(ctgan_train.head())

CTGAN TRAINING COLUMNS
Rows: 10000
Columns: 13


,Age_2024,Annual_Income_2024,Has_Current_Account,Has_Savings,Has_Credit_Card,Has_Personal_Loan,Has_Mortgage,Credit_Card_Revolver,Current_Account_Balance,Savings_Balance,Credit_Card_Balance,Personal_Loan_Balance,Mortgage_Balance
203,41,24999.5,1,1,1,0,1,0,1172.0,2772.0,0.0,0.00,15981.17
419,90,199999.5,1,1,1,1,1,0,12254.0,98029.0,0.0,23826.86,287442.30
635,58,44999.5,1,1,0,1,0,0,2511.0,9491.0,0.0,8227.58,0.00
851,46,34999.5,1,1,1,0,0,0,1836.0,4722.0,0.0,0.00,0.00
1067,29,12499.5,1,1,1,0,0,0,517.0,1237.0,0.0,0.00,0.00


In [257]:
# ============================================================
# CELL 93 — CTGAN DATA QUALITY CHECK
# ============================================================

print("=" * 70)
print("CTGAN DATA QUALITY CHECK")
print("=" * 70)

print("\nMissing values:")
display(
    ctgan_train.isna().sum()
)

print("\nData types:")
display(
    ctgan_train.dtypes
)

print("\nCategorical distributions:")

categorical_columns = [
    "Has_Current_Account",
    "Has_Savings",
    "Has_Credit_Card",
    "Has_Personal_Loan",
    "Has_Mortgage",
    "Credit_Card_Revolver"
]

for col in categorical_columns:

    print(f"\n{col}")

    display(
        ctgan_train[col].value_counts(
            normalize=True
        ).sort_index()
    )

CTGAN DATA QUALITY CHECK

Missing values:


Age_2024                   0
Annual_Income_2024         0
Has_Current_Account        0
Has_Savings                0
Has_Credit_Card            0
Has_Personal_Loan          0
Has_Mortgage               0
Credit_Card_Revolver       0
Current_Account_Balance    0
Savings_Balance            0
Credit_Card_Balance        0
Personal_Loan_Balance      0
Mortgage_Balance           0
dtype: int64


Data types:


Age_2024                     int64
Annual_Income_2024         float64
Has_Current_Account          int32
Has_Savings                  int32
Has_Credit_Card              int32
Has_Personal_Loan            int32
Has_Mortgage                 int64
Credit_Card_Revolver         int64
Current_Account_Balance    float64
Savings_Balance            float64
Credit_Card_Balance        float64
Personal_Loan_Balance      float64
Mortgage_Balance           float64
dtype: object


Categorical distributions:

Has_Current_Account


...

Has_Personal_Loan
0    0.8587
1    0.1413
Name: proportion, dtype: float64


Has_Mortgage


Has_Mortgage
0    0.7379
1    0.2621
Name: proportion, dtype: float64


Credit_Card_Revolver


Credit_Card_Revolver
0    0.8139
1    0.1861
Name: proportion, dtype: float64

In [258]:
# ============================================================
# CELL 94 — CHECK SDV
# ============================================================

try:
    import sdv

    print("SDV installed successfully.")
    print("SDV version:", sdv.__version__)

except ImportError:
    print("SDV is NOT installed.")
    print("Run: pip install sdv")

SDV installed successfully.
SDV version: 1.10.0


In [259]:
# ============================================================
# CELL 95 — CREATE CTGAN METADATA
# ============================================================

from sdv.metadata import SingleTableMetadata

# ------------------------------------------------------------
# Create metadata
# ------------------------------------------------------------

metadata = SingleTableMetadata()

metadata.detect_from_dataframe(
    data=ctgan_train
)

# ------------------------------------------------------------
# Inspect detected metadata
# ------------------------------------------------------------

print("=" * 70)
print("CTGAN METADATA")
print("=" * 70)

print(
    metadata.to_dict()
)

CTGAN METADATA
{'columns': {'Age_2024': {'sdtype': 'numerical'}, 'Annual_Income_2024': {'sdtype': 'numerical'}, 'Has_Current_Account': {'sdtype': 'categorical'}, 'Has_Savings': {'sdtype': 'categorical'}, 'Has_Credit_Card': {'sdtype': 'categorical'}, 'Has_Personal_Loan': {'sdtype': 'categorical'}, 'Has_Mortgage': {'sdtype': 'categorical'}, 'Credit_Card_Revolver': {'sdtype': 'categorical'}, 'Current_Account_Balance': {'sdtype': 'numerical'}, 'Savings_Balance': {'sdtype': 'numerical'}, 'Credit_Card_Balance': {'sdtype': 'numerical'}, 'Personal_Loan_Balance': {'sdtype': 'numerical'}, 'Mortgage_Balance': {'sdtype': 'numerical'}}, 'METADATA_SPEC_VERSION': 'SINGLE_TABLE_V1'}


In [260]:
# ============================================================
# CELL 96 — CREATE CTGAN MODEL
# ============================================================

from sdv.single_table import CTGANSynthesizer

ctgan = CTGANSynthesizer(
    metadata,
    epochs=300,
    batch_size=500,
    verbose=True
)

print("=" * 70)
print("CTGAN MODEL CREATED")
print("=" * 70)

print(ctgan)

CTGAN MODEL CREATED


C:\Users\Charlie\AppData\Roaming\Python\Python310\site-packages\sdv\single_table\base.py:79: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [261]:
# ============================================================
# CELL 97 — SAVE METADATA AND TRAIN CTGAN
# ============================================================

# ------------------------------------------------------------
# Save metadata for reproducibility
# ------------------------------------------------------------

metadata.save_to_json(
    "ctgan_metadata.json"
)

print("Metadata saved successfully.")


# ------------------------------------------------------------
# Train CTGAN
# ------------------------------------------------------------

print("=" * 70)
print("STARTING CTGAN TRAINING")
print("=" * 70)

ctgan.fit(
    ctgan_train
)

print("=" * 70)
print("CTGAN TRAINING COMPLETED")
print("=" * 70)

Metadata saved successfully.
STARTING CTGAN TRAINING


Gen. (-0.38) | Discrim. (-0.39): 100%|██████████| 300/300 [04:07<00:00,  1.21it/s]

CTGAN TRAINING COMPLETED


In [263]:
# ============================================================
# CELL 100 — CTGAN LOGICAL CONSISTENCY CHECK
# ============================================================

print("=" * 70)
print("CTGAN LOGICAL CONSISTENCY CHECK")
print("=" * 70)


# ------------------------------------------------------------
# 1. Personal Loan
# ------------------------------------------------------------

personal_loan_violation = ctgan_test[
    (ctgan_test["Has_Personal_Loan"] == 0) &
    (ctgan_test["Personal_Loan_Balance"] > 0.01)
]

print("\nPERSONAL LOAN")
print("-" * 60)

print(
    "Has_Personal_Loan = 0:",
    (ctgan_test["Has_Personal_Loan"] == 0).sum()
)

print(
    "Non-holder with positive balance:",
    len(personal_loan_violation)
)


# ------------------------------------------------------------
# 2. Mortgage
# ------------------------------------------------------------

mortgage_violation = ctgan_test[
    (ctgan_test["Has_Mortgage"] == 0) &
    (ctgan_test["Mortgage_Balance"] > 0.01)
]

print("\nMORTGAGE")
print("-" * 60)

print(
    "Has_Mortgage = 0:",
    (ctgan_test["Has_Mortgage"] == 0).sum()
)

print(
    "Non-holder with positive balance:",
    len(mortgage_violation)
)


# ------------------------------------------------------------
# 3. Credit Card
# ------------------------------------------------------------

credit_card_violation = ctgan_test[
    (ctgan_test["Has_Credit_Card"] == 0) &
    (ctgan_test["Credit_Card_Balance"] > 0.01)
]

print("\nCREDIT CARD")
print("-" * 60)

print(
    "Has_Credit_Card = 0:",
    (ctgan_test["Has_Credit_Card"] == 0).sum()
)

print(
    "Non-card customer with positive balance:",
    len(credit_card_violation)
)


# ------------------------------------------------------------
# 4. Credit Card Revolver
# ------------------------------------------------------------

revolver_violation = ctgan_test[
    (ctgan_test["Credit_Card_Revolver"] == 0) &
    (ctgan_test["Credit_Card_Balance"] > 0.01)
]

print("\nCREDIT CARD REVOLVER")
print("-" * 60)

print(
    "Credit_Card_Revolver = 0:",
    (ctgan_test["Credit_Card_Revolver"] == 0).sum()
)

print(
    "Non-revolver with positive balance:",
    len(revolver_violation)
)


# ------------------------------------------------------------
# 5. Current Account
# ------------------------------------------------------------

current_account_violation = ctgan_test[
    (ctgan_test["Has_Current_Account"] == 0) &
    (ctgan_test["Current_Account_Balance"] > 0.01)
]

print("\nCURRENT ACCOUNT")
print("-" * 60)

print(
    "Has_Current_Account = 0:",
    (ctgan_test["Has_Current_Account"] == 0).sum()
)

print(
    "Non-holder with positive balance:",
    len(current_account_violation)
)


# ------------------------------------------------------------
# 6. Savings
# ------------------------------------------------------------

savings_violation = ctgan_test[
    (ctgan_test["Has_Savings"] == 0) &
    (ctgan_test["Savings_Balance"] > 0.01)
]

print("\nSAVINGS")
print("-" * 60)

print(
    "Has_Savings = 0:",
    (ctgan_test["Has_Savings"] == 0).sum()
)

print(
    "Non-holder with positive balance:",
    len(savings_violation)
)


# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

total_violations = (
    len(personal_loan_violation)
    + len(mortgage_violation)
    + len(credit_card_violation)
    + len(revolver_violation)
    + len(current_account_violation)
    + len(savings_violation)
)

print("\n" + "=" * 70)
print("TOTAL LOGICAL VIOLATIONS")
print("=" * 70)

print(
    "Total:",
    total_violations
)

CTGAN LOGICAL CONSISTENCY CHECK

PERSONAL LOAN
------------------------------------------------------------
Has_Personal_Loan = 0: 825
Non-holder with positive balance: 609

MORTGAGE
------------------------------------------------------------
Has_Mortgage = 0: 695
Non-holder with positive balance: 338

CREDIT CARD
------------------------------------------------------------
Has_Credit_Card = 0: 416
Non-card customer with positive balance: 79

CREDIT CARD REVOLVER
------------------------------------------------------------
Credit_Card_Revolver = 0: 782
Non-revolver with positive balance: 161

CURRENT ACCOUNT
------------------------------------------------------------
Has_Current_Account = 0: 87
Non-holder with positive balance: 20

SAVINGS
------------------------------------------------------------
Has_Savings = 0: 126
Non-holder with positive balance: 74

TOTAL LOGICAL VIOLATIONS
Total: 1281


In [264]:
# ============================================================
# CELL 101 — COMPLETE CTGAN LOGICAL VALIDATION
# ============================================================

rules = {

    "Current Account":
        (
            "Has_Current_Account",
            "Current_Account_Balance"
        ),

    "Savings":
        (
            "Has_Savings",
            "Savings_Balance"
        ),

    "Credit Card":
        (
            "Has_Credit_Card",
            "Credit_Card_Balance"
        ),

    "Personal Loan":
        (
            "Has_Personal_Loan",
            "Personal_Loan_Balance"
        ),

    "Mortgage":
        (
            "Has_Mortgage",
            "Mortgage_Balance"
        )
}


print("=" * 70)
print("COMPLETE CTGAN LOGICAL VALIDATION")
print("=" * 70)


for product, (holder_col, balance_col) in rules.items():

    violations = ctgan_test[
        (ctgan_test[holder_col] == 0) &
        (ctgan_test[balance_col] > 0.01)
    ]

    print("\n" + product)
    print("-" * 60)

    print(
        "Non-holders:",
        (ctgan_test[holder_col] == 0).sum()
    )

    print(
        "Non-holders with positive balance:",
        len(violations)
    )

    print(
        "Violation rate:",
        f"{len(violations) / len(ctgan_test) * 100:.2f}%"
    )


# ------------------------------------------------------------
# Credit-card revolver rule
# ------------------------------------------------------------

revolver_violations = ctgan_test[
    (ctgan_test["Credit_Card_Revolver"] == 0) &
    (ctgan_test["Credit_Card_Balance"] > 0.01)
]

print("\nCredit Card Revolver")
print("-" * 60)

print(
    "Non-revolvers:",
    (ctgan_test["Credit_Card_Revolver"] == 0).sum()
)

print(
    "Non-revolvers with positive balance:",
    len(revolver_violations)
)

print(
    "Violation rate:",
    f"{len(revolver_violations) / len(ctgan_test) * 100:.2f}%"
)

COMPLETE CTGAN LOGICAL VALIDATION

Current Account
------------------------------------------------------------
Non-holders: 87
Non-holders with positive balance: 20
Violation rate: 2.00%

Savings
------------------------------------------------------------
Non-holders: 126
Non-holders with positive balance: 74
Violation rate: 7.40%

Credit Card
------------------------------------------------------------
Non-holders: 416
Non-holders with positive balance: 79
Violation rate: 7.90%

Personal Loan
------------------------------------------------------------
Non-holders: 825
Non-holders with positive balance: 609
Violation rate: 60.90%

Mortgage
------------------------------------------------------------
Non-holders: 695
Non-holders with positive balance: 338
Violation rate: 33.80%

Credit Card Revolver
------------------------------------------------------------
Non-revolvers: 782
Non-revolvers with positive balance: 161
Violation rate: 16.10%


In [265]:
# ============================================================
# CELL 102 — CTGAN BUSINESS-RULE CORRECTION
# ============================================================

ctgan_corrected = ctgan_test.copy()

# ------------------------------------------------------------
# CURRENT ACCOUNT
# ------------------------------------------------------------

ctgan_corrected.loc[
    ctgan_corrected["Has_Current_Account"] == 0,
    "Current_Account_Balance"
] = 0.0


# ------------------------------------------------------------
# SAVINGS
# ------------------------------------------------------------

ctgan_corrected.loc[
    ctgan_corrected["Has_Savings"] == 0,
    "Savings_Balance"
] = 0.0


# ------------------------------------------------------------
# CREDIT CARD
# ------------------------------------------------------------

ctgan_corrected.loc[
    ctgan_corrected["Has_Credit_Card"] == 0,
    "Credit_Card_Balance"
] = 0.0


# ------------------------------------------------------------
# PERSONAL LOAN
# ------------------------------------------------------------

ctgan_corrected.loc[
    ctgan_corrected["Has_Personal_Loan"] == 0,
    "Personal_Loan_Balance"
] = 0.0


# ------------------------------------------------------------
# MORTGAGE
# ------------------------------------------------------------

ctgan_corrected.loc[
    ctgan_corrected["Has_Mortgage"] == 0,
    "Mortgage_Balance"
] = 0.0


# ------------------------------------------------------------
# CREDIT CARD REVOLVER
# ------------------------------------------------------------

ctgan_corrected.loc[
    ctgan_corrected["Credit_Card_Revolver"] == 0,
    "Credit_Card_Balance"
] = 0.0


print("=" * 70)
print("CTGAN BUSINESS-RULE CORRECTION COMPLETE")
print("=" * 70)

display(
    ctgan_corrected.head(20)
)

CTGAN BUSINESS-RULE CORRECTION COMPLETE


,Age_2024,Annual_Income_2024,Has_Current_Account,Has_Savings,Has_Credit_Card,Has_Personal_Loan,Has_Mortgage,Credit_Card_Revolver,Current_Account_Balance,Savings_Balance,Credit_Card_Balance,Personal_Loan_Balance,Mortgage_Balance
0,60,64106.7,1,1,0,0,1,0,2265.0,34970.0,0.00,0.00,38508.95
1,72,45560.8,1,1,1,0,0,0,4568.0,36537.0,0.00,0.00,0.00
2,47,24542.3,1,0,1,0,0,0,879.0,0.0,0.00,0.00,0.00
3,26,11103.7,1,1,1,0,0,1,738.0,2427.0,1784.57,0.00,0.00
4,59,43817.8,1,1,0,0,1,0,1782.0,9468.0,0.00,0.00,30735.94
5,73,36303.8,1,1,1,0,0,0,4539.0,36086.0,0.00,0.00,0.00
6,56,37439.0,1,1,0,0,0,0,2115.0,9386.0,0.00,0.00,0.00
7,42,52114.4,1,1,0,1,0,0,1316.0,2660.0,0.00,12024.28,0.00
8,45,26609.0,1,1,1,0,1,1,540.0,2227.0,4571.62,0.00,11034.30
9,48,34683.1,1,1,1,0,1,1,8248.0,2937.0,4760.47,0.00,16600.03


In [266]:
# ============================================================
# CELL 103 — POST-CTGAN VALIDATION
# ============================================================

print("=" * 70)
print("POST-CTGAN LOGICAL VALIDATION")
print("=" * 70)


rules = {
    "Current Account":
        ("Has_Current_Account", "Current_Account_Balance"),

    "Savings":
        ("Has_Savings", "Savings_Balance"),

    "Credit Card":
        ("Has_Credit_Card", "Credit_Card_Balance"),

    "Personal Loan":
        ("Has_Personal_Loan", "Personal_Loan_Balance"),

    "Mortgage":
        ("Has_Mortgage", "Mortgage_Balance")
}


for product, (holder, balance) in rules.items():

    violations = ctgan_corrected[
        (ctgan_corrected[holder] == 0) &
        (ctgan_corrected[balance] > 0.01)
    ]

    print(
        f"{product:<20}: {len(violations)} violations"
    )


# Credit-card revolver
revolver_violations = ctgan_corrected[
    (ctgan_corrected["Credit_Card_Revolver"] == 0) &
    (ctgan_corrected["Credit_Card_Balance"] > 0.01)
]

print(
    f"{'Credit Card Revolver':<20}: "
    f"{len(revolver_violations)} violations"
)


print("=" * 70)

POST-CTGAN LOGICAL VALIDATION
Current Account     : 0 violations
Savings             : 0 violations
Credit Card         : 0 violations
Personal Loan       : 0 violations
Mortgage            : 0 violations
Credit Card Revolver: 0 violations


In [267]:
# ============================================================
# CELL 104 — ORIGINAL VS CTGAN DISTRIBUTION COMPARISON
# ============================================================

print("=" * 70)
print("ORIGINAL vs CTGAN DISTRIBUTION COMPARISON")
print("=" * 70)


# ------------------------------------------------------------
# CATEGORICAL VARIABLES
# ------------------------------------------------------------

categorical_columns = [
    "Has_Current_Account",
    "Has_Savings",
    "Has_Credit_Card",
    "Has_Personal_Loan",
    "Has_Mortgage",
    "Credit_Card_Revolver"
]

print("\nCATEGORICAL DISTRIBUTIONS")
print("=" * 70)

for col in categorical_columns:

    original_rate = (
        ctgan_train[col].astype(int).mean()
    )

    ctgan_rate = (
        ctgan_corrected[col].astype(int).mean()
    )

    difference = ctgan_rate - original_rate

    print(
        f"{col:<25}"
        f"Original: {original_rate:.4f}   "
        f"CTGAN: {ctgan_rate:.4f}   "
        f"Difference: {difference:+.4f}"
    )


# ------------------------------------------------------------
# NUMERICAL VARIABLES
# ------------------------------------------------------------

numerical_columns = [
    "Age_2024",
    "Annual_Income_2024",
    "Current_Account_Balance",
    "Savings_Balance",
    "Credit_Card_Balance",
    "Personal_Loan_Balance",
    "Mortgage_Balance"
]

print("\n")
print("NUMERICAL DISTRIBUTIONS")
print("=" * 70)


comparison_rows = []


for col in numerical_columns:

    original_mean = ctgan_train[col].mean()
    ctgan_mean = ctgan_corrected[col].mean()

    original_median = ctgan_train[col].median()
    ctgan_median = ctgan_corrected[col].median()

    original_std = ctgan_train[col].std()
    ctgan_std = ctgan_corrected[col].std()

    mean_difference_pct = (
        (ctgan_mean - original_mean)
        / original_mean
        * 100
        if original_mean != 0
        else 0
    )

    comparison_rows.append({

        "Variable": col,

        "Original_Mean":
            original_mean,

        "CTGAN_Mean":
            ctgan_mean,

        "Mean_Difference_%":
            mean_difference_pct,

        "Original_Median":
            original_median,

        "CTGAN_Median":
            ctgan_median,

        "Original_Std":
            original_std,

        "CTGAN_Std":
            ctgan_std
    })


distribution_comparison = pd.DataFrame(
    comparison_rows
)

display(
    distribution_comparison
)

ORIGINAL vs CTGAN DISTRIBUTION COMPARISON

CATEGORICAL DISTRIBUTIONS
Has_Current_Account      Original: 0.9736   CTGAN: 0.9130   Difference: -0.0606
Has_Savings              Original: 0.8919   CTGAN: 0.8740   Difference: -0.0179
Has_Credit_Card          Original: 0.6506   CTGAN: 0.5840   Difference: -0.0666
Has_Personal_Loan        Original: 0.1413   CTGAN: 0.1750   Difference: +0.0337
Has_Mortgage             Original: 0.2621   CTGAN: 0.3050   Difference: +0.0429
Credit_Card_Revolver     Original: 0.1861   CTGAN: 0.2180   Difference: +0.0319


NUMERICAL DISTRIBUTIONS


,Variable,Original_Mean,CTGAN_Mean,Mean_Difference_%,Original_Median,CTGAN_Median,Original_Std,CTGAN_Std
0,Age_2024,48.823600,52.77900,8.101410,48.0,51.0,18.790753,19.992445
1,Annual_Income_2024,52951.774650,85047.89160,60.613865,24999.5,33643.1,94339.088368,154082.632463
2,Current_Account_Balance,1900.733700,1617.59100,-14.896495,1414.0,1231.0,1837.373177,1739.214413
3,Savings_Balance,16557.081500,20404.15200,23.235197,3715.0,3206.5,27857.194169,32569.016431
4,Credit_Card_Balance,970.576005,981.62444,1.138338,0.0,0.0,2740.856918,2406.563143
5,Personal_Loan_Balance,1310.218782,1555.96523,18.756138,0.0,0.0,4372.530146,4640.100459
6,Mortgage_Balance,26081.216909,58556.05924,124.514291,0.0,0.0,113781.936208,200019.963607


In [268]:
# ============================================================
# CELL 105 — FINANCIAL RELATIONSHIP VALIDATION
# ============================================================

print("=" * 70)
print("FINANCIAL RELATIONSHIP VALIDATION")
print("=" * 70)


relationships = [

    (
        "Income vs Savings",
        "Annual_Income_2024",
        "Savings_Balance"
    ),

    (
        "Income vs Personal Loan",
        "Annual_Income_2024",
        "Personal_Loan_Balance"
    ),

    (
        "Income vs Mortgage",
        "Annual_Income_2024",
        "Mortgage_Balance"
    ),

    (
        "Income vs Credit Card",
        "Annual_Income_2024",
        "Credit_Card_Balance"
    ),

    (
        "Income vs Current Account",
        "Annual_Income_2024",
        "Current_Account_Balance"
    )
]


for name, x, y in relationships:

    original_corr = (
        ctgan_train[[x, y]]
        .corr()
        .iloc[0, 1]
    )

    ctgan_corr = (
        ctgan_corrected[[x, y]]
        .corr()
        .iloc[0, 1]
    )

    print(
        f"{name:<30}"
        f"Original: {original_corr:+.4f}   "
        f"CTGAN: {ctgan_corr:+.4f}"
    )

FINANCIAL RELATIONSHIP VALIDATION
Income vs Savings             Original: +0.7159   CTGAN: +0.7149
Income vs Personal Loan       Original: +0.1400   CTGAN: -0.0308
Income vs Mortgage            Original: +0.7684   CTGAN: +0.8093
Income vs Credit Card         Original: +0.2154   CTGAN: -0.0784
Income vs Current Account     Original: +0.2120   CTGAN: -0.0829


In [269]:
# ============================================================
# CELL 106 — SDV SYNTHETIC DATA QUALITY REPORT
# ============================================================

from sdv.evaluation.single_table import evaluate_quality

print("=" * 70)
print("SDV SYNTHETIC DATA QUALITY REPORT")
print("=" * 70)

quality_report = evaluate_quality(
    real_data=ctgan_train,
    synthetic_data=ctgan_corrected,
    metadata=metadata
)

print("\nOverall Quality Score:")
print(
    quality_report.get_score()
)

print("\nQuality Report:")
display(
    quality_report
)

SDV SYNTHETIC DATA QUALITY REPORT
Generating report ...
(2/2) Evaluating Column Pair Trends: : 100%|██████████| 78/78 [00:01<00:00, 54.61it/s]

Overall Score: 68.1%

Properties:
- Column Shapes: 93.49%
- Column Pair Trends: 42.71%

Overall Quality Score:
0.680979984095684

Quality Report:


In [290]:
# ============================================================
# CELL 122 — PREPARE TVAE TRAINING DATA
# ============================================================

print("=" * 70)
print("TVAE TRAINING DATA")
print("=" * 70)

tvae_train = ctgan_train.copy()

# Remove identifiers/date if present
columns_to_remove = [
    "Customer_ID",
    "Date"
]

tvae_train = tvae_train.drop(
    columns=[
        col for col in columns_to_remove
        if col in tvae_train.columns
    ],
    errors="ignore"
)

print("Rows:", len(tvae_train))
print("Columns:", len(tvae_train.columns))

print("\nColumns:")
print(tvae_train.columns.tolist())

print("\nMissing values:")
print(tvae_train.isna().sum())

display(tvae_train.head(10))

TVAE TRAINING DATA
Rows: 10000
Columns: 13

Columns:
['Age_2024', 'Annual_Income_2024', 'Has_Current_Account', 'Has_Savings', 'Has_Credit_Card', 'Has_Personal_Loan', 'Has_Mortgage', 'Credit_Card_Revolver', 'Current_Account_Balance', 'Savings_Balance', 'Credit_Card_Balance', 'Personal_Loan_Balance', 'Mortgage_Balance']

Missing values:
Age_2024                   0
Annual_Income_2024         0
Has_Current_Account        0
Has_Savings                0
Has_Credit_Card            0
Has_Personal_Loan          0
Has_Mortgage               0
Credit_Card_Revolver       0
Current_Account_Balance    0
Savings_Balance            0
Credit_Card_Balance        0
Personal_Loan_Balance      0
Mortgage_Balance           0
dtype: int64


,Age_2024,Annual_Income_2024,Has_Current_Account,Has_Savings,Has_Credit_Card,Has_Personal_Loan,Has_Mortgage,Credit_Card_Revolver,Current_Account_Balance,Savings_Balance,Credit_Card_Balance,Personal_Loan_Balance,Mortgage_Balance
203,41,24999.5,1,1,1,0,1,0,1172.0,2772.0,0.0,0.00,15981.17
419,90,199999.5,1,1,1,1,1,0,12254.0,98029.0,0.0,23826.86,287442.30
635,58,44999.5,1,1,0,1,0,0,2511.0,9491.0,0.0,8227.58,0.00
851,46,34999.5,1,1,1,0,0,0,1836.0,4722.0,0.0,0.00,0.00
1067,29,12499.5,1,1,1,0,0,0,517.0,1237.0,0.0,0.00,0.00
1283,27,12499.5,1,0,0,0,0,0,517.0,0.0,0.0,0.00,0.00
1499,24,7499.5,1,1,0,0,0,0,292.0,584.0,0.0,0.00,0.00
1715,73,84999.5,1,1,1,0,0,0,5028.0,46093.0,0.0,0.00,0.00
1931,60,34999.5,1,1,0,0,0,0,1838.0,7879.0,0.0,0.00,0.00
2147,55,44999.5,1,1,1,0,0,0,2484.0,9389.0,0.0,0.00,0.00


In [291]:
# ============================================================
# CELL 123 — TVAE METADATA
# ============================================================

from sdv.metadata import SingleTableMetadata

tvae_metadata = SingleTableMetadata()

tvae_metadata.detect_from_dataframe(
    data=tvae_train
)

print("=" * 70)
print("TVAE METADATA")
print("=" * 70)

print(
    tvae_metadata.to_dict()
)

TVAE METADATA
{'columns': {'Age_2024': {'sdtype': 'numerical'}, 'Annual_Income_2024': {'sdtype': 'numerical'}, 'Has_Current_Account': {'sdtype': 'categorical'}, 'Has_Savings': {'sdtype': 'categorical'}, 'Has_Credit_Card': {'sdtype': 'categorical'}, 'Has_Personal_Loan': {'sdtype': 'categorical'}, 'Has_Mortgage': {'sdtype': 'categorical'}, 'Credit_Card_Revolver': {'sdtype': 'categorical'}, 'Current_Account_Balance': {'sdtype': 'numerical'}, 'Savings_Balance': {'sdtype': 'numerical'}, 'Credit_Card_Balance': {'sdtype': 'numerical'}, 'Personal_Loan_Balance': {'sdtype': 'numerical'}, 'Mortgage_Balance': {'sdtype': 'numerical'}}, 'METADATA_SPEC_VERSION': 'SINGLE_TABLE_V1'}


In [293]:
# ============================================================
# CELL 124 — CREATE TVAE MODEL
# ============================================================

from sdv.single_table import TVAESynthesizer

print("=" * 70)
print("CREATING TVAE MODEL")
print("=" * 70)

tvae = TVAESynthesizer(
    metadata=tvae_metadata,
    epochs=300
)

print("\nTVAE model created successfully:")
print(tvae)

CREATING TVAE MODEL

TVAE model created successfully:


C:\Users\Charlie\AppData\Roaming\Python\Python310\site-packages\sdv\single_table\base.py:79: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [294]:
# ============================================================
# CELL 125 — TRAIN TVAE
# ============================================================

print("=" * 70)
print("TRAINING TVAE")
print("=" * 70)

tvae.fit(tvae_train)

print("\nTVAE TRAINING COMPLETE")

TRAINING TVAE

TVAE TRAINING COMPLETE


In [295]:
# ============================================================
# CELL 126 — GENERATE TVAE TEST SAMPLE
# ============================================================

print("=" * 70)
print("GENERATING TVAE TEST SAMPLE")
print("=" * 70)

tvae_test = tvae.sample(
    num_rows=1000
)

print("Generated rows:", len(tvae_test))
print("Generated columns:", len(tvae_test.columns))

print("\nColumns:")
print(tvae_test.columns.tolist())

display(
    tvae_test.head(20)
)

GENERATING TVAE TEST SAMPLE
Generated rows: 1000
Generated columns: 13

Columns:
['Age_2024', 'Annual_Income_2024', 'Has_Current_Account', 'Has_Savings', 'Has_Credit_Card', 'Has_Personal_Loan', 'Has_Mortgage', 'Credit_Card_Revolver', 'Current_Account_Balance', 'Savings_Balance', 'Credit_Card_Balance', 'Personal_Loan_Balance', 'Mortgage_Balance']


C:\Users\Charlie\AppData\Roaming\Python\Python310\site-packages\ctgan\data_transformer.py:196: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[ 4.03089384e-02 -3.67082178e-01  2.59835449e-01 -1.21019675e-01
  2.55725240e-01  5.32820152e-03 -4.27045161e-01 -1.39724546e-01
  2.38697272e-01 -1.31238324e-01 -7.75443522e-02  2.69532467e-01
 -2.98445051e-01  7.79782767e-02 -1.23765106e-01 -4.12574709e-01
  2.25517455e-01 -4.72183151e-01  5.19303636e-01 -1.52946492e-02
  2.37818931e-01 -7.82677877e-02  4.02419143e-01 -5.85378205e-01
  4.69760284e-02 -4.17580955e-01  2.51869401e-01 -1.60271436e-01
  2.15075146e-01  3.39362359e-03  3.16108178e-01  3.12605595e-01
  1.39492436e-01 -1.35151987e-01 -8.38628917e-02  9.96464187e-05
 -3.30430864e-01 -4.22057000e-01 -1.75143937e-01  1.30962069e-01
 -3.70030263e-02 -1.47200307e-01  2.70105065e-01  1.07565517e-02
  1.50474712e-01  1.56416391e-01  1.56200556e-01 -1.85980885e-01
  1.556

,Age_2024,Annual_Income_2024,Has_Current_Account,Has_Savings,Has_Credit_Card,Has_Personal_Loan,Has_Mortgage,Credit_Card_Revolver,Current_Account_Balance,Savings_Balance,Credit_Card_Balance,Personal_Loan_Balance,Mortgage_Balance
0,31,15685.5,1,1,1,1,0,1,622.0,2157.0,1745.22,6448.64,59.24
1,50,33833.7,1,1,0,0,0,0,1820.0,4144.0,1.11,0.00,1.37
2,66,35118.5,1,1,1,1,0,0,2094.0,8311.0,0.00,9944.42,0.00
3,51,24880.3,1,1,1,0,1,0,1483.0,2960.0,0.34,0.00,15997.77
4,61,32718.8,1,0,1,0,0,0,1813.0,0.0,5.77,0.00,0.00
5,70,85151.9,1,1,1,0,0,1,3263.0,45995.0,5395.02,0.44,11.96
6,49,4893.9,1,1,1,0,0,0,713.0,515.0,0.00,0.00,24.72
7,34,16934.0,1,1,0,0,0,0,952.0,3011.0,3.41,0.54,0.00
8,35,16107.1,1,1,1,1,1,0,1506.0,2442.0,0.48,5184.49,9519.57
9,51,23984.5,1,1,1,0,1,0,1594.0,3711.0,0.00,0.00,18957.87


In [296]:
# ============================================================
# CELL 127 — TVAE LOGICAL CONSISTENCY CHECK
# ============================================================

print("=" * 70)
print("TVAE LOGICAL CONSISTENCY CHECK")
print("=" * 70)

# ------------------------------------------------------------
# Remove tiny floating-point values
# ------------------------------------------------------------

tvae_log_test = tvae_test.copy()

balance_columns = [
    "Current_Account_Balance",
    "Savings_Balance",
    "Credit_Card_Balance",
    "Personal_Loan_Balance",
    "Mortgage_Balance"
]

for col in balance_columns:

    tvae_log_test.loc[
        tvae_log_test[col] < 1.0,
        col
    ] = 0.0


# ------------------------------------------------------------
# Current Account
# ------------------------------------------------------------

current_violation = (
    (tvae_log_test["Has_Current_Account"] == 0) &
    (tvae_log_test["Current_Account_Balance"] > 0)
).sum()


# ------------------------------------------------------------
# Savings
# ------------------------------------------------------------

savings_violation = (
    (tvae_log_test["Has_Savings"] == 0) &
    (tvae_log_test["Savings_Balance"] > 0)
).sum()


# ------------------------------------------------------------
# Credit Card
# ------------------------------------------------------------

credit_card_violation = (
    (tvae_log_test["Has_Credit_Card"] == 0) &
    (tvae_log_test["Credit_Card_Balance"] > 0)
).sum()


# ------------------------------------------------------------
# Personal Loan
# ------------------------------------------------------------

personal_loan_violation = (
    (tvae_log_test["Has_Personal_Loan"] == 0) &
    (tvae_log_test["Personal_Loan_Balance"] > 0)
).sum()


# ------------------------------------------------------------
# Mortgage
# ------------------------------------------------------------

mortgage_violation = (
    (tvae_log_test["Has_Mortgage"] == 0) &
    (tvae_log_test["Mortgage_Balance"] > 0)
).sum()


# ------------------------------------------------------------
# Credit Card Revolver
# ------------------------------------------------------------

revolver_violation = (
    (tvae_log_test["Credit_Card_Revolver"] == 0) &
    (tvae_log_test["Credit_Card_Balance"] > 0)
).sum()


# ============================================================
# RESULTS
# ============================================================

print("\nCurrent Account violations:",
      current_violation)

print("Savings violations:",
      savings_violation)

print("Credit Card violations:",
      credit_card_violation)

print("Personal Loan violations:",
      personal_loan_violation)

print("Mortgage violations:",
      mortgage_violation)

print("Credit Card Revolver violations:",
      revolver_violation)


total_violations = (
    current_violation +
    savings_violation +
    credit_card_violation +
    personal_loan_violation +
    mortgage_violation +
    revolver_violation
)


print("\n" + "=" * 70)
print("TOTAL TVAE LOGICAL VIOLATIONS")
print("=" * 70)

print(
    "Total:",
    total_violations
)

print(
    "Violation rate:",
    f"{total_violations / len(tvae_log_test) * 100:.2f}%"
)

TVAE LOGICAL CONSISTENCY CHECK

Current Account violations: 12
Savings violations: 60
Credit Card violations: 105
Personal Loan violations: 246
Mortgage violations: 321
Credit Card Revolver violations: 284

TOTAL TVAE LOGICAL VIOLATIONS
Total: 1028
Violation rate: 102.80%


In [297]:
# ============================================================
# CELL 128 — TVAE SYNTHETIC DATA QUALITY REPORT
# ============================================================

from sdv.evaluation.single_table import evaluate_quality

print("=" * 70)
print("TVAE SYNTHETIC DATA QUALITY REPORT")
print("=" * 70)

tvae_quality_report = evaluate_quality(
    real_data=tvae_train,
    synthetic_data=tvae_test,
    metadata=tvae_metadata
)

# ------------------------------------------------------------
# Overall score
# ------------------------------------------------------------

tvae_quality_score = (
    tvae_quality_report.get_score()
)

print("\nOverall Quality Score:")
print(
    f"{tvae_quality_score:.4f}"
)

print(
    f"Percentage: {tvae_quality_score * 100:.2f}%"
)

# ------------------------------------------------------------
# Full report
# ------------------------------------------------------------

print("\nQuality Report:")

display(
    tvae_quality_report
)

TVAE SYNTHETIC DATA QUALITY REPORT
Generating report ...
(2/2) Evaluating Column Pair Trends: : 100%|██████████| 78/78 [00:02<00:00, 33.25it/s]

Overall Score: 64.08%

Properties:
- Column Shapes: 84.59%
- Column Pair Trends: 43.57%

Overall Quality Score:
0.6408
Percentage: 64.08%

Quality Report:


In [298]:
# ============================================================
# CELL 129 — CTGAN CUSTOMER-LEVEL TRAINING DATA
# ============================================================

print("=" * 70)
print("CTGAN CUSTOMER-LEVEL TRAINING DATA")
print("=" * 70)

customer_ctgan_columns = [
    "Age_2024",
    "Annual_Income_2024",
    "Has_Current_Account",
    "Has_Savings",
    "Has_Credit_Card",
    "Has_Personal_Loan",
    "Has_Mortgage",
    "Credit_Card_Revolver"
]

ctgan_customer_train = ctgan_train[
    customer_ctgan_columns
].copy()

print("Rows:", len(ctgan_customer_train))

print("Columns:")
print(
    ctgan_customer_train.columns.tolist()
)

print("\nMissing values:")

display(
    ctgan_customer_train.isna().sum()
)

display(
    ctgan_customer_train.head(20)
)

CTGAN CUSTOMER-LEVEL TRAINING DATA
Rows: 10000
Columns:
['Age_2024', 'Annual_Income_2024', 'Has_Current_Account', 'Has_Savings', 'Has_Credit_Card', 'Has_Personal_Loan', 'Has_Mortgage', 'Credit_Card_Revolver']

Missing values:


Age_2024                0
Annual_Income_2024      0
Has_Current_Account     0
Has_Savings             0
Has_Credit_Card         0
Has_Personal_Loan       0
Has_Mortgage            0
Credit_Card_Revolver    0
dtype: int64

,Age_2024,Annual_Income_2024,Has_Current_Account,Has_Savings,Has_Credit_Card,Has_Personal_Loan,Has_Mortgage,Credit_Card_Revolver
203,41,24999.5,1,1,1,0,1,0
419,90,199999.5,1,1,1,1,1,0
635,58,44999.5,1,1,0,1,0,0
851,46,34999.5,1,1,1,0,0,0
1067,29,12499.5,1,1,1,0,0,0
1283,27,12499.5,1,0,0,0,0,0
1499,24,7499.5,1,1,0,0,0,0
1715,73,84999.5,1,1,1,0,0,0
1931,60,34999.5,1,1,0,0,0,0
2147,55,44999.5,1,1,1,0,0,0


In [299]:
# ============================================================
# CELL 130 — CUSTOMER-LEVEL CTGAN METADATA
# ============================================================

from sdv.metadata import SingleTableMetadata

print("=" * 70)
print("CREATING CUSTOMER-LEVEL CTGAN METADATA")
print("=" * 70)

customer_ctgan_metadata = SingleTableMetadata()

customer_ctgan_metadata.detect_from_dataframe(
    data=ctgan_customer_train
)

print("\nMetadata:")
print(
    customer_ctgan_metadata.to_dict()
)

CREATING CUSTOMER-LEVEL CTGAN METADATA

Metadata:
{'columns': {'Age_2024': {'sdtype': 'numerical'}, 'Annual_Income_2024': {'sdtype': 'numerical'}, 'Has_Current_Account': {'sdtype': 'categorical'}, 'Has_Savings': {'sdtype': 'categorical'}, 'Has_Credit_Card': {'sdtype': 'categorical'}, 'Has_Personal_Loan': {'sdtype': 'categorical'}, 'Has_Mortgage': {'sdtype': 'categorical'}, 'Credit_Card_Revolver': {'sdtype': 'categorical'}}, 'METADATA_SPEC_VERSION': 'SINGLE_TABLE_V1'}


In [300]:
# ============================================================
# CELL 131 — CREATE CUSTOMER-LEVEL CTGAN
# ============================================================

from sdv.single_table import CTGANSynthesizer

print("=" * 70)
print("CREATING CUSTOMER-LEVEL CTGAN")
print("=" * 70)

customer_ctgan = CTGANSynthesizer(
    metadata=customer_ctgan_metadata,
    epochs=300
)

print("\nCustomer-level CTGAN created:")
print(customer_ctgan)

CREATING CUSTOMER-LEVEL CTGAN

Customer-level CTGAN created:


C:\Users\Charlie\AppData\Roaming\Python\Python310\site-packages\sdv\single_table\base.py:79: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(


In [301]:
# ============================================================
# CELL 132 — TRAIN CUSTOMER-LEVEL CTGAN
# ============================================================

print("=" * 70)
print("TRAINING CUSTOMER-LEVEL CTGAN")
print("=" * 70)

customer_ctgan.fit(
    ctgan_customer_train
)

print("\nCUSTOMER-LEVEL CTGAN TRAINING COMPLETE")

TRAINING CUSTOMER-LEVEL CTGAN

CUSTOMER-LEVEL CTGAN TRAINING COMPLETE


In [302]:
# ============================================================
# CELL 133 — GENERATE CUSTOMER-LEVEL CTGAN TEST SAMPLE
# ============================================================

print("=" * 70)
print("GENERATING CUSTOMER-LEVEL CTGAN TEST SAMPLE")
print("=" * 70)

customer_ctgan_test = customer_ctgan.sample(
    num_rows=1000
)

print(
    "Generated rows:",
    len(customer_ctgan_test)
)

print(
    "Generated columns:",
    len(customer_ctgan_test.columns)
)

print("\nColumns:")
print(
    customer_ctgan_test.columns.tolist()
)

display(
    customer_ctgan_test.head(20)
)

GENERATING CUSTOMER-LEVEL CTGAN TEST SAMPLE
Generated rows: 1000
Generated columns: 8

Columns:
['Age_2024', 'Annual_Income_2024', 'Has_Current_Account', 'Has_Savings', 'Has_Credit_Card', 'Has_Personal_Loan', 'Has_Mortgage', 'Credit_Card_Revolver']


,Age_2024,Annual_Income_2024,Has_Current_Account,Has_Savings,Has_Credit_Card,Has_Personal_Loan,Has_Mortgage,Credit_Card_Revolver
0,49,23833.8,1,1,1,0,1,1
1,71,63262.2,1,1,0,1,0,0
2,56,22610.9,1,0,1,0,0,0
3,46,24045.0,1,1,1,1,0,0
4,56,34077.3,1,0,0,0,1,0
5,50,33659.2,1,1,1,0,0,0
6,24,9571.1,1,0,1,0,0,0
7,67,49197.1,1,1,1,1,0,0
8,56,40062.2,1,1,1,1,0,1
9,40,35387.7,1,1,1,0,1,0


In [303]:
# ============================================================
# CELL 134 — CUSTOMER OWNERSHIP DISTRIBUTION VALIDATION
# ============================================================

print("=" * 70)
print("CUSTOMER-LEVEL CTGAN DISTRIBUTION VALIDATION")
print("=" * 70)

ownership_columns = [
    "Has_Current_Account",
    "Has_Savings",
    "Has_Credit_Card",
    "Has_Personal_Loan",
    "Has_Mortgage",
    "Credit_Card_Revolver"
]

for col in ownership_columns:

    original_rate = ctgan_customer_train[col].mean()
    synthetic_rate = customer_ctgan_test[col].mean()
    difference = synthetic_rate - original_rate

    print(
        f"{col:<25} "
        f"Original: {original_rate:.4f}   "
        f"CTGAN: {synthetic_rate:.4f}   "
        f"Difference: {difference:+.4f}"
    )

CUSTOMER-LEVEL CTGAN DISTRIBUTION VALIDATION
Has_Current_Account       Original: 0.9736   CTGAN: 0.9090   Difference: -0.0646
Has_Savings               Original: 0.8919   CTGAN: 0.7630   Difference: -0.1289
Has_Credit_Card           Original: 0.6506   CTGAN: 0.7070   Difference: +0.0564
Has_Personal_Loan         Original: 0.1413   CTGAN: 0.2980   Difference: +0.1567
Has_Mortgage              Original: 0.2621   CTGAN: 0.2580   Difference: -0.0041
Credit_Card_Revolver      Original: 0.1861   CTGAN: 0.2390   Difference: +0.0529


In [304]:
# ============================================================
# CELL 136 — CONTROLLED CUSTOMER OWNERSHIP GENERATION
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("GENERATING CONTROLLED SYNTHETIC CUSTOMER OWNERSHIP")
print("=" * 70)

# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

n_customers = 10000

# Reproducibility
rng = np.random.default_rng(42)

# ------------------------------------------------------------
# Target ownership proportions
# Taken from the original 2024 customer population
# ------------------------------------------------------------

target_rates = {
    "Has_Current_Account": 0.9736,
    "Has_Savings": 0.8919,
    "Has_Credit_Card": 0.6506,
    "Has_Personal_Loan": 0.1413,
    "Has_Mortgage": 0.2621
}

# Credit-card revolver rate among ALL customers
target_revolver_rate = 0.1861

# ------------------------------------------------------------
# Create customer IDs
# ------------------------------------------------------------

controlled_customers = pd.DataFrame({
    "Customer_ID": np.arange(1, n_customers + 1)
})

# ------------------------------------------------------------
# Generate Age and Income
# ------------------------------------------------------------

# Use CTGAN-generated customer characteristics
# rather than CTGAN-generated ownership

characteristics = customer_ctgan_test[
    [
        "Age_2024",
        "Annual_Income_2024"
    ]
].copy()

# We need 10,000 observations.
# Generate directly from the customer-level CTGAN.
characteristics = customer_ctgan.sample(
    num_rows=n_customers
)[
    [
        "Age_2024",
        "Annual_Income_2024"
    ]
].copy()

characteristics = characteristics.reset_index(drop=True)

controlled_customers[
    [
        "Age_2024",
        "Annual_Income_2024"
    ]
] = characteristics

# ------------------------------------------------------------
# Generate exact ownership counts
# ------------------------------------------------------------

for column, rate in target_rates.items():

    target_count = round(
        n_customers * rate
    )

    values = np.zeros(
        n_customers,
        dtype=int
    )

    selected = rng.choice(
        n_customers,
        size=target_count,
        replace=False
    )

    values[selected] = 1

    controlled_customers[column] = values


# ------------------------------------------------------------
# Credit Card Revolver
# ------------------------------------------------------------

target_revolver_count = round(
    n_customers * target_revolver_rate
)

revolver_values = np.zeros(
    n_customers,
    dtype=int
)

# Revolvers MUST have a credit card
credit_card_customers = controlled_customers.index[
    controlled_customers["Has_Credit_Card"] == 1
]

# Revolver count cannot exceed credit-card holders
target_revolver_count = min(
    target_revolver_count,
    len(credit_card_customers)
)

selected_revolvers = rng.choice(
    credit_card_customers,
    size=target_revolver_count,
    replace=False
)

revolver_values[selected_revolvers] = 1

controlled_customers[
    "Credit_Card_Revolver"
] = revolver_values


# ============================================================
# CHECK
# ============================================================

print("\nGenerated customers:")
print(
    len(controlled_customers)
)

print("\nOwnership distributions:")

for column in [
    "Has_Current_Account",
    "Has_Savings",
    "Has_Credit_Card",
    "Has_Personal_Loan",
    "Has_Mortgage",
    "Credit_Card_Revolver"
]:

    print(
        f"{column:<25}"
        f"{controlled_customers[column].mean():.4f}"
    )

display(
    controlled_customers.head(20)
)

GENERATING CONTROLLED SYNTHETIC CUSTOMER OWNERSHIP

Generated customers:
10000

Ownership distributions:
Has_Current_Account      0.9736
Has_Savings              0.8919
Has_Credit_Card          0.6506
Has_Personal_Loan        0.1413
Has_Mortgage             0.2621
Credit_Card_Revolver     0.1861


,Customer_ID,Age_2024,Annual_Income_2024,Has_Current_Account,Has_Savings,Has_Credit_Card,Has_Personal_Loan,Has_Mortgage,Credit_Card_Revolver
0,1,28,9473.7,1,1,1,0,1,1
1,2,42,25991.0,1,1,0,1,1,0
2,3,29,6200.0,1,1,1,0,1,0
3,4,47,16453.0,1,1,1,1,0,0
4,5,19,8435.3,1,1,1,1,0,1
5,6,60,33537.6,1,1,1,0,0,0
6,7,24,8183.0,1,1,1,0,0,1
7,8,18,2500.0,1,1,0,0,0,0
8,9,40,25466.5,1,1,0,0,0,0
9,10,41,35409.2,1,1,1,1,1,1


In [305]:
# ============================================================
# CELL 137 — OWNERSHIP LOGICAL VALIDATION
# ============================================================

print("=" * 70)
print("OWNERSHIP LOGICAL VALIDATION")
print("=" * 70)

# Revolver without credit card
revolver_without_card = (
    (controlled_customers["Credit_Card_Revolver"] == 1) &
    (controlled_customers["Has_Credit_Card"] == 0)
).sum()

print(
    "Revolver without credit card:",
    revolver_without_card
)

print("\nTarget vs generated:")

for column, target in {
    "Has_Current_Account": 0.9736,
    "Has_Savings": 0.8919,
    "Has_Credit_Card": 0.6506,
    "Has_Personal_Loan": 0.1413,
    "Has_Mortgage": 0.2621,
    "Credit_Card_Revolver": 0.1861
}.items():

    actual = controlled_customers[column].mean()

    print(
        f"{column:<25}"
        f"Target: {target:.4f}   "
        f"Actual: {actual:.4f}   "
        f"Difference: {actual-target:+.4f}"
    )

OWNERSHIP LOGICAL VALIDATION
Revolver without credit card: 0

Target vs generated:
Has_Current_Account      Target: 0.9736   Actual: 0.9736   Difference: +0.0000
Has_Savings              Target: 0.8919   Actual: 0.8919   Difference: +0.0000
Has_Credit_Card          Target: 0.6506   Actual: 0.6506   Difference: +0.0000
Has_Personal_Loan        Target: 0.1413   Actual: 0.1413   Difference: +0.0000
Has_Mortgage             Target: 0.2621   Actual: 0.2621   Difference: +0.0000
Credit_Card_Revolver     Target: 0.1861   Actual: 0.1861   Difference: +0.0000


In [306]:
# ============================================================
# CELL 138 — INITIALISE 2024 CUSTOMER BALANCES
# ============================================================

print("=" * 70)
print("INITIALISING 2024 CUSTOMER BALANCES")
print("=" * 70)

balance_columns = [
    "Current_Account_Balance",
    "Savings_Balance",
    "Credit_Card_Balance",
    "Personal_Loan_Balance",
    "Mortgage_Balance"
]

for col in balance_columns:
    controlled_customers[col] = 0.0


print("Rows:", len(controlled_customers))
print("Customers:", controlled_customers["Customer_ID"].nunique())

print("\nBalance columns:")
print(balance_columns)

display(
    controlled_customers.head(10)
)

INITIALISING 2024 CUSTOMER BALANCES
Rows: 10000
Customers: 10000

Balance columns:
['Current_Account_Balance', 'Savings_Balance', 'Credit_Card_Balance', 'Personal_Loan_Balance', 'Mortgage_Balance']


,Customer_ID,Age_2024,Annual_Income_2024,Has_Current_Account,Has_Savings,Has_Credit_Card,Has_Personal_Loan,Has_Mortgage,Credit_Card_Revolver,Current_Account_Balance,Savings_Balance,Credit_Card_Balance,Personal_Loan_Balance,Mortgage_Balance
0,1,28,9473.7,1,1,1,0,1,1,0.0,0.0,0.0,0.0,0.0
1,2,42,25991.0,1,1,0,1,1,0,0.0,0.0,0.0,0.0,0.0
2,3,29,6200.0,1,1,1,0,1,0,0.0,0.0,0.0,0.0,0.0
3,4,47,16453.0,1,1,1,1,0,0,0.0,0.0,0.0,0.0,0.0
4,5,19,8435.3,1,1,1,1,0,1,0.0,0.0,0.0,0.0,0.0
5,6,60,33537.6,1,1,1,0,0,0,0.0,0.0,0.0,0.0,0.0
6,7,24,8183.0,1,1,1,0,0,1,0.0,0.0,0.0,0.0,0.0
7,8,18,2500.0,1,1,0,0,0,0,0.0,0.0,0.0,0.0,0.0
8,9,40,25466.5,1,1,0,0,0,0,0.0,0.0,0.0,0.0,0.0
9,10,41,35409.2,1,1,1,1,1,1,0.0,0.0,0.0,0.0,0.0


In [307]:
# ============================================================
# CELL 139 — PRE-BALANCE VALIDATION
# ============================================================

print("=" * 70)
print("PRE-BALANCE VALIDATION")
print("=" * 70)

checks = {
    "Current Account": (
        controlled_customers["Has_Current_Account"] == 1
    ).sum(),

    "Savings": (
        controlled_customers["Has_Savings"] == 1
    ).sum(),

    "Credit Card": (
        controlled_customers["Has_Credit_Card"] == 1
    ).sum(),

    "Personal Loan": (
        controlled_customers["Has_Personal_Loan"] == 1
    ).sum(),

    "Mortgage": (
        controlled_customers["Has_Mortgage"] == 1
    ).sum(),

    "Credit Card Revolver": (
        controlled_customers["Credit_Card_Revolver"] == 1
    ).sum()
}

for product, count in checks.items():
    print(f"{product:<25}: {count}")

PRE-BALANCE VALIDATION
Current Account          : 9736
Savings                  : 8919
Credit Card              : 6506
Personal Loan            : 1413
Mortgage                 : 2621
Credit Card Revolver     : 1861


In [ ]:
# ============================================================
# CELL 141 — CHECK 2024 BALANCE SNAPSWHAOT
# ============================================================

required_balance_columns = [
    "Customer_ID",
    "Current_Account_Balance",
    "Savings_Balance",
    "Credit_Card_Balance",
    "Personal_Loan_Balance",
    "Mortgage_Balance"
]

print("=" * 70)
print("CHECKING 2024 BALANCE SNAPSHOT")
print("=" * 70)

missing_columns = [
    col for col in required_balance_columns
    if col not in synthetic_income.columns
]

print("Missing columns:")
print(missing_columns)

if len(missing_columns) == 0:

    balance_2024 = synthetic_income[
        required_balance_columns
    ].copy()

    print("\nRows:", len(balance_2024))
    print(
        "Unique customers:",
        balance_2024["Customer_ID"].nunique()
    )

    display(
        balance_2024.head(10)
    )

CHECKING 2024 BALANCE SNAPSHOT
Missing columns:
['Current_Account_Balance', 'Savings_Balance', 'Credit_Card_Balance', 'Personal_Loan_Balance', 'Mortgage_Balance']


In [312]:
# ============================================================
# CELL 142 — FIND DATAFRAME WITH 2024 BALANCE COLUMNS
# ============================================================

print("=" * 70)
print("SEARCHING FOR 2024 BALANCE DATA")
print("=" * 70)

balance_keywords = [
    "Current_Account_Balance",
    "Savings_Balance",
    "Credit_Card_Balance",
    "Personal_Loan_Balance",
    "Mortgage_Balance"
]

found_dataframes = []

# IMPORTANT:
# Convert globals() to a list before iterating
for name, obj in list(globals().items()):

    if isinstance(obj, pd.DataFrame):

        matching = [
            col for col in obj.columns
            if col in balance_keywords
        ]

        if matching:

            found_dataframes.append(name)

            print("\n" + "-" * 70)
            print("DATAFRAME:", name)
            print("Shape:", obj.shape)
            print("Balance columns found:", matching)

            if "Customer_ID" in obj.columns:
                print(
                    "Unique customers:",
                    obj["Customer_ID"].nunique()
                )

            if "Date" in obj.columns:
                print(
                    "Date range:",
                    obj["Date"].min(),
                    "to",
                    obj["Date"].max()
                )

print("\n" + "=" * 70)
print("DATAFRAMES FOUND")
print("=" * 70)

print(found_dataframes)

SEARCHING FOR 2024 BALANCE DATA

----------------------------------------------------------------------
DATAFRAME: savings_history
Shape: (1926504, 3)
Balance columns found: ['Savings_Balance']
Unique customers: 8919
Date range: 2008-01-31 00:00:00 to 2025-12-31 00:00:00

----------------------------------------------------------------------
DATAFRAME: current_account_history
Shape: (2102976, 3)
Balance columns found: ['Current_Account_Balance']
Unique customers: 9736
Date range: 2008-01-31 00:00:00 to 2025-12-31 00:00:00

----------------------------------------------------------------------
DATAFRAME: credit_card_history
Shape: (401976, 3)
Balance columns found: ['Credit_Card_Balance']
Unique customers: 1861
Date range: 2008-01-31 00:00:00 to 2025-12-31 00:00:00

----------------------------------------------------------------------
DATAFRAME: personal_loan_history
Shape: (305208, 3)
Balance columns found: ['Personal_Loan_Balance']
Unique customers: 1413
Date range: 2008-01-31 00:00:

In [313]:
# ============================================================
# CELL 143 — EXTRACT DECEMBER 2024 BALANCE SNAPSHOT
# ============================================================

print("=" * 70)
print("DECEMBER 2024 BALANCE SNAPSHOT")
print("=" * 70)

baseline_date = pd.Timestamp("2024-12-31")


# ------------------------------------------------------------
# Current Account
# ------------------------------------------------------------

current_2024 = current_account_history[
    current_account_history["Date"] == baseline_date
][
    [
        "Customer_ID",
        "Current_Account_Balance"
    ]
].copy()


# ------------------------------------------------------------
# Savings
# ------------------------------------------------------------

savings_2024 = savings_history[
    savings_history["Date"] == baseline_date
][
    [
        "Customer_ID",
        "Savings_Balance"
    ]
].copy()


# ------------------------------------------------------------
# Credit Card
# ------------------------------------------------------------

credit_card_2024 = credit_card_history[
    credit_card_history["Date"] == baseline_date
][
    [
        "Customer_ID",
        "Credit_Card_Balance"
    ]
].copy()


# ------------------------------------------------------------
# Personal Loan
# ------------------------------------------------------------

personal_loan_2024 = personal_loan_history[
    personal_loan_history["Date"] == baseline_date
][
    [
        "Customer_ID",
        "Personal_Loan_Balance"
    ]
].copy()


# ------------------------------------------------------------
# Mortgage
# ------------------------------------------------------------

mortgage_2024 = mortgage_history[
    mortgage_history["Date"] == baseline_date
][
    [
        "Customer_ID",
        "Mortgage_Balance"
    ]
].copy()


# ============================================================
# CHECK
# ============================================================

print("\nDecember 2024 customer counts:")

print(
    "Current Account:",
    current_2024["Customer_ID"].nunique()
)

print(
    "Savings:",
    savings_2024["Customer_ID"].nunique()
)

print(
    "Credit Card Revolvers:",
    credit_card_2024["Customer_ID"].nunique()
)

print(
    "Personal Loan:",
    personal_loan_2024["Customer_ID"].nunique()
)

print(
    "Mortgage:",
    mortgage_2024["Customer_ID"].nunique()
)


print("\nRows:")

print(
    "Current Account:",
    len(current_2024)
)

print(
    "Savings:",
    len(savings_2024)
)

print(
    "Credit Card:",
    len(credit_card_2024)
)

print(
    "Personal Loan:",
    len(personal_loan_2024)
)

print(
    "Mortgage:",
    len(mortgage_2024)
)

DECEMBER 2024 BALANCE SNAPSHOT

December 2024 customer counts:
Current Account: 9736
Savings: 8919
Credit Card Revolvers: 1861
Personal Loan: 1413
Mortgage: 2621

Rows:
Current Account: 9736
Savings: 8919
Credit Card: 1861
Personal Loan: 1413
Mortgage: 2621


In [317]:
# ============================================================
# CHECK EXACT COLUMNS IN THE DECEMBER 2024 DATAFRAMES
# ============================================================

print("=" * 70)
print("CHECKING DECEMBER 2024 BALANCE DATAFRAMES")
print("=" * 70)

dataframes_to_check = {
    "current_2024": current_2024,
    "savings_2024": savings_2024,
    "credit_card_2024": credit_card_2024,
    "personal_loan_2024": personal_loan_2024,
    "mortgage_2024": mortgage_2024
}

for name, df in dataframes_to_check.items():

    print("\n" + "-" * 70)
    print(name)

    print("Shape:", df.shape)

    print("Columns:")
    print(df.columns.tolist())

    print("\nFirst 3 rows:")
    display(df.head(3))

CHECKING DECEMBER 2024 BALANCE DATAFRAMES

----------------------------------------------------------------------
current_2024
Shape: (9736, 2)
Columns:
['Customer_ID', 'Current_Account_Balance']

First 3 rows:


,Customer_ID,Current_Account_Balance
203,1.0,1172.0
419,2.0,12254.0
635,3.0,2511.0



----------------------------------------------------------------------
savings_2024
Shape: (8919, 2)
Columns:
['Customer_ID', 'Savings_Balance']

First 3 rows:


,Customer_ID,Savings_Balance
203,1.0,2772.0
419,2.0,98029.0
635,3.0,9491.0



----------------------------------------------------------------------
credit_card_2024
Shape: (1861, 2)
Columns:
['Customer_ID', 'Credit_Card_Balance']

First 3 rows:


,Customer_ID,Credit_Card_Balance
203,17.0,4636.29
419,23.0,3483.98
635,27.0,4151.67



----------------------------------------------------------------------
personal_loan_2024
Shape: (1413, 2)
Columns:
['Customer_ID', 'Personal_Loan_Balance']

First 3 rows:


,Customer_ID,Personal_Loan_Balance
203,2.0,23826.86
419,3.0,8227.58
635,19.0,9084.51



----------------------------------------------------------------------
mortgage_2024
Shape: (2621, 2)
Columns:
['Customer_ID', 'Mortgage_Balance']

First 3 rows:


,Customer_ID,Mortgage_Balance
203,1.0,15981.17
419,2.0,287442.30
635,13.0,68254.94


In [321]:
# ============================================================
# CELL 144 — FINAL DECEMBER 2024 BALANCE ATTACHMENT
# ============================================================

print("=" * 70)
print("FINAL DECEMBER 2024 BALANCE ATTACHMENT")
print("=" * 70)


# ============================================================
# BALANCE COLUMN NAMES
# ============================================================

balance_columns = [
    "Current_Account_Balance",
    "Savings_Balance",
    "Credit_Card_Balance",
    "Personal_Loan_Balance",
    "Mortgage_Balance"
]


# ============================================================
# START FROM CONTROLLED CUSTOMERS
# ============================================================

final_2024 = controlled_customers.copy()

print("\nStarting columns:")
print(final_2024.columns.tolist())


# ============================================================
# REMOVE ANY OLD BALANCE COLUMNS
# ============================================================

columns_to_remove = []

for col in final_2024.columns:

    for balance_col in balance_columns:

        if (
            col == balance_col
            or col == balance_col + "_x"
            or col == balance_col + "_y"
        ):
            columns_to_remove.append(col)


if columns_to_remove:

    print("\nRemoving existing balance columns:")
    print(columns_to_remove)

    final_2024 = final_2024.drop(
        columns=columns_to_remove
    )

else:

    print("\nNo existing balance columns found.")


# ============================================================
# PREPARE CLEAN DECEMBER 2024 TABLES
# ============================================================

current_clean = current_2024[
    ["Customer_ID", "Current_Account_Balance"]
].copy()

savings_clean = savings_2024[
    ["Customer_ID", "Savings_Balance"]
].copy()

credit_clean = credit_card_2024[
    ["Customer_ID", "Credit_Card_Balance"]
].copy()

loan_clean = personal_loan_2024[
    ["Customer_ID", "Personal_Loan_Balance"]
].copy()

mortgage_clean = mortgage_2024[
    ["Customer_ID", "Mortgage_Balance"]
].copy()


# ============================================================
# MERGE DECEMBER 2024 BALANCES
# ============================================================

final_2024 = final_2024.merge(
    current_clean,
    on="Customer_ID",
    how="left",
    validate="one_to_one"
)

final_2024 = final_2024.merge(
    savings_clean,
    on="Customer_ID",
    how="left",
    validate="one_to_one"
)

final_2024 = final_2024.merge(
    credit_clean,
    on="Customer_ID",
    how="left",
    validate="one_to_one"
)

final_2024 = final_2024.merge(
    loan_clean,
    on="Customer_ID",
    how="left",
    validate="one_to_one"
)

final_2024 = final_2024.merge(
    mortgage_clean,
    on="Customer_ID",
    how="left",
    validate="one_to_one"
)


# ============================================================
# FILL MISSING BALANCES
# ============================================================

for col in balance_columns:

    final_2024[col] = pd.to_numeric(
        final_2024[col],
        errors="coerce"
    ).fillna(0.0)


# ============================================================
# VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("DECEMBER 2024 BALANCE SNAPSHOT CREATED")
print("=" * 70)

print(
    "\nRows:",
    len(final_2024)
)

print(
    "Unique customers:",
    final_2024["Customer_ID"].nunique()
)

print(
    "Duplicate Customer_ID:",
    final_2024["Customer_ID"].duplicated().sum()
)


print("\nBalance validation:")

for col in balance_columns:

    print(
        f"{col:<30} "
        f"Missing: {final_2024[col].isna().sum():,} "
        f"Positive: {(final_2024[col] > 0).sum():,}"
    )


print("\nFinal columns:")
print(final_2024.columns.tolist())


display(
    final_2024.head(20)
)

FINAL DECEMBER 2024 BALANCE ATTACHMENT

Starting columns:
['Customer_ID', 'Age_2024', 'Annual_Income_2024', 'Has_Current_Account', 'Has_Savings', 'Has_Credit_Card', 'Has_Personal_Loan', 'Has_Mortgage', 'Credit_Card_Revolver', 'Current_Account_Balance', 'Savings_Balance', 'Credit_Card_Balance', 'Personal_Loan_Balance', 'Mortgage_Balance']

Removing existing balance columns:
['Current_Account_Balance', 'Savings_Balance', 'Credit_Card_Balance', 'Personal_Loan_Balance', 'Mortgage_Balance']

DECEMBER 2024 BALANCE SNAPSHOT CREATED

Rows: 10000
Unique customers: 10000
Duplicate Customer_ID: 0

Balance validation:
Current_Account_Balance        Missing: 0 Positive: 9,736
Savings_Balance                Missing: 0 Positive: 8,919
Credit_Card_Balance            Missing: 0 Positive: 1,861
Personal_Loan_Balance          Missing: 0 Positive: 1,413
Mortgage_Balance               Missing: 0 Positive: 2,621

Final columns:
['Customer_ID', 'Age_2024', 'Annual_Income_2024', 'Has_Current_Account', 'Has_Sa

,Customer_ID,Age_2024,Annual_Income_2024,Has_Current_Account,Has_Savings,Has_Credit_Card,Has_Personal_Loan,Has_Mortgage,Credit_Card_Revolver,Current_Account_Balance,Savings_Balance,Credit_Card_Balance,Personal_Loan_Balance,Mortgage_Balance
0,1,28,9473.7,1,1,1,0,1,1,1172.0,2772.0,0.00,0.00,15981.17
1,2,42,25991.0,1,1,0,1,1,0,12254.0,98029.0,0.00,23826.86,287442.30
2,3,29,6200.0,1,1,1,0,1,0,2511.0,9491.0,0.00,8227.58,0.00
3,4,47,16453.0,1,1,1,1,0,0,1836.0,4722.0,0.00,0.00,0.00
4,5,19,8435.3,1,1,1,1,0,1,517.0,1237.0,0.00,0.00,0.00
5,6,60,33537.6,1,1,1,0,0,0,517.0,0.0,0.00,0.00,0.00
6,7,24,8183.0,1,1,1,0,0,1,292.0,584.0,0.00,0.00,0.00
7,8,18,2500.0,1,1,0,0,0,0,5028.0,46093.0,0.00,0.00,0.00
8,9,40,25466.5,1,1,0,0,0,0,1838.0,7879.0,0.00,0.00,0.00
9,10,41,35409.2,1,1,1,1,1,1,2484.0,9389.0,0.00,0.00,0.00


In [322]:
# ============================================================
# CELL 145 — FINAL DECEMBER 2024 LOGICAL VALIDATION
# ============================================================

print("=" * 70)
print("FINAL DECEMBER 2024 LOGICAL VALIDATION")
print("=" * 70)


# ------------------------------------------------------------
# Current Account
# ------------------------------------------------------------

current_violation = (
    (final_2024["Has_Current_Account"] == 0) &
    (final_2024["Current_Account_Balance"] > 0)
).sum()


# ------------------------------------------------------------
# Savings
# ------------------------------------------------------------

savings_violation = (
    (final_2024["Has_Savings"] == 0) &
    (final_2024["Savings_Balance"] > 0)
).sum()


# ------------------------------------------------------------
# Credit Card
# ------------------------------------------------------------

credit_card_violation = (
    (final_2024["Has_Credit_Card"] == 0) &
    (final_2024["Credit_Card_Balance"] > 0)
).sum()


# ------------------------------------------------------------
# Personal Loan
# ------------------------------------------------------------

personal_loan_violation = (
    (final_2024["Has_Personal_Loan"] == 0) &
    (final_2024["Personal_Loan_Balance"] > 0)
).sum()


# ------------------------------------------------------------
# Mortgage
# ------------------------------------------------------------

mortgage_violation = (
    (final_2024["Has_Mortgage"] == 0) &
    (final_2024["Mortgage_Balance"] > 0)
).sum()


# ------------------------------------------------------------
# Revolver
# ------------------------------------------------------------

revolver_violation = (
    (final_2024["Credit_Card_Revolver"] == 0) &
    (final_2024["Credit_Card_Balance"] > 0)
).sum()


# ============================================================
# RESULTS
# ============================================================

print("\nCurrent Account violations:",
      current_violation)

print("Savings violations:",
      savings_violation)

print("Credit Card violations:",
      credit_card_violation)

print("Personal Loan violations:",
      personal_loan_violation)

print("Mortgage violations:",
      mortgage_violation)

print("Credit Card Revolver violations:",
      revolver_violation)


total_violations = (
    current_violation +
    savings_violation +
    credit_card_violation +
    personal_loan_violation +
    mortgage_violation +
    revolver_violation
)


print("\n" + "=" * 70)
print("TOTAL LOGICAL VIOLATIONS")
print("=" * 70)

print("Total:", total_violations)

print(
    "Violation rate:",
    f"{total_violations / len(final_2024) * 100:.2f}%"
)

FINAL DECEMBER 2024 LOGICAL VALIDATION

Current Account violations: 260
Savings violations: 960
Credit Card violations: 661
Personal Loan violations: 1205
Mortgage violations: 1920
Credit Card Revolver violations: 1517

TOTAL LOGICAL VIOLATIONS
Total: 6523
Violation rate: 65.23%


In [323]:
# ============================================================
# CELL 146 — ALIGN OWNERSHIP WITH DECEMBER 2024 BALANCE HOLDERS
# ============================================================

print("=" * 70)
print("ALIGNING OWNERSHIP WITH DECEMBER 2024 BALANCE HOLDERS")
print("=" * 70)


# ------------------------------------------------------------
# Make a fresh copy
# ------------------------------------------------------------

final_2024 = controlled_customers.copy()


# ------------------------------------------------------------
# HOLDER ID SETS FROM DECEMBER 2024
# ------------------------------------------------------------

current_ids = set(
    current_2024["Customer_ID"]
)

savings_ids = set(
    savings_2024["Customer_ID"]
)

personal_loan_ids = set(
    personal_loan_2024["Customer_ID"]
)

mortgage_ids = set(
    mortgage_2024["Customer_ID"]
)

revolver_ids = set(
    credit_card_2024["Customer_ID"]
)


# ------------------------------------------------------------
# ALIGN OWNERSHIP
# ------------------------------------------------------------

final_2024["Has_Current_Account"] = (
    final_2024["Customer_ID"]
    .isin(current_ids)
    .astype(int)
)

final_2024["Has_Savings"] = (
    final_2024["Customer_ID"]
    .isin(savings_ids)
    .astype(int)
)

final_2024["Has_Personal_Loan"] = (
    final_2024["Customer_ID"]
    .isin(personal_loan_ids)
    .astype(int)
)

final_2024["Has_Mortgage"] = (
    final_2024["Customer_ID"]
    .isin(mortgage_ids)
    .astype(int)
)


# ------------------------------------------------------------
# CREDIT CARD
#
# Keep the existing 6,506 credit-card ownership.
# The previous validation showed that all revolvers
# already belonged to credit-card customers.
# ------------------------------------------------------------

final_2024["Has_Credit_Card"] = (
    final_2024["Has_Credit_Card"]
    .astype(int)
)

final_2024["Credit_Card_Revolver"] = (
    final_2024["Customer_ID"]
    .isin(revolver_ids)
    .astype(int)
)


# ------------------------------------------------------------
# ATTACH BALANCES
# ------------------------------------------------------------

balance_columns = [
    "Current_Account_Balance",
    "Savings_Balance",
    "Credit_Card_Balance",
    "Personal_Loan_Balance",
    "Mortgage_Balance"
]


# Remove old balances if they exist
final_2024 = final_2024.drop(
    columns=[
        col for col in balance_columns
        if col in final_2024.columns
    ],
    errors="ignore"
)


# Current Account
final_2024 = final_2024.merge(
    current_2024[
        ["Customer_ID", "Current_Account_Balance"]
    ],
    on="Customer_ID",
    how="left",
    validate="one_to_one"
)


# Savings
final_2024 = final_2024.merge(
    savings_2024[
        ["Customer_ID", "Savings_Balance"]
    ],
    on="Customer_ID",
    how="left",
    validate="one_to_one"
)


# Credit Card
final_2024 = final_2024.merge(
    credit_card_2024[
        ["Customer_ID", "Credit_Card_Balance"]
    ],
    on="Customer_ID",
    how="left",
    validate="one_to_one"
)


# Personal Loan
final_2024 = final_2024.merge(
    personal_loan_2024[
        ["Customer_ID", "Personal_Loan_Balance"]
    ],
    on="Customer_ID",
    how="left",
    validate="one_to_one"
)


# Mortgage
final_2024 = final_2024.merge(
    mortgage_2024[
        ["Customer_ID", "Mortgage_Balance"]
    ],
    on="Customer_ID",
    how="left",
    validate="one_to_one"
)


# ------------------------------------------------------------
# MISSING BALANCES = ZERO
# ------------------------------------------------------------

for col in balance_columns:

    final_2024[col] = pd.to_numeric(
        final_2024[col],
        errors="coerce"
    ).fillna(0.0)


# ============================================================
# CHECK OWNERSHIP COUNTS
# ============================================================

print("\n" + "=" * 70)
print("OWNERSHIP COUNTS AFTER ALIGNMENT")
print("=" * 70)

print(
    "Current Account:",
    final_2024["Has_Current_Account"].sum()
)

print(
    "Savings:",
    final_2024["Has_Savings"].sum()
)

print(
    "Credit Card:",
    final_2024["Has_Credit_Card"].sum()
)

print(
    "Personal Loan:",
    final_2024["Has_Personal_Loan"].sum()
)

print(
    "Mortgage:",
    final_2024["Has_Mortgage"].sum()
)

print(
    "Credit Card Revolver:",
    final_2024["Credit_Card_Revolver"].sum()
)


# ============================================================
# STRUCTURAL CHECK
# ============================================================

print("\n" + "=" * 70)
print("STRUCTURAL CHECK")
print("=" * 70)

print("Rows:", len(final_2024))

print(
    "Unique customers:",
    final_2024["Customer_ID"].nunique()
)

print(
    "Duplicate Customer_ID:",
    final_2024["Customer_ID"].duplicated().sum()
)

ALIGNING OWNERSHIP WITH DECEMBER 2024 BALANCE HOLDERS

OWNERSHIP COUNTS AFTER ALIGNMENT
Current Account: 9736
Savings: 8919
Credit Card: 6506
Personal Loan: 1413
Mortgage: 2621
Credit Card Revolver: 1861

STRUCTURAL CHECK
Rows: 10000
Unique customers: 10000
Duplicate Customer_ID: 0


In [324]:
# ============================================================
# CELL 147 — FINAL LOGICAL VALIDATION AFTER ALIGNMENT
# ============================================================

print("=" * 70)
print("FINAL LOGICAL VALIDATION AFTER OWNERSHIP ALIGNMENT")
print("=" * 70)


# ------------------------------------------------------------
# Current Account
# ------------------------------------------------------------

current_violation = (
    (final_2024["Has_Current_Account"] == 0) &
    (final_2024["Current_Account_Balance"] > 0)
).sum()


# ------------------------------------------------------------
# Savings
# ------------------------------------------------------------

savings_violation = (
    (final_2024["Has_Savings"] == 0) &
    (final_2024["Savings_Balance"] > 0)
).sum()


# ------------------------------------------------------------
# Credit Card
# ------------------------------------------------------------

credit_card_violation = (
    (final_2024["Has_Credit_Card"] == 0) &
    (final_2024["Credit_Card_Balance"] > 0)
).sum()


# ------------------------------------------------------------
# Personal Loan
# ------------------------------------------------------------

personal_loan_violation = (
    (final_2024["Has_Personal_Loan"] == 0) &
    (final_2024["Personal_Loan_Balance"] > 0)
).sum()


# ------------------------------------------------------------
# Mortgage
# ------------------------------------------------------------

mortgage_violation = (
    (final_2024["Has_Mortgage"] == 0) &
    (final_2024["Mortgage_Balance"] > 0)
).sum()


# ------------------------------------------------------------
# Revolver
# ------------------------------------------------------------

revolver_violation = (
    (final_2024["Credit_Card_Revolver"] == 0) &
    (final_2024["Credit_Card_Balance"] > 0)
).sum()


# ============================================================
# RESULTS
# ============================================================

print("\nCurrent Account violations:",
      current_violation)

print("Savings violations:",
      savings_violation)

print("Credit Card violations:",
      credit_card_violation)

print("Personal Loan violations:",
      personal_loan_violation)

print("Mortgage violations:",
      mortgage_violation)

print("Credit Card Revolver violations:",
      revolver_violation)


# ============================================================
# TOTAL
# ============================================================

total_violations = (
    current_violation +
    savings_violation +
    credit_card_violation +
    personal_loan_violation +
    mortgage_violation +
    revolver_violation
)

print("\n" + "=" * 70)
print("TOTAL LOGICAL VIOLATIONS")
print("=" * 70)

print("Total:", total_violations)

print(
    "Violation rate:",
    f"{total_violations / len(final_2024) * 100:.2f}%"
)

FINAL LOGICAL VALIDATION AFTER OWNERSHIP ALIGNMENT

Current Account violations: 0
Savings violations: 0
Credit Card violations: 661
Personal Loan violations: 0
Mortgage violations: 0
Credit Card Revolver violations: 0

TOTAL LOGICAL VIOLATIONS
Total: 661
Violation rate: 6.61%


In [325]:
# ============================================================
# CELL 148 — ALIGN CREDIT CARD OWNERSHIP WITH REVOLVERS
# ============================================================

print("=" * 70)
print("FIXING CREDIT CARD OWNERSHIP")
print("=" * 70)


# ------------------------------------------------------------
# Existing credit-card customers
# ------------------------------------------------------------

existing_card_ids = set(
    final_2024.loc[
        final_2024["Has_Credit_Card"] == 1,
        "Customer_ID"
    ]
)


# ------------------------------------------------------------
# Actual revolving customers
# ------------------------------------------------------------

revolver_ids = set(
    final_2024.loc[
        final_2024["Credit_Card_Revolver"] == 1,
        "Customer_ID"
    ]
)


print("\nExisting credit-card customers:",
      len(existing_card_ids))

print("Revolving customers:",
      len(revolver_ids))

print(
    "Revolvers already marked as cardholders:",
    len(existing_card_ids.intersection(revolver_ids))
)

print(
    "Revolvers missing from cardholders:",
    len(revolver_ids - existing_card_ids)
)


# ============================================================
# TARGET
# ============================================================

target_credit_cards = 6506


# ------------------------------------------------------------
# Revolvers MUST be credit-card holders
# ------------------------------------------------------------

non_revolver_existing_cards = (
    existing_card_ids - revolver_ids
)


required_non_revolver_cards = (
    target_credit_cards - len(revolver_ids)
)


print(
    "\nRequired non-revolver cardholders:",
    required_non_revolver_cards
)


# ------------------------------------------------------------
# Select non-revolving cardholders
# ------------------------------------------------------------

# Keep the required number from the existing
# non-revolving cardholder population.

selected_non_revolver_cards = set(
    sorted(non_revolver_existing_cards)
    [:required_non_revolver_cards]
)


# ------------------------------------------------------------
# Final credit-card population
# ------------------------------------------------------------

final_credit_card_ids = (
    revolver_ids |
    selected_non_revolver_cards
)


# ------------------------------------------------------------
# Assign ownership
# ------------------------------------------------------------

final_2024["Has_Credit_Card"] = (
    final_2024["Customer_ID"]
    .isin(final_credit_card_ids)
    .astype(int)
)


# ============================================================
# VALIDATION
# ============================================================

print("\n" + "=" * 70)
print("CREDIT CARD OWNERSHIP AFTER ALIGNMENT")
print("=" * 70)

print(
    "Credit-card customers:",
    final_2024["Has_Credit_Card"].sum()
)

print(
    "Revolving customers:",
    final_2024["Credit_Card_Revolver"].sum()
)


# ------------------------------------------------------------
# Check revolver ownership
# ------------------------------------------------------------

revolver_without_card = (
    (final_2024["Credit_Card_Revolver"] == 1) &
    (final_2024["Has_Credit_Card"] == 0)
).sum()

print(
    "Revolvers without credit card:",
    revolver_without_card
)

FIXING CREDIT CARD OWNERSHIP

Existing credit-card customers: 6506
Revolving customers: 1861
Revolvers already marked as cardholders: 1200
Revolvers missing from cardholders: 661

Required non-revolver cardholders: 4645

CREDIT CARD OWNERSHIP AFTER ALIGNMENT
Credit-card customers: 6506
Revolving customers: 1861
Revolvers without credit card: 0


In [326]:
# ============================================================
# CELL 149 — FINAL LOGICAL VALIDATION
# ============================================================

print("=" * 70)
print("FINAL DECEMBER 2024 LOGICAL VALIDATION")
print("=" * 70)


current_violation = (
    (final_2024["Has_Current_Account"] == 0) &
    (final_2024["Current_Account_Balance"] > 0)
).sum()

savings_violation = (
    (final_2024["Has_Savings"] == 0) &
    (final_2024["Savings_Balance"] > 0)
).sum()

credit_card_violation = (
    (final_2024["Has_Credit_Card"] == 0) &
    (final_2024["Credit_Card_Balance"] > 0)
).sum()

personal_loan_violation = (
    (final_2024["Has_Personal_Loan"] == 0) &
    (final_2024["Personal_Loan_Balance"] > 0)
).sum()

mortgage_violation = (
    (final_2024["Has_Mortgage"] == 0) &
    (final_2024["Mortgage_Balance"] > 0)
).sum()

revolver_violation = (
    (final_2024["Credit_Card_Revolver"] == 0) &
    (final_2024["Credit_Card_Balance"] > 0)
).sum()


print("\nCurrent Account violations:",
      current_violation)

print("Savings violations:",
      savings_violation)

print("Credit Card violations:",
      credit_card_violation)

print("Personal Loan violations:",
      personal_loan_violation)

print("Mortgage violations:",
      mortgage_violation)

print("Credit Card Revolver violations:",
      revolver_violation)


total_violations = (
    current_violation
    + savings_violation
    + credit_card_violation
    + personal_loan_violation
    + mortgage_violation
    + revolver_violation
)


print("\n" + "=" * 70)
print("TOTAL LOGICAL VIOLATIONS")
print("=" * 70)

print("Total:", total_violations)

print(
    "Violation rate:",
    f"{total_violations / len(final_2024) * 100:.2f}%"
)

FINAL DECEMBER 2024 LOGICAL VALIDATION

Current Account violations: 0
Savings violations: 0
Credit Card violations: 0
Personal Loan violations: 0
Mortgage violations: 0
Credit Card Revolver violations: 0

TOTAL LOGICAL VIOLATIONS
Total: 0
Violation rate: 0.00%


In [327]:
# ============================================================
# CELL 150 — DECEMBER 2024 SYNTHETIC PORTFOLIO TOTALS
# ============================================================

print("=" * 70)
print("DECEMBER 2024 SYNTHETIC PORTFOLIO TOTALS")
print("=" * 70)

portfolio_columns = {
    "Credit Card": "Credit_Card_Balance",
    "Personal Loan": "Personal_Loan_Balance",
    "Mortgage": "Mortgage_Balance"
}

for product, column in portfolio_columns.items():

    total = final_2024[column].sum()

    customers = (
        final_2024[column] > 0
    ).sum()

    average = (
        total / customers
        if customers > 0
        else 0
    )

    print("\n" + "-" * 70)
    print(product)

    print(
        f"Synthetic customers: {customers:,}"
    )

    print(
        f"Synthetic total: £{total:,.2f}"
    )

    print(
        f"Average balance: £{average:,.2f}"
    )

DECEMBER 2024 SYNTHETIC PORTFOLIO TOTALS

----------------------------------------------------------------------
Credit Card
Synthetic customers: 1,861
Synthetic total: £9,705,760.05
Average balance: £5,215.35

----------------------------------------------------------------------
Personal Loan
Synthetic customers: 1,413
Synthetic total: £13,102,187.82
Average balance: £9,272.60

----------------------------------------------------------------------
Mortgage
Synthetic customers: 2,621
Synthetic total: £260,812,169.09
Average balance: £99,508.65


In [328]:
# ============================================================
# CELL 151 — POPULATION-SCALED BoE RECONCILIATION
# ============================================================

print("=" * 70)
print("POPULATION-SCALED BoE RECONCILIATION")
print("=" * 70)


# ============================================================
# UK POPULATION TARGETS
# ============================================================

fca_population = {
    "Credit Card": 10_100_000,
    "Personal Loan": 7_300_000,
    "Mortgage": 14_800_000
}


# ============================================================
# BoE PORTFOLIO TARGETS
# ============================================================

boe_targets = {
    "Credit Card": 52_675_000_000,
    "Personal Loan": 67_690_000_000,
    "Mortgage": 1_472_728_000_000
}


# ============================================================
# SYNTHETIC TOTALS
# ============================================================

synthetic_totals = {
    "Credit Card":
        final_2024["Credit_Card_Balance"].sum(),

    "Personal Loan":
        final_2024["Personal_Loan_Balance"].sum(),

    "Mortgage":
        final_2024["Mortgage_Balance"].sum()
}


# ============================================================
# CALCULATE SCALED PORTFOLIOS
# ============================================================

scaled_portfolios = {}


for product in synthetic_totals:

    synthetic_total = synthetic_totals[product]

    population = fca_population[product]

    target = boe_targets[product]

    scaled_total = (
        synthetic_total
        / 10000
        * population
    )

    difference = scaled_total - target

    difference_pct = (
        difference / target * 100
    )

    scaled_portfolios[product] = scaled_total


    print("\n" + "-" * 70)
    print(product)

    print(
        f"Synthetic total (£): "
        f"{synthetic_total:,.2f}"
    )

    print(
        f"FCA population: "
        f"{population:,}"
    )

    print(
        f"Scaled UK portfolio (£): "
        f"{scaled_total:,.2f}"
    )

    print(
        f"BoE target (£): "
        f"{target:,.2f}"
    )

    print(
        f"Difference (£): "
        f"{difference:,.2f}"
    )

    print(
        f"Difference (%): "
        f"{difference_pct:.4f}%"
    )


# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("RECONCILIATION SUMMARY")
print("=" * 70)

for product in scaled_portfolios:

    print(
        f"{product:<20} "
        f"Scaled: £{scaled_portfolios[product]:,.2f} | "
        f"Target: £{boe_targets[product]:,.2f}"
    )

POPULATION-SCALED BoE RECONCILIATION

----------------------------------------------------------------------
Credit Card
Synthetic total (£): 9,705,760.05
FCA population: 10,100,000
Scaled UK portfolio (£): 9,802,817,650.50
BoE target (£): 52,675,000,000.00
Difference (£): -42,872,182,349.50
Difference (%): -81.3900%

----------------------------------------------------------------------
Personal Loan
Synthetic total (£): 13,102,187.82
FCA population: 7,300,000
Scaled UK portfolio (£): 9,564,597,108.60
BoE target (£): 67,690,000,000.00
Difference (£): -58,125,402,891.40
Difference (%): -85.8700%

----------------------------------------------------------------------
Mortgage
Synthetic total (£): 260,812,169.09
FCA population: 14,800,000
Scaled UK portfolio (£): 386,002,010,253.20
BoE target (£): 1,472,728,000,000.00
Difference (£): -1,086,725,989,746.80
Difference (%): -73.7900%

RECONCILIATION SUMMARY
Credit Card          Scaled: £9,802,817,650.50 | Target: £52,675,000,000.00
Personal

In [329]:
# ============================================================
# CELL 152 — CORRECT HOLDER-BASED BoE RECONCILIATION
# ============================================================

print("=" * 70)
print("CORRECT HOLDER-BASED BoE RECONCILIATION")
print("=" * 70)


# ============================================================
# UK PRODUCT POPULATIONS
# ============================================================

uk_population = {
    "Credit Card": 10_100_000,
    "Personal Loan": 7_300_000,
    "Mortgage": 14_800_000
}


# ============================================================
# BoE TARGETS
# ============================================================

boe_targets = {
    "Credit Card": 52_675_000_000,
    "Personal Loan": 67_690_000_000,
    "Mortgage": 1_472_728_000_000
}


# ============================================================
# SYNTHETIC DATA
# ============================================================

products = {
    "Credit Card": "Credit_Card_Balance",
    "Personal Loan": "Personal_Loan_Balance",
    "Mortgage": "Mortgage_Balance"
}


results = []


for product, column in products.items():

    # Only customers with a positive balance
    positive_balances = final_2024.loc[
        final_2024[column] > 0,
        column
    ]

    synthetic_customers = len(
        positive_balances
    )

    synthetic_total = (
        positive_balances.sum()
    )

    average_balance = (
        synthetic_total /
        synthetic_customers
    )

    population = uk_population[product]

    boe_target = boe_targets[product]

    # --------------------------------------------------------
    # CORRECT SCALING
    # Average balance per product holder
    # × UK product population
    # --------------------------------------------------------

    scaled_portfolio = (
        average_balance *
        population
    )

    difference = (
        scaled_portfolio -
        boe_target
    )

    difference_pct = (
        difference /
        boe_target *
        100
    )


    results.append({
        "Product": product,
        "Synthetic_Customers": synthetic_customers,
        "Synthetic_Total": synthetic_total,
        "Average_Balance": average_balance,
        "UK_Population": population,
        "Scaled_Portfolio": scaled_portfolio,
        "BoE_Target": boe_target,
        "Difference": difference,
        "Difference_Percent": difference_pct
    })


    # --------------------------------------------------------
    # PRINT
    # --------------------------------------------------------

    print("\n" + "-" * 70)
    print(product)

    print(
        f"Synthetic customers: "
        f"{synthetic_customers:,}"
    )

    print(
        f"Synthetic total (£): "
        f"{synthetic_total:,.2f}"
    )

    print(
        f"Average synthetic balance (£): "
        f"{average_balance:,.2f}"
    )

    print(
        f"UK population: "
        f"{population:,}"
    )

    print(
        f"Scaled UK portfolio (£): "
        f"{scaled_portfolio:,.2f}"
    )

    print(
        f"BoE target (£): "
        f"{boe_target:,.2f}"
    )

    print(
        f"Difference (£): "
        f"{difference:,.2f}"
    )

    print(
        f"Difference (%): "
        f"{difference_pct:.6f}%"
    )


# ============================================================
# SUMMARY
# ============================================================

reconciliation_df = pd.DataFrame(results)

print("\n" + "=" * 70)
print("RECONCILIATION SUMMARY")
print("=" * 70)

display(
    reconciliation_df[
        [
            "Product",
            "Synthetic_Customers",
            "Average_Balance",
            "Scaled_Portfolio",
            "BoE_Target",
            "Difference",
            "Difference_Percent"
        ]
    ]
)

CORRECT HOLDER-BASED BoE RECONCILIATION

----------------------------------------------------------------------
Credit Card
Synthetic customers: 1,861
Synthetic total (£): 9,705,760.05
Average synthetic balance (£): 5,215.35
UK population: 10,100,000
Scaled UK portfolio (£): 52,675,000,808.71
BoE target (£): 52,675,000,000.00
Difference (£): 808.71
Difference (%): 0.000002%

----------------------------------------------------------------------
Personal Loan
Synthetic customers: 1,413
Synthetic total (£): 13,102,187.82
Average synthetic balance (£): 9,272.60
UK population: 7,300,000
Scaled UK portfolio (£): 67,690,000,768.58
BoE target (£): 67,690,000,000.00
Difference (£): 768.58
Difference (%): 0.000001%

----------------------------------------------------------------------
Mortgage
Synthetic customers: 2,621
Synthetic total (£): 260,812,169.09
Average synthetic balance (£): 99,508.65
UK population: 14,800,000
Scaled UK portfolio (£): 1,472,728,005,544.45
BoE target (£): 1,472,728,0

,Product,Synthetic_Customers,Average_Balance,Scaled_Portfolio,BoE_Target,Difference,Difference_Percent
0,Credit Card,1861,5215.346615,5.267500e+10,52675000000,808.705002,1.535273e-06
1,Personal Loan,1413,9272.602845,6.769000e+10,67690000000,768.577492,1.135437e-06
2,Mortgage,2621,99508.649023,1.472728e+12,1472728000000,5544.448486,3.764747e-07


In [332]:
# ============================================================
# CELL 154 — LOAD AND VALIDATE FINAL MACRO DATA
# ============================================================

print("=" * 70)
print("LOADING FINAL MACROECONOMIC + BANKING DATA")
print("=" * 70)

macro_df = pd.read_csv(
    "Final_DS.csv"
)

# ------------------------------------------------------------
# Convert Date
# ------------------------------------------------------------

macro_df["Date"] = pd.to_datetime(
    macro_df["Date"],
    format="%Y-%m"
)


# ------------------------------------------------------------
# Sort chronologically
# ------------------------------------------------------------

macro_df = macro_df.sort_values(
    "Date"
).reset_index(drop=True)


# ============================================================
# BASIC VALIDATION
# ============================================================

print("\nShape:")
print(macro_df.shape)

print("\nDate range:")
print(
    macro_df["Date"].min(),
    "to",
    macro_df["Date"].max()
)

print("\nDuplicate dates:")
print(
    macro_df["Date"].duplicated().sum()
)

print("\nMissing values:")
print(
    macro_df.isna().sum()
)

print("\nColumns:")
print(
    macro_df.columns.tolist()
)


# ============================================================
# CHECK MONTHLY CONTINUITY
# ============================================================

expected_months = pd.date_range(
    start=macro_df["Date"].min(),
    end=macro_df["Date"].max(),
    freq="MS"
)

actual_months = (
    macro_df["Date"]
    .dt.to_period("M")
)

expected_periods = (
    expected_months
    .to_period("M")
)

missing_months = (
    expected_periods
    .difference(actual_months)
)


print("\nExpected monthly observations:",
      len(expected_months))

print("Actual observations:",
      len(macro_df))

print("Missing months:",
      len(missing_months))

if len(missing_months) > 0:
    print("\nMissing months:")
    print(missing_months.tolist())


# ============================================================
# DISPLAY
# ============================================================

print("\nFirst 5 months:")
display(
    macro_df.head()
)

print("\nLast 5 months:")
display(
    macro_df.tail()
)

LOADING FINAL MACROECONOMIC + BANKING DATA

Shape:
(216, 14)

Date range:
2008-01-01 00:00:00 to 2025-12-01 00:00:00

Duplicate dates:
0

Missing values:
Date                   0
Year                   0
Month                  0
Bank_Rate              0
CPI                    0
Unemployment_Rate      0
House_Price_Index      0
GDP_Growth             0
Current_Accounts       0
Savings_Accounts       0
Mortgage_Approvals     0
Net_Consumer_Credit    0
Credit_Card_Lending    0
Consumer_Credit        0
dtype: int64

Columns:
['Date', 'Year', 'Month', 'Bank_Rate', 'CPI', 'Unemployment_Rate', 'House_Price_Index', 'GDP_Growth', 'Current_Accounts', 'Savings_Accounts', 'Mortgage_Approvals', 'Net_Consumer_Credit', 'Credit_Card_Lending', 'Consumer_Credit']

Expected monthly observations: 216
Actual observations: 216
Missing months: 0

First 5 months:


,Date,Year,Month,Bank_Rate,CPI,Unemployment_Rate,House_Price_Index,GDP_Growth,Current_Accounts,Savings_Accounts,Mortgage_Approvals,Net_Consumer_Credit,Credit_Card_Lending,Consumer_Credit
0,2008-01-01,2008,January,5.50,2.2,5.2,65.6,0.4,33417,224159,71326,288,183,105
1,2008-02-01,2008,February,5.25,2.5,5.2,65.0,0.4,31492,226772,71064,1200,363,837
2,2008-03-01,2008,March,5.25,2.5,5.3,64.5,0.4,32823,227769,60454,432,319,113
3,2008-04-01,2008,April,5.00,3.0,5.2,64.7,-0.5,31895,233766,57134,514,179,335
4,2008-05-01,2008,May,5.00,3.3,5.4,65.1,-0.5,32210,237820,40546,370,752,-382



Last 5 months:


,Date,Year,Month,Bank_Rate,CPI,Unemployment_Rate,House_Price_Index,GDP_Growth,Current_Accounts,Savings_Accounts,Mortgage_Approvals,Net_Consumer_Credit,Credit_Card_Lending,Consumer_Credit
211,2025-08-01,2025,August,4.00,3.8,5.0,103.9,0.2,299456,263815,64468,1016,755,261
212,2025-09-01,2025,September,4.00,3.8,5.1,103.6,0.2,300172,262299,65432,800,710,90
213,2025-10-01,2025,October,4.00,3.6,5.1,104.0,0.2,298370,262686,64561,1103,649,454
214,2025-11-01,2025,November,4.00,3.2,5.2,104.3,0.2,300654,261181,64544,1137,958,179
215,2025-12-01,2025,December,3.75,3.4,5.2,102.9,0.2,298697,261025,61626,913,724,189


In [334]:
# ============================================================
# CELL 155 — PREPARE MONTHLY MACROECONOMIC FACTORS
# ============================================================

print("=" * 70)
print("PREPARING MONTHLY MACROECONOMIC FACTORS")
print("=" * 70)

# ------------------------------------------------------------
# Required columns
# ------------------------------------------------------------

macro_columns = [
    "Date",
    "Bank_Rate",
    "CPI",
    "Unemployment_Rate",
    "House_Price_Index",
    "GDP_Growth",
    "Current_Accounts",
    "Savings_Accounts",
    "Mortgage_Approvals",
    "Net_Consumer_Credit",
    "Credit_Card_Lending",
    "Consumer_Credit"
]


# ------------------------------------------------------------
# Create clean monthly dataframe
# ------------------------------------------------------------

monthly_macro = macro_df[
    macro_columns
].copy()


# ------------------------------------------------------------
# Sort by date
# ------------------------------------------------------------

monthly_macro = (
    monthly_macro
    .sort_values("Date")
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Check numeric columns
# ------------------------------------------------------------

numeric_columns = [
    col for col in macro_columns
    if col != "Date"
]

for col in numeric_columns:

    monthly_macro[col] = pd.to_numeric(
        monthly_macro[col],
        errors="coerce"
    )


# ============================================================
# VALIDATION
# ============================================================

print("\nShape:")
print(monthly_macro.shape)

print("\nDate range:")
print(
    monthly_macro["Date"].min(),
    "to",
    monthly_macro["Date"].max()
)

print("\nDuplicate dates:")
print(
    monthly_macro["Date"].duplicated().sum()
)

print("\nMissing values:")

display(
    monthly_macro.isna().sum()
)


# ============================================================
# CHECK MONTHLY FREQUENCY
# ============================================================

date_difference = (
    monthly_macro["Date"]
    .diff()
    .dropna()
)

print("\nDate intervals:")
print(
    date_difference.value_counts().head()
)



PREPARING MONTHLY MACROECONOMIC FACTORS

Shape:
(216, 12)

Date range:
2008-01-01 00:00:00 to 2025-12-01 00:00:00

Duplicate dates:
0

Missing values:


Date                   0
Bank_Rate              0
CPI                    0
Unemployment_Rate      0
House_Price_Index      0
GDP_Growth             0
Current_Accounts       0
Savings_Accounts       0
Mortgage_Approvals     0
Net_Consumer_Credit    0
Credit_Card_Lending    0
Consumer_Credit        0
dtype: int64


Date intervals:
Date
31 days    125
30 days     72
28 days     13
29 days      5
Name: count, dtype: int64


In [335]:
# ============================================================
# CELL 156 — MONTHLY ECONOMIC CHANGE FEATURES
# ============================================================

print("=" * 70)
print("CREATING MONTHLY ECONOMIC CHANGE FEATURES")
print("=" * 70)

monthly_economic = monthly_macro.copy()


# ============================================================
# 1. BANK RATE CHANGE
# ============================================================

monthly_economic["Bank_Rate_Change"] = (
    monthly_economic["Bank_Rate"].diff()
)


# ============================================================
# 2. CPI CHANGE
# ============================================================

monthly_economic["CPI_Change"] = (
    monthly_economic["CPI"].diff()
)


# ============================================================
# 3. UNEMPLOYMENT CHANGE
# ============================================================

monthly_economic["Unemployment_Change"] = (
    monthly_economic["Unemployment_Rate"].diff()
)


# ============================================================
# 4. HOUSE PRICE CHANGE
# ============================================================

monthly_economic["House_Price_Change_Pct"] = (
    monthly_economic["House_Price_Index"]
    .pct_change()
    * 100
)


# ============================================================
# 5. CURRENT ACCOUNT CHANGE
# ============================================================

monthly_economic["Current_Accounts_Change_Pct"] = (
    monthly_economic["Current_Accounts"]
    .pct_change()
    * 100
)


# ============================================================
# 6. SAVINGS ACCOUNT CHANGE
# ============================================================

monthly_economic["Savings_Accounts_Change_Pct"] = (
    monthly_economic["Savings_Accounts"]
    .pct_change()
    * 100
)


# ============================================================
# 7. MORTGAGE APPROVAL CHANGE
# ============================================================

monthly_economic["Mortgage_Approvals_Change_Pct"] = (
    monthly_economic["Mortgage_Approvals"]
    .pct_change()
    * 100
)


# ============================================================
# 8. CONSUMER CREDIT CHANGE
# ============================================================

monthly_economic["Consumer_Credit_Change_Pct"] = (
    monthly_economic["Consumer_Credit"]
    .pct_change()
    * 100
)


# ============================================================
# 9. CREDIT CARD LENDING CHANGE
# ============================================================

monthly_economic["Credit_Card_Lending_Change_Pct"] = (
    monthly_economic["Credit_Card_Lending"]
    .pct_change()
    * 100
)


# ============================================================
# 10. NET CONSUMER CREDIT CHANGE
# ============================================================

monthly_economic["Net_Consumer_Credit_Change_Pct"] = (
    monthly_economic["Net_Consumer_Credit"]
    .pct_change()
    * 100
)


# ============================================================
# 11. ECONOMIC STRESS INDICATOR
# ============================================================

monthly_economic["Economic_Stress"] = (
    (
        monthly_economic["Bank_Rate"]
        > monthly_economic["Bank_Rate"].median()
    ).astype(int)
    +
    (
        monthly_economic["CPI"]
        > monthly_economic["CPI"].median()
    ).astype(int)
    +
    (
        monthly_economic["Unemployment_Rate"]
        > monthly_economic["Unemployment_Rate"].median()
    ).astype(int)
    +
    (
        monthly_economic["GDP_Growth"] < 0
    ).astype(int)
)


# ============================================================
# VALIDATION
# ============================================================

print("\nShape:")
print(monthly_economic.shape)

print("\nNew columns:")

new_columns = [
    col for col in monthly_economic.columns
    if col not in monthly_macro.columns
]

print(new_columns)


print("\nMissing values in new features:")

display(
    monthly_economic[new_columns].isna().sum()
)


# ============================================================
# FIRST 10 MONTHS
# ============================================================

print("\nFirst 10 months:")

display(
    monthly_economic[
        [
            "Date",
            "Bank_Rate",
            "Bank_Rate_Change",
            "CPI",
            "CPI_Change",
            "Unemployment_Rate",
            "Unemployment_Change",
            "GDP_Growth",
            "Economic_Stress"
        ]
    ].head(10)
)


# ============================================================
# FINAL CHECK
# ============================================================

print("\n" + "=" * 70)
print("MONTHLY ECONOMIC FEATURES READY")
print("=" * 70)

CREATING MONTHLY ECONOMIC CHANGE FEATURES

Shape:
(216, 23)

New columns:
['Bank_Rate_Change', 'CPI_Change', 'Unemployment_Change', 'House_Price_Change_Pct', 'Current_Accounts_Change_Pct', 'Savings_Accounts_Change_Pct', 'Mortgage_Approvals_Change_Pct', 'Consumer_Credit_Change_Pct', 'Credit_Card_Lending_Change_Pct', 'Net_Consumer_Credit_Change_Pct', 'Economic_Stress']

Missing values in new features:


Bank_Rate_Change                  1
CPI_Change                        1
Unemployment_Change               1
House_Price_Change_Pct            1
Current_Accounts_Change_Pct       1
Savings_Accounts_Change_Pct       1
Mortgage_Approvals_Change_Pct     1
Consumer_Credit_Change_Pct        1
Credit_Card_Lending_Change_Pct    1
Net_Consumer_Credit_Change_Pct    1
Economic_Stress                   0
dtype: int64


First 10 months:


,Date,Bank_Rate,Bank_Rate_Change,CPI,CPI_Change,Unemployment_Rate,Unemployment_Change,GDP_Growth,Economic_Stress
0,2008-01-01,5.50,NaN,2.2,NaN,5.2,NaN,0.4,2
1,2008-02-01,5.25,-0.25,2.5,0.3,5.2,0.0,0.4,2
2,2008-03-01,5.25,0.00,2.5,0.0,5.3,0.1,0.4,2
3,2008-04-01,5.00,-0.25,3.0,0.5,5.2,-0.1,-0.5,4
4,2008-05-01,5.00,0.00,3.3,0.3,5.4,0.2,-0.5,4
5,2008-06-01,5.00,0.00,3.8,0.5,5.5,0.1,-0.5,4
6,2008-07-01,5.00,0.00,4.4,0.6,5.7,0.2,-1.6,4
7,2008-08-01,5.00,0.00,4.7,0.3,5.9,0.2,-1.6,4
8,2008-09-01,5.00,0.00,5.2,0.5,6.0,0.1,-1.6,4
9,2008-10-01,4.50,-0.50,4.5,-0.7,6.2,0.2,-2.1,4



MONTHLY ECONOMIC FEATURES READY


In [336]:
# ============================================================
# CELL 157 — FINALISE MONTHLY ECONOMIC FEATURES
# ============================================================

print("=" * 70)
print("FINALISING MONTHLY ECONOMIC FEATURES")
print("=" * 70)

change_columns = [
    "Bank_Rate_Change",
    "CPI_Change",
    "Unemployment_Change",
    "House_Price_Change_Pct",
    "Current_Accounts_Change_Pct",
    "Savings_Accounts_Change_Pct",
    "Mortgage_Approvals_Change_Pct",
    "Consumer_Credit_Change_Pct",
    "Credit_Card_Lending_Change_Pct",
    "Net_Consumer_Credit_Change_Pct"
]

# Only January 2008 has no previous-month observation.
monthly_economic[change_columns] = (
    monthly_economic[change_columns]
    .fillna(0)
)


# ============================================================
# FINAL MISSING-VALUE CHECK
# ============================================================

print("\nMissing values:")

print(
    monthly_economic.isna().sum()
)


# ============================================================
# CHECK ECONOMIC STRESS
# ============================================================

print("\nEconomic Stress distribution:")

display(
    monthly_economic[
        "Economic_Stress"
    ].value_counts()
    .sort_index()
)


# ============================================================
# ECONOMIC STRESS BY YEAR
# ============================================================

print("\nEconomic Stress by year:")

stress_by_year = (
    monthly_economic
    .groupby("Date")
    ["Economic_Stress"]
    .first()
)

display(
    monthly_economic[
        [
            "Date",
            "Bank_Rate",
            "CPI",
            "Unemployment_Rate",
            "GDP_Growth",
            "Economic_Stress"
        ]
    ].head(12)
)


print("\n" + "=" * 70)
print("MONTHLY ECONOMIC DATA READY FOR CUSTOMER SIMULATION")
print("=" * 70)

FINALISING MONTHLY ECONOMIC FEATURES

Missing values:
Date                              0
Bank_Rate                         0
CPI                               0
Unemployment_Rate                 0
House_Price_Index                 0
GDP_Growth                        0
Current_Accounts                  0
Savings_Accounts                  0
Mortgage_Approvals                0
Net_Consumer_Credit               0
Credit_Card_Lending               0
Consumer_Credit                   0
Bank_Rate_Change                  0
CPI_Change                        0
Unemployment_Change               0
House_Price_Change_Pct            0
Current_Accounts_Change_Pct       0
Savings_Accounts_Change_Pct       0
Mortgage_Approvals_Change_Pct     0
Consumer_Credit_Change_Pct        0
Credit_Card_Lending_Change_Pct    0
Net_Consumer_Credit_Change_Pct    0
Economic_Stress                   0
dtype: int64

Economic Stress distribution:


Economic_Stress
0    26
1    91
2    72
3    16
4    11
Name: count, dtype: int64


Economic Stress by year:


,Date,Bank_Rate,CPI,Unemployment_Rate,GDP_Growth,Economic_Stress
0,2008-01-01,5.50,2.2,5.2,0.4,2
1,2008-02-01,5.25,2.5,5.2,0.4,2
2,2008-03-01,5.25,2.5,5.3,0.4,2
3,2008-04-01,5.00,3.0,5.2,-0.5,4
4,2008-05-01,5.00,3.3,5.4,-0.5,4
5,2008-06-01,5.00,3.8,5.5,-0.5,4
6,2008-07-01,5.00,4.4,5.7,-1.6,4
7,2008-08-01,5.00,4.7,5.9,-1.6,4
8,2008-09-01,5.00,5.2,6.0,-1.6,4
9,2008-10-01,4.50,4.5,6.2,-2.1,4



MONTHLY ECONOMIC DATA READY FOR CUSTOMER SIMULATION


In [337]:
# ============================================================
# CELL 158 — CHECK EXISTING REGRESSION COEFFICIENTS
# ============================================================

print("=" * 70)
print("CHECKING EXISTING REGRESSION MODELS")
print("=" * 70)

# Find regression/model objects currently in memory

for name, obj in list(globals().items()):

    obj_type = type(obj).__name__

    if any(
        keyword in obj_type.lower()
        for keyword in [
            "regression",
            "linear",
            "logistic",
            "ols"
        ]
    ):

        print(
            f"\n{name}: {obj_type}"
        )

        # Try to display coefficients
        if hasattr(obj, "coef_"):

            print("Coefficients:")
            print(obj.coef_)

        if hasattr(obj, "intercept_"):

            print("Intercept:")
            print(obj.intercept_)


print("\n" + "=" * 70)
print("CHECK COMPLETE")
print("=" * 70)

CHECKING EXISTING REGRESSION MODELS

CHECK COMPLETE
